# 논문 A 「종량 하한과 가격 과소신고: 한국 농산물 선택 세율의 증거」가 쓴 프로그램

이 노트북은 논문 A(`논문A_종량하한과_과소신고.md`)의 본문과 표가 쓴 파이썬 프로그램 전부를 한곳에 모은 **기록용** 노트북이다. 1~13은 자료를 만든 스크립트를 저장소의 파일에서 **그대로** 옮긴 셀이고(첫 줄 주석에 파일 경로), 14는 세 논문 공용 셀 창고 `cache/cells.py`에서 논문 A의 수치를 만드는 셀만 골라 옮긴 것이다. 14만 따로 담은 실행용 노트북이 같은 폴더의 `논문A_분석.ipynb`다. 빌더는 `cache/build_nb_paperA.py`이며 스크립트나 셀 창고를 고치면 다시 만든다.

**실행 환경.** conda 환경 `kcsdb`(duckdb·pandas·numpy·statsmodels·requests·olefile·openpyxl). 무역통계 DB(`kcsdb.duckdb`)와 HSK 별표는 KCSDB2 저장소에서 읽기 전용으로 쓴다(환경변수 `KCSDB2_ROOT`·`KCSDB2_PATH`), 세율 DB는 이 저장소의 `data/processed/kcstariff.duckdb`. 스크립트 셀은 파일로 실행하도록 쓴 것이라(`__file__` 기준 경로, `if __name__ == "__main__":` 진입점) 이 노트북을 위에서 아래로 그냥 실행하면 포털·법령센터·Comtrade·ECOS에 실제로 요청을 보내고 DB를 다시 쓴다. 다시 하려는 것만 파일로 골라 돌린다. 분석은 `논문A_분석.ipynb`를 `research/`에서 실행한다(약 5분, `outputs/`의 표와 무역통계 DB만 읽는다).

**저장된 프로그램이 없는 작업.** 사전세액심사 목록은 브라우저 페이지 컨텍스트에서 자바스크립트로 받았고(6, 코드는 그대로 둔다) JSON을 CSV로 옮기는 것은 대화형이었다. 유통이력관리 고시 별표 48판은 5의 HWP·HWPX 읽기 함수로 텍스트를 뽑은 뒤 대화형으로 판본별·구간·통합 표를 만들었다. 품목분류 적용기준 규칙 별표 43개 항목은 PDF 텍스트를 뽑아 손으로 `분류기준_규칙_별표.csv`에 옮겼고 7의 스크립트가 그것을 읽는다. 파일럿 계열(건조생강·건고추의 원산지×월 수입)은 대화형으로 뽑았던 것을 13의 셀로 복원해 옛 파일과 전부 같음을 확인했다.

| 순서 | 무엇 | 셀 | 논문 A에서 쓰는 곳 | 산출 |
|---|---|---|---|---|
| 1 | 관세율표·주요세율보기 스무 해 | `scripts/01_fetch_tariff.py` | II.1 세율 | `kcstariff.duckdb`의 `tariff_code`·`tariff_rate` |
| 2 | 실행세율과 종량 하한 | `scripts/02_build_applied_rate.py` | II.1·II.2 | `fct_applied_rate`(`floor_won_kg`) |
| 3 | 양허관세 별표 1(가·나) | `research/scripts/25_parse_concession_annex.py` | II.2 종량 대안 코드 92개 | `양허관세_별표1_2025.csv` |
| 4 | 원/달러 환율 월평균 | `research/scripts/26_fetch_ecos_fx.py` | III.1 하한 비교 | `환율_월별_USDKRW.csv` |
| 5 | 유통이력 별표 48판 | HWP·HWPX 읽기 함수 | II.3 집행 지정 | 텍스트(표는 대화형) |
| 6 | 사전세액심사 월별 목록 | 자바스크립트(페이지 컨텍스트) | II.3 집행 지정 | JSON(CSV는 대화형) |
| 7 | 코드 쌍(별표·규칙) | `research/scripts/16_build_code_pairs.py` | II.4 패널 코드, III.2 대조군, 변환비 쌍 | `코드쌍_목록.csv`·`코드쌍_코드목록.txt` |
| 8 | 품목 상세(코드 쌍 후보 285개) | `research/scripts/17_fetch_item_detail.py` | II.1 그 밖 FTA, II.3 사전세액심사 표시 | `품목상세_세율_2012_2026.csv`·`품목상세_부가정보_2012_2026.csv` |
| 9 | 신설 코드 처리군 | `research/scripts/18_build_treatments.py` | II.4 패널 코드, III.2 대조군에서 제외 | `처리군_목록.csv` |
| 10 | 원산지별 실행세율 | `research/scripts/19_extend_applied_rate.py` | II.1 | `실행세율_원산지별_쌍코드_2012_2026.csv` |
| 11 | 통관 패널 | `research/scripts/20_build_panel.py` | II.4, IV.1 | `outputs/panel/` |
| 12 | 미러 통계와 갭 회귀 | `research/scripts/21_fetch_comtrade_mirror.py`·`22_mirror_gap.py` | II.5, III.4, IV.2 표 3, IV.5 표 7 | `comtrade_mirror_hs6_2012_2024.csv`·`미러_갭_*.csv`·`미러_건조생강.csv` |
| 13 | 파일럿 계열 | 재생성 셀 | IV.1·IV.2 건고추·건조생강 단가 | `pilot_series.csv` |
| 14 | 분석 | 셀 창고의 논문 A 셀(= `논문A_분석.ipynb`) | IV장 표 2~7, V장 강건성 | `논문A_수치.json`·`논문A_검증_결과.json` |

## 1. 관세율표·주요세율보기 — 세율 두 화면 스무 해

논문 II.1. 캐시가 있으면 `--parse-only`, 검증만 하려면 `--verify-only`(`outputs/수집_검증.csv`에 두 화면 일치율).

In [ ]:
# ===== 파일: scripts/01_fetch_tariff.py (그대로 옮김, 23,775바이트) =====
"""
01_fetch_tariff.py — 연도별 HSK 10단위 실행세율 수집 (관세법령정보포털)

무엇을 푸는가:
  KCSDB2에는 관세율이 없다. 품목분류와 세율 차이를 함께 보려면(세율이 높은 품목의 수입이
  세율 낮은 비슷한 품목으로 옮겨 가는가) 코드마다 그해 적용된 세율이 있어야 한다.
  공공데이터포털의 품목번호별 관세율표(15051179)는 현행판만 있어 과거를 못 준다.

  관세법령정보포털은 2002년부터 연도별로 두 화면을 준다. 둘을 합쳐야 실행세율이 된다.
    관세율표(openULS0201005Q)   기본세율 + 탄력·양허 세율(WTO 협정세율 C, 농림축산물
                                양허관세 W1·W2, 조정관세 L, 할당관세 P, 특별긴급관세 T,
                                아시아·태평양 협정세율 E 등). 연중 변경 표시는 없다.
    주요세율보기(openULS0201017Q) 기본·WTO·아시아태평양 + FTA 일곱 상대(중국·EU·미국·
                                아세안·인도·베트남·캐나다). 연중에 바뀌면 기간을 나눠
                                보여 준다(한-EU FTA는 7월 1일 인하). 조정·할당·양허는 없다.
  그 밖의 FTA(칠레·EFTA·호주 등)는 품목 상세 화면에만 있어 코드마다 따로 받아야 하므로
  여기서는 받지 않는다. 연구 대상 코드가 정해지면 그것만 받는다.

받는 방법:
  두 화면 모두 hsfdCd에 류(2자리)를 주면 그 류 전체가 온다. 연도당 97회씩이다.
  관세율표 화면은 KCSDB2의 `03j_fetch_hsk_table.py`와 같은 요청이라 캐시(data/raw/clip_hsk)를
  함께 쓴다 — 2007~2010년은 이미 받아 두었다.

입력: 없음 (포털에서 받는다)
출력:
  data/raw/clip_hsk/<연도>/<류>.html            관세율표 캐시 (03j와 공유)
  data/raw/clip_tariff_main/<연도>/<류>.html    주요세율보기 캐시
  data/processed/kcstariff.duckdb               별도 DB 파일 (KCSDB2와 ATTACH로 결합)
    tariff_code   year, hs10, name_ko, source              그해 관세율표의 10단위 코드
    tariff_rate   year, hs10, rate_cd, rate_txt, adval, specific, valid_from, valid_to, source
                  source는 table(관세율표) · main(주요세율보기의 FTA 열) · fill(관세율표 화면을
                  받지 못한 코드를 주요세율보기의 기본·WTO·아시아태평양 세율로 채운 것)
    dim_rate_cd   rate_cd, rate_nm, source
    meta_fetch    page, year, ryu, fetched_at, bytes

세율 값:
  rate_txt는 화면 문구 그대로다. adval은 종가세율(%), specific은 종량세액(원)이다.
  「270% 또는 6,210원」은 둘 다 채운다. 어느 쪽을 적용하는지는 여기서 정하지 않는다.
  FTA 세율은 특혜를 신청했을 때 적용 가능한 세율이지 실제 납부 세율이 아니다.
  포털은 10단위 세율이 참고용이고 법적 효력이 없다고 밝힌다.

검증:
  ① 두 화면의 코드 집합이 같은가 ② 두 화면에 함께 나오는 기본세율·WTO 세율이 같은가
  ③ 알려진 값 몇 개(건고추 270%, 고추다진양념 조정관세 45% 등) ④ 그해 수입액 중
  세율표에 코드가 있는 몫.

실행:
  python scripts\\01_fetch_tariff.py                       # 2007~2026 전부
  python scripts\\01_fetch_tariff.py --years 2025 --ryu 9 21
  python scripts\\01_fetch_tariff.py --parse-only          # 캐시에서 다시 적재만
"""

from __future__ import annotations

import argparse
import html
import logging
import os
import re
import sys
import time
from datetime import date, datetime
from pathlib import Path

import duckdb
import pandas as pd
import requests

PROJECT_ROOT = Path(__file__).resolve().parent.parent
# 원본 캐시는 크다(관세율표 700MB·주요세율 450MB). 이미 받은 캐시가 다른 곳에 있으면 KCSTARIFF_RAW로 가리킨다.
RAW = Path(os.environ.get("KCSTARIFF_RAW", PROJECT_ROOT / "data" / "raw"))
DB_OUT = PROJECT_ROOT / "data" / "processed" / "kcstariff.duckdb"
# 수입액 커버리지 검증은 KCSDB2(무역통계 DB)가 있을 때만 한다. 없으면 건너뛴다.
DB_TRADE = Path(os.environ.get("KCSDB2_PATH", PROJECT_ROOT.parent / "KCSDB2" / "data" / "processed" / "kcsdb.duckdb"))
LOG_DIR = PROJECT_ROOT / "logs"
OUT_DIR = PROJECT_ROOT / "outputs"          # 검증 수치 CSV(수집_검증.csv)
LOG_DIR.mkdir(exist_ok=True)

LOG_PATH = LOG_DIR / f"fetch_tariff_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(LOG_PATH, encoding="utf-8"), logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

BASE = "https://unipass.customs.go.kr/clip/hsinfosrch/"
PAGES = {
    "table": ("openULS0201005Q.do", RAW / "clip_hsk"),
    "main": ("openULS0201017Q.do", RAW / "clip_tariff_main"),
}
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Referer": "https://unipass.customs.go.kr/clip/index.do",
    "Content-Type": "application/x-www-form-urlencoded",
}
YEARS = range(2007, 2027)
RYU = range(1, 100)
ERROR_MARK = "프로그램 오류발생"

# 주요세율보기의 열 이름. 화면 머리글은 '중국'처럼 짧아 이름을 여기서 준다.
MAIN_NAMES = {
    "FCN1": "한ㆍ중국 FTA협정세율(선택1)", "FEU1": "한ㆍEU FTA협정세율(선택1)",
    "FUS1": "한ㆍ미 FTA 협정세율(선택1)", "FAS1": "한ㆍ아세안 FTA협정세율(선택1)",
    "FIN1": "한ㆍ인도 FTA협정세율(선택1)", "FVN1": "한ㆍ베트남 FTA협정세율(선택1)",
    "FCA1": "한ㆍ캐나다 FTA협정세율(선택1)",
}


def fetch(sess: requests.Session, page: str, year: int, ryu: int) -> str:
    """류 하나를 받는다. 관세율표는 03j와 캐시를 함께 쓰므로 요청 변수도 03j와 같게 둔다."""
    ymd = "20070101" if page == "table" else f"{year}0101"
    data = dict(cntyCd="KR", aplyYy=str(year), cntyNm="한국", compareCrrspndNation="KR",
                sctYear=ymd, hstdYear=ymd, manlOrgnTpcd="01", tabTpcd="3",
                sctCd="01", hstdCd=f"{ryu:02d}", hsfdCd=f"{ryu:02d}", searchVal=f"{ryu:02d}")
    r = sess.post(BASE + PAGES[page][0], data=data, timeout=60)
    r.raise_for_status()
    # 서버가 표를 절반쯤 보내다 오류 안내문으로 끝내는 일이 있다(2011년 제90류). 상태 코드는
    # 200이라 문구로 가려낸다 — 그대로 캐시하면 코드 수백 개가 조용히 빠진다.
    if ERROR_MARK in r.text:
        raise requests.RequestException("응답이 오류 안내문으로 끝났다")
    return r.text


def cell_text(x: str) -> str:
    return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", x))).replace(chr(0xa0), " ").strip()


def parse_rate(txt: str) -> tuple[float | None, float | None]:
    """'270% 또는 6,210원' → (270.0, 6210.0). 못 읽으면 None."""
    a = re.search(r"([\d.]+)\s*%", txt)
    s = re.search(r"([\d,]+(?:\.\d+)?)\s*원", txt)
    return (float(a.group(1)) if a else None,
            float(s.group(1).replace(",", "")) if s else None)


def parse_table(page: str, year: int) -> tuple[list, list]:
    """관세율표 화면 → (코드 행, 세율 행).

    10단위 행마다 기본세율 칸이 있고, 탄력·양허 세율은 표에 비어 있는 span을 페이지의
    스크립트(f_KorAdTax)가 행 번호(korAdTax_N)로 채운다. 행 번호로 둘을 잇는다.
    """
    codes, rates, rowno = [], [], {}
    for tr in re.findall(r"<tr[^>]*>(.*?)</tr>", page, re.S):
        h = re.search(r'name="hsSgn_Mn" value="(\d{10})"', tr)
        k = re.search(r'korAdTax_(\d+)', tr)
        if not (h and k):
            continue
        tds = [cell_text(x) for x in re.findall(r"<td[^>]*>(.*?)</td>", tr, re.S)]
        hs10 = h.group(1)
        rowno[k.group(1)] = hs10
        codes.append((year, hs10, tds[3] if len(tds) > 3 else ""))
        if len(tds) > 5 and tds[5]:
            rates.append((year, hs10, "A", tds[5], None, None, "table"))

    js = page[page.find("function f_KorAdTax"):]
    js = js[:js.find("</script>")]
    for blk in re.split(r'showID = "#korAdTax_" \+ "', js)[1:]:
        n = blk[:blk.find('"')]
        cds = re.findall(r'<span title="([^"]+)"><a href="#"><span class="textColor">(\w+)</span>', blk)
        vals = re.findall(r"htm2 \+= '<span><a href=\"#\">([^<]*)</a>", blk)
        if len(cds) != len(vals):
            logger.warning("  %d년 행 %s: 세율 구분 %d개와 값 %d개가 어긋난다", year, n, len(cds), len(vals))
        hs10 = rowno.get(n)
        if hs10 is None:
            continue
        for (nm, cd), v in zip(cds, vals):
            rates.append((year, hs10, cd, v.strip(), nm, None, "table"))
    return codes, rates


PERIOD = re.compile(r"(.*?)\s*-\s*\((\d{4}-\d{2}-\d{2})\s*~\s*(\d{4}-\d{2}-\d{2})?\)")


def parse_main(page: str, year: int) -> tuple[list, list]:
    """주요세율보기 화면 → 세율 행. 머리글 마지막 줄의 구분기호(A, C, FCN1 …)가 열 순서다.

    첫 줄에도 'E1'이라는 표시가 있어 머리글 전체에서 구분기호를 모으면 열이 하나 늘어난다.
    """
    thead = page[page.find("<thead"):page.find("</thead>")]
    last = re.findall(r"<tr[^>]*>(.*?)</tr>", thead, re.S)[-1]
    cols = [c for c in (cell_text(x) for x in re.findall(r"<th[^>]*>(.*?)</th>", last, re.S))
            if re.fullmatch(r"[A-Z][A-Z0-9]*", c)]
    out, names = [], []
    body = page[page.find("<tbody"):]
    for tr in re.findall(r"<tr[^>]*>(.*?)</tr>", body, re.S):
        tds = [cell_text(x) for x in re.findall(r"<td[^>]*>(.*?)</td>", tr, re.S)]
        if len(tds) < 4 + len(cols) or not (re.fullmatch(r"\d{4}", tds[0])
                                             and re.fullmatch(r"\d{2}", tds[1])
                                             and re.fullmatch(r"\d{4}", tds[2])):
            continue
        hs10 = tds[0] + tds[1] + tds[2]
        names.append((year, hs10, tds[3]))
        for cd, v in zip(cols, tds[4:4 + len(cols)]):
            if not v:
                continue
            segs = PERIOD.findall(v)
            if segs:
                for val, f, t in segs:
                    out.append((year, hs10, cd, val.strip(), None, (f, t or None), "main"))
            else:
                out.append((year, hs10, cd, v, None, None, "main"))
    return out, names


def collect(years, ryus, pages, delay, parse_only):
    sess = requests.Session()
    sess.headers.update(HEADERS)
    meta = []
    n_req = 0
    for y in years:
        for page in pages:
            cache = PAGES[page][1] / str(y)
            cache.mkdir(parents=True, exist_ok=True)
            for ryu in ryus:
                f = cache / f"{ryu:02d}.html"
                if f.exists() or parse_only:
                    continue
                for attempt in range(3):
                    try:
                        txt = fetch(sess, page, y, ryu)
                        break
                    except requests.RequestException as e:
                        logger.warning("  %s %d년 제%02d류 실패(%d회): %s", page, y, ryu, attempt + 1, e)
                        time.sleep(5 * (attempt + 1))
                else:
                    # 같은 자리에서 매번 끊기는 류가 있다(2011년 관세율표 제90류는 9006519000
                    # 행에서 서버 오류가 난다. 호 단위로 받아도 같다). 캐시하지 않고 넘어가며,
                    # build가 그 코드들을 주요세율보기로 채운다.
                    logger.error("  %s %d년 제%02d류를 받지 못해 건너뛴다", page, y, ryu)
                    continue
                f.write_text(txt, encoding="utf-8")
                meta.append((page, y, ryu, datetime.now(), len(txt)))
                n_req += 1
                time.sleep(delay)
            logger.info("%d년 %s 캐시 준비 (이번에 받은 누적 %d회)", y, page, n_req)
    return meta


def build(years, ryus):
    codes, rates, main_codes = [], [], []
    for y in years:
        for ryu in ryus:
            ft = PAGES["table"][1] / str(y) / f"{ryu:02d}.html"
            fm = PAGES["main"][1] / str(y) / f"{ryu:02d}.html"
            for f in (ft, fm):
                if f.exists() and ERROR_MARK in f.read_text(encoding="utf-8"):
                    logger.warning("  오류 안내문이 섞인 캐시: %s (지우고 다시 받을 것)", f)
            if ft.exists():
                c, r = parse_table(ft.read_text(encoding="utf-8"), y)
                codes += c
                rates += r
            if fm.exists():
                r, c = parse_main(fm.read_text(encoding="utf-8"), y)
                rates += r
                main_codes += c

    code = (pd.DataFrame(codes, columns=["year", "hs10", "name_ko"])
            .drop_duplicates(["year", "hs10"]).assign(source="table"))
    mc = pd.DataFrame(main_codes, columns=["year", "hs10", "name_ko"]).drop_duplicates(["year", "hs10"])
    fill = (mc.merge(code[["year", "hs10"]], how="left", indicator=True)
            .query("_merge == 'left_only'").drop(columns="_merge"))
    code = pd.concat([code, fill.assign(source="main")], ignore_index=True)
    rt = pd.DataFrame(rates, columns=["year", "hs10", "rate_cd", "rate_txt", "rate_nm", "period", "source"])

    # 관세율표 화면을 받지 못한 코드는 주요세율보기의 기본·WTO·아시아태평양 세율로 채운다.
    # 그 코드에 조정·할당·양허관세가 있었다면 빠진다(2011년 제90류는 앞뒤 해에 그런 세율이 없다).
    fk = set(zip(fill.year, fill.hs10))
    isfill = ((rt.source == "main") & ~rt.rate_cd.str.startswith("F")
              & pd.Series([(y, h) in fk for y, h in zip(rt.year, rt.hs10)], index=rt.index))
    rt.loc[isfill, "source"] = "fill"
    if len(fill):
        logger.warning("  관세율표 화면에 없어 주요세율보기로 채운 코드: %s",
                       fill.groupby("year").size().to_dict())
    rt["valid_from"] = [date(y, 1, 1) if p is None else date.fromisoformat(p[0])
                        for y, p in zip(rt.year, rt.period)]
    rt["valid_to"] = [date(y, 12, 31) if p is None or p[1] is None else date.fromisoformat(p[1])
                      for y, p in zip(rt.year, rt.period)]
    ad = rt.rate_txt.map(parse_rate)
    rt["adval"] = [a for a, _ in ad]
    rt["specific"] = [s for _, s in ad]

    names = rt.dropna(subset=["rate_nm"]).drop_duplicates("rate_cd")[["rate_cd", "rate_nm"]]
    names["source"] = "table"
    fta = pd.DataFrame([(k, v, "main") for k, v in MAIN_NAMES.items()], columns=names.columns)
    names = pd.concat([names, fta]).drop_duplicates("rate_cd")
    names.loc[names.rate_cd == "A", ["rate_nm", "source"]] = ["기본세율", "table"]
    if "A" not in set(names.rate_cd):
        names.loc[len(names)] = ["A", "기본세율", "table"]
    return code, rt, names


def verify(code: pd.DataFrame, rt: pd.DataFrame):
    """두 화면의 대조와 알려진 값 확인, 수입액 커버리지. 로그에 찍는 수치를 outputs/수집_검증.csv(item, key, value)에도 남긴다."""
    rows = []   # (item, key, value) — 논문·노트북이 대조할 수 있게 파일로 남긴다
    tb, mn = rt[rt.source == "table"], rt[rt.source == "main"]
    for y in sorted(code.year.unique()):
        ct = set(code[(code.year == y) & (code.source == "table")].hs10)
        cm = set(mn[mn.year == y].hs10)
        if cm:
            logger.info("  %d년 코드: 관세율표 %d / 주요세율보기 %d (한쪽에만 %d)",
                        y, len(ct), len(cm), len(ct ^ cm))
            rows += [("코드 집합 한쪽에만", y, len(ct ^ cm)), ("코드 관세율표", y, len(ct)), ("코드 주요세율보기", y, len(cm))]

    # 두 화면에 함께 나오는 기본세율(A)과 WTO 세율(C)이 같은가. 문구는 천 단위 쉼표가
    # 화면마다 달라('1218원'과 '1,218원') 숫자로 견준다.
    key = ["year", "hs10", "adval", "specific", "rate_txt"]
    for cd in ("A", "C"):
        a = tb[tb.rate_cd == cd].drop_duplicates(["year", "hs10"])[key]
        b = mn[mn.rate_cd == cd].drop_duplicates(["year", "hs10"])[key]
        m = a.merge(b, on=["year", "hs10"])
        if len(m):
            ok = (m.adval_x.fillna(-1) == m.adval_y.fillna(-1)) & (m.specific_x.fillna(-1) == m.specific_y.fillna(-1))
            logger.info("  %s 세율 두 화면 일치 %.2f%% (%d쌍)", cd, 100 * ok.mean(), len(m))
            rows += [(f"두 화면 일치율 {cd}", "전체", round(100 * ok.mean(), 2)), (f"두 화면 대조 쌍 {cd}", "전체", len(m))]
            if ok.mean() < 0.99:
                logger.warning("  어긋나는 예: %s", m[~ok].head(5).to_dict("records"))
    rt = rt[rt.source.isin(["table", "fill"]) | rt.rate_cd.str.startswith("F")]   # 적재 대상만 남긴다

    unread = rt[rt.adval.isna() & rt.specific.isna()]
    logger.info("  숫자로 못 읽은 세율 %d행 (%.3f%%) 예: %s", len(unread), 100 * len(unread) / max(len(rt), 1),
                unread.rate_txt.value_counts().head(5).to_dict())
    rows.append(("숫자로 못 읽은 세율 행", "전체", len(unread)))

    known = [  # (연도, 코드, 구분, 종가, 종량, 시작일)
        (2025, "0904210000", "W2", 270.0, 6210.0, None),
        (2009, "0910101000", "W2", 377.3, 931.0, None),
        (2025, "2103909050", "L", 45.0, None, None),
        (2025, "2103909050", "FCN1", 44.5, None, None),
        (2025, "2103909050", "FEU1", 5.6, None, date(2025, 1, 1)),
        (2025, "2103909050", "FEU1", 2.8, None, date(2025, 7, 1)),
        (2012, "8703231010", "C", 8.0, None, None),
        (2025, "0710807000", "C", 27.0, None, None),
    ]
    for y, h, cd, a, s, f in known:
        if y not in set(rt.year):
            continue
        hit = rt[(rt.year == y) & (rt.hs10 == h) & (rt.rate_cd == cd)]
        if f is not None:
            hit = hit[hit.valid_from == f]
        assert len(hit) == 1, f"{y} {h} {cd}: {len(hit)}행"
        r = hit.iloc[0]
        assert r["adval"] == a and (s is None or r["specific"] == s), f"{y} {h} {cd}: {r['rate_txt']}"
    logger.info("  알려진 값 확인 통과")
    rows.append(("알려진 값 확인", "건수", len([k for k in known if k[0] in set(rt.year)])))

    if DB_TRADE.exists():
        con = duckdb.connect(str(DB_TRADE), read_only=True)
        try:
            imp = con.execute("SELECT yyyymm//100 AS year, hs10, SUM(imp_dlr) AS v FROM fact_trade "
                              "GROUP BY 1, 2").df()
        finally:
            con.close()
        imp = imp[imp.year.isin(code.year.unique())]
        # 채움(fill) 코드까지 넣은 커버리지와, 관세율표 화면만으로 잰 커버리지(채움 전)를 함께 남긴다.
        # 2011년 제90류가 빠졌을 때 후자가 97.38%로 떨어져 결함이 드러났다.
        for label, sub in [("코드 있음", code), ("코드 있음(채움 전)", code[code.source == "table"])]:
            have = sub[["year", "hs10"]].drop_duplicates().assign(ok=1)
            m = imp.merge(have, on=["year", "hs10"], how="left")
            cov = m.assign(ok=m.ok.fillna(0)).groupby("year").apply(lambda g: (g.v * g.ok).sum() / g.v.sum())
            logger.info("  수입액 중 그해 세율표에 %s 몫: %s", label,
                        ", ".join(f"{y} {100 * c:.2f}%" for y, c in cov.items()))
            rows += [(f"수입액 커버리지 {label}", int(y), round(100 * c, 2)) for y, c in cov.items()]
    OUT_DIR.mkdir(exist_ok=True)
    pd.DataFrame(rows, columns=["item", "key", "value"]).to_csv(OUT_DIR / "수집_검증.csv", index=False, encoding="utf-8-sig")
    logger.info("  검증 수치 %d행 → %s", len(rows), OUT_DIR / "수집_검증.csv")


def load(code, rt, names, meta):
    """새 파일에 쓰고 바꿔 끼운다.

    DuckDB는 CREATE OR REPLACE로 표를 갈아도 파일이 줄지 않아 다시 적재할 때마다 커진다
    (한 번 다시 적재하자 27MB가 53MB가 됐다). 받은 기록(meta_fetch)만 옛 파일에서 옮겨 온다.
    """
    tmp = DB_OUT.with_name(DB_OUT.stem + ".tmp.duckdb")
    tmp.unlink(missing_ok=True)
    con = duckdb.connect(str(tmp))
    try:
        con.register("code_df", code)
        con.register("rt_df", rt.drop(columns=["period", "rate_nm"]))
        con.register("nm_df", names)
        con.execute("""
            CREATE OR REPLACE TABLE tariff_code AS
            SELECT CAST(year AS INTEGER) AS year, hs10, name_ko, source FROM code_df ORDER BY year, hs10""")
        con.execute("""
            CREATE OR REPLACE TABLE tariff_rate AS
            SELECT CAST(year AS INTEGER) AS year, hs10, rate_cd, rate_txt,
                   CAST(adval AS DOUBLE) AS adval, CAST(specific AS DOUBLE) AS specific,
                   CAST(valid_from AS DATE) AS valid_from, CAST(valid_to AS DATE) AS valid_to, source
            FROM rt_df ORDER BY year, hs10, rate_cd, valid_from""")
        con.execute("CREATE OR REPLACE TABLE dim_rate_cd AS SELECT * FROM nm_df ORDER BY rate_cd")
        con.execute("""CREATE TABLE meta_fetch
                       (page VARCHAR, year INTEGER, ryu INTEGER, fetched_at TIMESTAMP, bytes INTEGER)""")
        if DB_OUT.exists():
            con.execute(f"ATTACH '{DB_OUT.as_posix()}' AS old (READ_ONLY)")
            has = con.execute("SELECT COUNT(*) FROM duckdb_tables() "
                              "WHERE database_name = 'old' AND table_name = 'meta_fetch'").fetchone()[0]
            if has:
                con.execute("INSERT INTO meta_fetch SELECT * FROM old.meta_fetch")
            con.execute("DETACH old")
        if meta:
            con.executemany("INSERT INTO meta_fetch VALUES (?, ?, ?, ?, ?)", meta)
        con.execute("COMMENT ON TABLE tariff_rate IS '관세법령정보포털 연도별 관세율표(기본·탄력·양허)와 "
                    "주요세율보기(FTA 일곱 상대). FTA는 적용 가능 세율이지 납부 세율이 아니다. 포털 고지상 법적 효력 없음'")
        for t in ("tariff_code", "tariff_rate", "dim_rate_cd"):
            logger.info("  %s %d행", t, con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0])
    finally:
        con.close()
    os.replace(tmp, DB_OUT)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--years", type=int, nargs="+", default=list(YEARS))
    ap.add_argument("--ryu", type=int, nargs="+", default=list(RYU), help="시험용: 일부 류만")
    ap.add_argument("--pages", nargs="+", default=list(PAGES), choices=list(PAGES))
    ap.add_argument("--delay", type=float, default=0.7, help="요청 간격(초)")
    ap.add_argument("--parse-only", action="store_true", help="받지 않고 캐시만 적재")
    ap.add_argument("--verify-only", action="store_true", help="캐시를 파싱해 검증만 하고 적재하지 않는다(수집_검증.csv 갱신)")
    args = ap.parse_args()
    if args.verify_only:
        args.parse_only = True

    meta = collect(args.years, args.ryu, args.pages, args.delay, args.parse_only)
    code, rt, names = build(args.years, args.ryu)
    logger.info("파싱: 코드 %d, 세율 %d행", len(code), len(rt))
    verify(code, rt)
    if args.verify_only:
        logger.info("검증만 하고 적재는 하지 않는다"); return
    # 주요세율보기의 기본·WTO·아시아태평양 열은 관세율표와 겹치므로 대조에만 쓰고 FTA 열만 싣는다.
    # 관세율표 화면을 받지 못해 채운 코드(fill)는 예외다.
    rt = rt[rt.source.isin(["table", "fill"]) | rt.rate_cd.str.startswith("F")]
    load(code, rt, names, meta)
    logger.info("완료: %s", DB_OUT)


if __name__ == "__main__":
    main()

## 2. 실행세율과 종량 하한 — 관세법 제50조의 우선순위를 코드×연도에 적용

논문 II.1·II.2. 무협정 규정이 선택 세율이면 종량세를 종가세율로 나눈 종량 하한 `floor_won_kg`(원/kg)을 함께 만든다. `--load`로 DB에 적재.

In [ ]:
# ===== 파일: scripts/02_build_applied_rate.py (그대로 옮김, 13,508바이트) =====
"""
02_build_applied_rate.py — 연도×HS10의 실행세율(applicable rate)을 만든다.

근거: 관세법 제50조(세율 적용의 우선순위), FTA 관세특례법 제5조, 양허관세 규정 제6조.
조문 원문과 세율 구분 코드의 대응은 docs/실행세율_법령근거.md.

규칙(연구 문서 IV.1절):
  기본   = A(기본세율). 잠정세율은 자료에 없다.
  3순위  = P3(할당관세, 수입전량)가 있으면 P3, 아니면 L(조정관세)가 있으면 L, 아니면 기본.
           P1(할당관세 물량이내 추천)은 추천이 있어야 하므로 표시만 한다.
  2순위  = C(WTO 협정세율)·F(국제협력관세)는 3순위보다 낮을 때만 우선(제50조③ 본문).
           W2(농림축산물 양허관세, 별표 1의 나)는 기본세율보다 높아도 우선(제50조③ 단서)하되
           할당관세(P3)가 있으면 그것을 따른다. W1(추천)은 추천이 있어야 하므로 표시만 한다.
  1순위  = I(덤핑방지)·T1·T2(특별긴급)는 원산지·물량 조건이 붙는 추가 관세라 표시만 한다.
  협정   = FTA 일곱 상대(FCN1·FEU1·FUS1·FAS1·FIN1·FVN1·FCA1)와 APTA(E1·E2·E3)는
           무협정 적용세율보다 낮을 때만 적용(특례법 제5조①, 제50조③).
  연중 변경 = 구간별 세율을 그해 유효 일수로 가중평균한다(rate_*), 1월 1일 세율도 둔다(*_jan).
  종량 하한 = 선택 세율("N% 또는 M원")이 있는 규정에서 M/N(원/kg). 격차 변수에는 종가세율만 쓴다.

산출:
  outputs/fct_applied_rate.parquet (연도×HS10)
  outputs/dim_origin_regime.csv     (원산지→협정)
  data/processed/kcstariff.duckdb 의 fct_applied_rate, dim_origin_regime (--load 를 주면 적재)
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

import os
ROOT = Path(__file__).resolve().parent.parent
TARIFF = ROOT / "data" / "processed" / "kcstariff.duckdb"
KCS = Path(os.environ.get("KCSDB2_PATH", ROOT.parent / "KCSDB2" / "data" / "processed" / "kcsdb.duckdb"))   # 있으면 커버리지 검증
OUT = ROOT / "outputs"

FTA = {"FCN1": "cn", "FEU1": "eu", "FUS1": "us", "FAS1": "asean", "FIN1": "in", "FVN1": "vn", "FCA1": "ca"}

# 원산지(관세청 stat_cd = ISO2) → 협정. (regime, from_year, to_year)
EU27 = ["AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "GR", "HU", "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT", "RO", "SK", "SI", "ES", "SE"]
ASEAN = ["BN", "KH", "ID", "LA", "MY", "MM", "PH", "SG", "TH", "VN"]
APTA_E1 = ["CN", "IN", "LK", "MN"]      # 일반 협정세율
ORIGIN_REGIME = (
    [("CN", "FCN1", 2015, 9999), ("IN", "FIN1", 2010, 9999), ("US", "FUS1", 2012, 9999), ("VN", "FVN1", 2015, 9999), ("CA", "FCA1", 2015, 9999)]
    + [(c, "FEU1", 2011, 9999) for c in EU27 if c != "HR"] + [("HR", "FEU1", 2013, 9999)] + [("GB", "FEU1", 2011, 2020)]
    + [(c, "FAS1", 2008, 9999) for c in ASEAN]
    + [(c, "E1", 2007, 9999) for c in APTA_E1] + [("BD", "E2", 2007, 9999), ("LA", "E3", 2007, 9999)]
)


def load_rates() -> pd.DataFrame:
    con = duckdb.connect()
    con.execute(f"ATTACH '{TARIFF.as_posix()}' AS tr (READ_ONLY)")
    df = con.sql("SELECT year, hs10, rate_cd, rate_txt, adval, specific, valid_from, valid_to FROM tr.tariff_rate").df()
    con.close()
    df["valid_from"] = pd.to_datetime(df.valid_from)
    df["valid_to"] = pd.to_datetime(df.valid_to)
    return df


def weight_by_days(df: pd.DataFrame) -> pd.DataFrame:
    """(year, hs10, rate_cd)마다 유효 일수 가중 종가세율, 1월 1일 세율, 종량세(원/kg)를 만든다."""
    ystart = pd.to_datetime(df.year.astype(str) + "-01-01")
    yend = pd.to_datetime(df.year.astype(str) + "-12-31")
    vf = df.valid_from.fillna(ystart).clip(lower=ystart)
    vt = df.valid_to.fillna(yend).clip(upper=yend)
    df = df.assign(days=(vt - vf).dt.days.clip(lower=0) + 1, vf=vf)
    has = df.adval.notna()
    df["wnum"] = np.where(has, df.adval.fillna(0) * df.days, 0.0)
    df["wden"] = np.where(has, df.days, 0)
    keys = ["year", "hs10", "rate_cd"]
    g = df.groupby(keys, sort=False)
    out = g.agg(wnum=("wnum", "sum"), wden=("wden", "sum"), specific=("specific", "max"), txt=("rate_txt", "first")).reset_index()
    out["rate"] = np.where(out.wden > 0, out.wnum / out.wden.replace(0, np.nan), np.nan)
    # 1월 1일 세율: 시작일이 가장 이른 행(종가 있는 것 우선)
    first = df[has].sort_values(keys + ["vf"]).drop_duplicates(keys)[keys + ["adval"]].rename(columns={"adval": "rate_jan"})
    out = out.merge(first, on=keys, how="left")
    return out.drop(columns=["wnum", "wden"])


def build(df: pd.DataFrame) -> pd.DataFrame:
    w = weight_by_days(df)
    rate = w.pivot(index=["year", "hs10"], columns="rate_cd", values="rate")
    jan = w.pivot(index=["year", "hs10"], columns="rate_cd", values="rate_jan")
    spec = w.pivot(index=["year", "hs10"], columns="rate_cd", values="specific")
    txt = w.pivot(index=["year", "hs10"], columns="rate_cd", values="txt")
    t = pd.DataFrame(index=rate.index)
    for cd in ["A", "C", "F", "L", "P3", "W2", "W1", "E1", "E2", "E3"] + list(FTA):
        t[f"r_{cd}"] = rate.get(cd)
        t[f"j_{cd}"] = jan.get(cd)
    for cd in ["A", "C", "W2"]:
        t[f"spec_{cd}"] = spec.get(cd)
        t[f"txt_{cd}"] = txt.get(cd)
    for cd in ["I", "T1", "T2", "P1", "W1", "D", "G1", "G2"]:
        t[f"has_{cd}"] = rate.get(cd).notna() if cd in rate.columns else False

    # ---- 무협정 적용세율(MFN, 제50조) ----
    base = t.r_A
    domestic = t.r_P3.where(t.r_P3.notna(), t.r_L.where(t.r_L.notna(), base))
    regime = np.where(t.r_P3.notna(), "P3", np.where(t.r_L.notna(), "L", "A"))
    tier2 = t[["r_C", "r_F"]].min(axis=1)
    use2 = tier2.notna() & (domestic.isna() | (tier2 < domestic))
    mfn = domestic.where(~use2, tier2)
    regime = np.where(use2, np.where(t.r_C.notna() & (t.r_C <= t.r_F.fillna(np.inf)), "C", "F"), regime)
    # 양허 농림축산물(W2): 기본세율에 우선. 할당관세(P3)가 있으면 그것.
    usew = t.r_W2.notna() & t.r_P3.isna()
    mfn = mfn.where(~usew, t.r_W2)
    regime = np.where(usew, "W2", regime)
    t["mfn"] = mfn
    t["mfn_regime"] = regime
    # 1월 1일 기준 MFN(같은 규칙)
    domestic_j = t.j_P3.where(t.j_P3.notna(), t.j_L.where(t.j_L.notna(), t.j_A))
    tier2_j = t[["j_C", "j_F"]].min(axis=1)
    use2_j = tier2_j.notna() & (domestic_j.isna() | (tier2_j < domestic_j))
    mfn_j = domestic_j.where(~use2_j, tier2_j)
    t["mfn_jan"] = mfn_j.where(~usew, t.j_W2)

    # ---- 협정·APTA 적용세율: 무협정보다 낮을 때만 ----
    for cd, name in FTA.items():
        r = t[f"r_{cd}"]
        t[f"applied_{name}"] = np.where(r.notna() & (r < t.mfn), r, t.mfn)
    for cd, name in [("E1", "apta"), ("E2", "apta_bd"), ("E3", "apta_la")]:
        r = t[f"r_{cd}"]
        t[f"applied_{name}"] = np.where(r.notna() & (r < t.mfn), r, t.mfn)
    # 중국·인도는 FTA와 APTA 중 낮은 쪽, 베트남·라오스는 FTA와 아세안 중 낮은 쪽
    t["applied_cn"] = np.minimum(t.applied_cn, t.applied_apta)
    t["applied_in"] = np.minimum(t.applied_in, t.applied_apta)
    t["applied_vn"] = np.minimum(t.applied_vn, t.applied_asean)
    t["applied_la"] = np.minimum(t.applied_asean, t.applied_apta_la)

    # ---- 종량 하한(원/kg): MFN 규정이 선택 세율이면 종량/종가 ----
    spec_used = np.where(t.mfn_regime == "W2", t.spec_W2, np.where(t.mfn_regime == "C", t.spec_C, np.where(t.mfn_regime == "A", t.spec_A, np.nan)))
    spec_used = np.where(spec_used > 0, spec_used, np.nan)   # "N% 또는 0원"은 종량 대안이 아니다
    t["specific_won_kg"] = spec_used
    t["floor_won_kg"] = np.where((t.mfn > 0) & pd.notna(spec_used), spec_used / (t.mfn / 100.0), np.nan)
    t = t.reset_index()
    # 세율 미확정: 종량세만 있는 코드(영화필름), 두 별표에 함께 오른 코드(인삼 기타),
    # 관세화(2015) 전의 쌀(수입이 시장접근물량으로 제한되어 물량 밖 세율이 없다)
    t["undetermined_reason"] = ""
    t.loc[t.mfn.isna(), "undetermined_reason"] = "specific_only"
    t.loc[t.hs10 == "1211209900", "undetermined_reason"] = "two_annexes"
    t.loc[t.hs10.str.startswith("1006") & (t.year < 2015), "undetermined_reason"] = "rice_pre_tariffication"
    t["rate_undetermined"] = t.undetermined_reason != ""
    return t


def validate(t: pd.DataFrame) -> None:
    known = [  # (year, hs10, column, expected)
        (2025, "2103909050", "mfn", 45.0), (2025, "2103909050", "applied_cn", 44.5),
        (2025, "0904210000", "mfn", 270.0), (2025, "0904210000", "applied_cn", 270.0),
        (2025, "0710807000", "mfn", 27.0),
        (2025, "0910111000", "mfn", 377.3),
        (2025, "0710809090", "mfn", 27.0),
        (2023, "2103909090", "applied_eu", (11.2 * 181 + 8.4 * 184) / 365),
    ]
    bad = []
    for y, h, col, exp in known:
        row = t[(t.year == y) & (t.hs10 == h)]
        got = float(row[col].iloc[0]) if len(row) else np.nan
        if not np.isclose(got, exp, atol=0.05):
            bad.append((y, h, col, exp, got))
    if bad:
        for b in bad:
            print("  FAIL", b)
        raise SystemExit("알려진 값과 어긋난다")
    print(f"검증: 알려진 값 {len(known)}건 통과")
    if not KCS.exists():
        print("수입액 커버리지 검증은 KCSDB2가 없어 건너뛴다"); return
    con = duckdb.connect()
    con.execute(f"ATTACH '{KCS.as_posix()}' AS s (READ_ONLY)")
    con.register("t", t[["year", "hs10", "mfn"]])
    cov = con.sql("""
        WITH imp AS (SELECT yyyymm//100 yr, hs10, sum(imp_dlr) v FROM s.fact_trade GROUP BY 1,2)
        SELECT imp.yr, round(100.0*sum(CASE WHEN t.mfn IS NOT NULL THEN v ELSE 0 END)/sum(v),2) pct_rate,
               round(100.0*sum(CASE WHEN t.hs10 IS NOT NULL THEN v ELSE 0 END)/sum(v),2) pct_code
        FROM imp LEFT JOIN t ON t.year=imp.yr AND t.hs10=imp.hs10 GROUP BY 1 ORDER BY 1""").df()
    print("수입액 커버리지(%): 코드 있음 / 종가 실행세율 있음")
    print(cov.to_string(index=False))
    cov.to_csv(OUT / "실행세율_커버리지.csv", index=False, encoding="utf-8-sig")
    con.close()


def write_legal_sample(t: pd.DataFrame) -> None:
    """법령 대조 표본(2025년): 파일럿 코드 전부 + 별표 43개 항목의 호마다 2개 + 무작위 100개. 사람이 법령과 대조해 채우는 표."""
    import re
    rule = OUT / "분류기준_규칙_별표.csv"   # 연구 저장소의 표. 없으면 무작위 표본만 만든다
    hs4 = set()
    if rule.exists():
        r = pd.read_csv(rule, dtype=str)
        for c in ["충족 시 호", "반대편 호"]:
            for v in r[c].dropna():
                hs4 |= {h[:4] for h in re.findall(r"\d{4}", v)}
    # 파일럿 코드는 전부, 별표의 호마다 무작위 2개
    codes = {"0904210000", "0904220000", "0710807000", "2103909050", "2103909090", "0910111000", "0910112000",
             "0910113000", "0910123000", "0710809090", "0813400000", "0810909000", "0811909000", "1211209900"}
    y = t[t.year == 2025]
    pilot = y[y.hs10.isin(codes)].assign(sample="pilot")
    head = y[y.hs10.str[:4].isin(hs4) & ~y.hs10.isin(codes)].groupby(y.hs10.str[:4], group_keys=False).apply(lambda g: g.sample(min(2, len(g)), random_state=1)).assign(sample="heading")
    rest = y[~y.hs10.isin(codes) & ~y.hs10.isin(head.hs10)].sample(100, random_state=20260912).assign(sample="random")
    pair = pd.concat([pilot, head])
    cols = ["sample", "year", "hs10", "r_A", "r_C", "r_W2", "r_L", "r_P3", "mfn", "mfn_regime", "specific_won_kg", "floor_won_kg",
            "applied_cn", "applied_us", "applied_eu", "applied_asean", "applied_vn", "has_W1", "has_P1", "has_I", "has_T1", "has_T2", "rate_undetermined"]
    out = pd.concat([pair, rest])[cols].sort_values(["sample", "hs10"])
    out["법령_확인세율"] = ""
    out["확인_출처"] = ""
    out["비고"] = ""
    out.to_csv(OUT / "실행세율_법령대조.csv", index=False, encoding="utf-8-sig")
    print(f"법령 대조 표본: 파일럿 {len(pilot)} + 별표 호별 {len(head)} + 무작위 {len(rest)} → outputs/실행세율_법령대조.csv")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--load", action="store_true", help="kcstariff.duckdb 에 적재")
    args = ap.parse_args()
    df = load_rates()
    t = build(df)
    validate(t)
    OUT.mkdir(exist_ok=True)
    t.to_parquet(OUT / "fct_applied_rate.parquet", index=False)
    dim = pd.DataFrame(ORIGIN_REGIME, columns=["stat_cd", "regime", "from_year", "to_year"])
    dim.to_csv(OUT / "dim_origin_regime.csv", index=False, encoding="utf-8-sig")
    print(f"fct_applied_rate {len(t):,}행, 미확정 {int(t.rate_undetermined.sum()):,}행 {t[t.rate_undetermined].undetermined_reason.value_counts().to_dict()}, 종량 하한 있음 {int(t.floor_won_kg.notna().sum()):,}행")
    print("MFN 규정 분포:", t.mfn_regime.value_counts().to_dict())
    write_legal_sample(t)
    if args.load:
        con = duckdb.connect(str(TARIFF))
        con.execute(f"CREATE OR REPLACE TABLE fct_applied_rate AS SELECT * FROM read_parquet('{(OUT / 'fct_applied_rate.parquet').as_posix()}')")
        con.execute(f"CREATE OR REPLACE TABLE dim_origin_regime AS SELECT * FROM read_csv_auto('{(OUT / 'dim_origin_regime.csv').as_posix()}')")
        con.close()
        print("적재 완료")


if __name__ == "__main__":
    main()

## 3. 양허관세 규정 별표 1(가·나) — XLSX를 코드별 표로

논문 II.2. 「양자 중 고액(율)」 표기(`higher_of`)와 종량세로 종량 대안 코드 92개와 하한의 범위(88원~3만 4천 원)를 확인한다.

In [ ]:
# ===== 파일: research/scripts/25_parse_concession_annex.py (그대로 옮김, 4,626바이트) =====
"""
25_parse_concession_annex.py — 세계무역기구협정 등에 의한 양허관세 규정 별표 1(가·나) XLSX → 코드별 표(자료 논문 부록 A2.2).

2026-09-12에 대화형으로 만든 outputs/양허관세_별표1_2025.csv 를 2026-09-13에 스크립트로 복원했다. 복원본은 옛 CSV와 10,222행 전부 같다
(열 12개 값 비교, 부분 양허 155·종량 대안 92·두 별표 겹침 1211209900).

입력: data/external/양허관세_규정_별표1/ 의 XLSX 둘(법령센터 ZIP 안, 2024-12-31 개정본).
  가: 열 0~4 = 호(4자리)·소호(2)·세분(4)·품명·세율. 세 칸이 다 있는 행이 10단위 코드 행이고, 품명이 「-」로 시작하는 행은 바로 위 코드의
      세부 품목에만 양허한 「부분 양허」(품명 : 세율)다. 호·소호만 있는 행은 제목이라 버린다.
  나: 열 0~6 = 앞 6자리·2·2·품명·시장접근물량·물량 이내 세율·물량 초과 세율. 뒤 두 칸이 다 있는 행이 10단위 코드 행이다.
세율 문구는 그대로 두고(rate_txt) 종가(%)·종량(원/kg)·「양자 중 고액」 여부를 따로 읽는다. 가·나 양쪽에 있는 코드는 미확정으로 표시한다.
"""
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
EXT = ROOT / "data" / "external" / "양허관세_규정_별표1"
GA = EXT / "1-1. 세계무역기구협정 등에 의한 양허관세 규정 별표 1의 가.xlsx"
NA = EXT / "1-2. 세계무역기구협정 등에 의한 양허관세 규정 별표 1의 나.xlsx"
OUT = ROOT / "outputs" / "양허관세_별표1_2025.csv"


def cell(v, code=False) -> str:
    """code=True: 코드 칸(숫자 셀 1000.0 → '1000'). 세율 칸은 숫자 셀을 '18.0' 꼴 그대로 둔다."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    s = str(v).strip()
    return s[:-2] if (code and re.fullmatch(r"\d+\.0", s)) else s


def parse_rate(txt: str):
    m = re.search(r"(\d+(?:\.\d+)?)\s*%", txt)
    adval = float(m.group(1)) if m else (float(txt) if re.fullmatch(r"\d+(\.\d+)?", txt) else np.nan)
    m2 = re.search(r"([\d,]+(?:\.\d+)?)\s*원", txt)
    spec = float(m2.group(1).replace(",", "")) if m2 else np.nan
    return adval, spec, ("양자 중 고액" in txt)


def parse_ga() -> list[dict]:
    rows = []
    for r in pd.read_excel(GA, header=None, dtype=object).itertuples(index=False):
        c0, c1, c2 = (cell(v, code=True) for v in r[:3]); name, rate = (cell(v) for v in r[3:5])
        if re.fullmatch(r"\d{4}", c0) and re.fullmatch(r"\d{2}", c1) and re.fullmatch(r"\d{4}", c2):
            rows.append(dict(byeolpyo="1의 가", hs10=c0 + c1 + c2, name_ko=name, trq="", in_quota="", rate_txt=rate, partial=[]))
        elif name.startswith("-") and rows:
            rows[-1]["partial"].append(f"{name} : {rate}")   # 세율 칸이 비면 「품명 : 」로 남긴다(옛 산출과 같은 꼴)
    return rows


def parse_na() -> list[dict]:
    rows = []
    for r in pd.read_excel(NA, header=None, dtype=object).itertuples(index=False):
        c0, c1, c2 = (cell(v, code=True) for v in r[:3]); name, trq, inq, rate = (cell(v) for v in r[3:7])
        if re.fullmatch(r"\d{6}", c0) and re.fullmatch(r"\d{2}", c1) and re.fullmatch(r"\d{2}", c2):
            rows.append(dict(byeolpyo="1의 나", hs10=c0 + c1 + c2, name_ko=name, trq=trq, in_quota=inq, rate_txt=rate, partial=[]))
    return rows


def build() -> pd.DataFrame:
    out = []
    for d in parse_ga() + parse_na():
        adval, spec, hi = parse_rate(d["rate_txt"])
        out.append(dict(byeolpyo=d["byeolpyo"], hs10=d["hs10"], name_ko=d["name_ko"], trq=d["trq"], in_quota=d["in_quota"], rate_txt=d["rate_txt"],
                        adval=adval, specific_won_kg=spec, higher_of=hi, partial=" | ".join(d["partial"])))
    df = pd.DataFrame(out)
    both = set(df[df.byeolpyo == "1의 가"].hs10) & set(df[df.byeolpyo == "1의 나"].hs10)
    df["rate_undetermined"] = np.where(df.hs10.isin(both), "Y", "")
    df["undetermined_reason"] = np.where(df.hs10.isin(both), "별표 1의 가와 나에 모두 있어 물품 정의에 따라 세율이 갈림", "")
    return df


def main() -> None:
    df = build()
    df.to_csv(OUT, index=False, encoding="utf-8-sig")
    print(f"{OUT.name}: {len(df):,}행 {df.byeolpyo.value_counts().to_dict()}, 부분 양허 {int((df.partial != '').sum())}, 종량 대안 코드 {df[df.higher_of].hs10.nunique()}, 두 별표 겹침 {int((df.rate_undetermined == 'Y').sum())}행")


if __name__ == "__main__":
    main()

## 4. 한국은행 ECOS — 원/달러 매매기준율 월평균

논문 III.1. 신고 단가(달러/kg)를 원으로 바꾸어 종량 하한과 비교하는 환율.

In [ ]:
# ===== 파일: research/scripts/26_fetch_ecos_fx.py (그대로 옮김, 2,599바이트) =====
"""
26_fetch_ecos_fx.py — 한국은행 ECOS 731Y001(주요국 통화의 대원화 환율)에서 원/미국달러 매매기준율을 일별로 받아 월평균을 만든다(자료 논문 부록 A3.3).

2026-09-12에 대화형으로 만든 outputs/환율_월별_USDKRW.csv 를 2026-09-13에 스크립트로 복원했다. 복원본은 옛 CSV와 237개월 전부 같다(소수 2자리).
이 통계표는 월별 주기 조회가 없고 일별만 있으므로 해마다 일별 전부를 받아(한 해 1,000행 상한 안) 달마다 평균한다.
항목 코드는 StatisticItemList 로 확인한다(0000001 = 원/미국달러(매매기준율)). 키는 KCSDB2/config/api_key.env 의 ECOS_API_KEY (저장소에 올리지 않는다).
"""
from __future__ import annotations

import os
import time
from pathlib import Path

import pandas as pd
import requests

ROOT = Path(__file__).resolve().parents[1]
KCSDB2 = Path(os.environ.get("KCSDB2_ROOT", r"C:\Work\Projects\KCSDB2"))
OUT = ROOT / "outputs" / "환율_월별_USDKRW.csv"
STAT = "731Y001"


def api_key() -> str:
    for line in open(KCSDB2 / "config" / "api_key.env", encoding="utf-8"):
        if line.startswith("ECOS_API_KEY"):
            return line.split("=", 1)[1].strip().strip('"').strip("'")
    raise SystemExit("ECOS_API_KEY 없음")


def main(y0: int = 2007, y1: int = 2026) -> None:
    key = api_key()
    items = requests.get(f"https://ecos.bok.or.kr/api/StatisticItemList/{key}/json/kr/1/100/{STAT}", timeout=60).json()["StatisticItemList"]["row"]
    code = next(it["ITEM_CODE"] for it in items if "미국달러" in it["ITEM_NAME"])
    rows = []
    for y in range(y0, y1 + 1):
        for start in (1, 1001):
            r = requests.get(f"https://ecos.bok.or.kr/api/StatisticSearch/{key}/json/kr/{start}/{start + 999}/{STAT}/D/{y}0101/{y}1231/{code}", timeout=60).json()
            data = r.get("StatisticSearch", {}).get("row", [])
            rows += [(d["TIME"], float(d["DATA_VALUE"])) for d in data if d.get("DATA_VALUE") not in (None, "", "-")]
            if len(data) < 1000:
                break
            time.sleep(0.3)
        time.sleep(0.3)
    fx = pd.DataFrame(rows, columns=["date", "rate"]).drop_duplicates("date")
    fx["yyyymm"] = fx.date.str[:6].astype(int)
    mon = fx.groupby("yyyymm").rate.mean().round(2).reset_index().rename(columns={"rate": "krw_per_usd"})
    mon["year"] = mon.yyyymm // 100
    mon.to_csv(OUT, index=False, encoding="utf-8-sig")
    print(f"{OUT.name}: {len(mon)}개월 ({mon.yyyymm.min()}~{mon.yyyymm.max()}), 일별 {len(fx):,}행")


if __name__ == "__main__":
    main()

## 5. 유통이력관리 고시 별표 1 — HWP 5.0·HWPX 텍스트 읽기

논문 II.3. 관세청(2009~2022)·농림축산식품부·해양수산부 고시의 판본 48개에서 항목별 품목번호와 지정기간을 뽑는다. 표(판본별·지정 구간·통합 구간 237개, 코드 155개)는 이 텍스트에서 대화형으로 만들었다.

In [ ]:
import olefile, zlib, struct, zipfile, re
from pathlib import Path


def file_kind(path: Path) -> str:
    head = path.read_bytes()[:4]
    return "hwp" if head == b"\xd0\xcf\x11\xe0" else ("hwpx" if head[:2] == b"PK" else ("pdf" if head == b"%PDF" else "unknown"))


def hwp_text(path):
    """HWP 5.0(OLE): FileHeader 36번째 바이트의 최하위 비트가 압축 여부, BodyText/Section<n>의 PARA_TEXT(태그 67) 레코드가 문단."""
    ole = olefile.OleFileIO(str(path))
    compressed = bool(ole.openstream("FileHeader").read()[36] & 1)
    out = []
    for entry in sorted(ole.listdir()):
        if entry[0] != "BodyText":
            continue
        data = ole.openstream(entry).read()
        if compressed:
            data = zlib.decompress(data, -15)
        i = 0
        while i + 4 <= len(data):
            head = struct.unpack("<I", data[i:i + 4])[0]
            tag, size = head & 0x3FF, (head >> 20) & 0xFFF
            i += 4
            if size == 0xFFF:
                size = struct.unpack("<I", data[i:i + 4])[0]
                i += 4
            if tag == 67:                       # PARA_TEXT
                out.append(data[i:i + size].decode("utf-16le", errors="ignore"))
            i += size
    return "\n".join(re.sub(r"[\x00-\x1f]", " ", t) for t in out)


def hwpx_text(path):
    """HWPX(ZIP): Contents/section<n>.xml 의 <hp:t>…</hp:t> 문자열을 순서대로."""
    z = zipfile.ZipFile(path)
    out = []
    for name in sorted(z.namelist()):
        if re.match(r"Contents/section\d+\.xml", name):
            xml = z.read(name).decode("utf-8")
            out.append(" ".join(re.sub(r"<[^>]+>", "", t) for t in re.findall(r"<hp:t[^>]*>(.*?)</hp:t>", xml, re.S)))
    return "\n".join(out)


if __name__ == "__main__":
    base = Path(r"research/data/external/유통이력관리_고시_별표1")
    for p in sorted(base.glob("*.hwp*")):
        kind = file_kind(p)
        text = hwp_text(p) if kind == "hwp" else hwpx_text(p) if kind == "hwpx" else ""
        print(p.name, kind, len(text), "자")

## 6. 사전세액심사 대상물품 — 페이지 컨텍스트 `fetch`

논문 II.3. 관세법령정보포털의 조회 화면 `openULS0105013Q.do`를 브라우저로 연 뒤 같은 출처에서 `retrieveBtaaTrgtCmdt.do`를 2016년 1월부터 2026년 9월까지 매월 1일 기준으로 129회 보낸다(브라우저 밖 `requests`에는 빈 응답). 결과를 월별 CSV와 코드 48개의 이력 CSV(지정일·처음과 마지막으로 보인 달)로 옮기는 것은 대화형이었다.

```javascript
const out = [];
for (let y = 2016; y <= 2026; y++) for (let m = 1; m <= 12; m++) {
  const d = `${y}${String(m).padStart(2, "0")}01`;
  const r = await fetch("/clip/lworsrch/retrieveBtaaTrgtCmdt.do",
    { method: "POST", headers: { "Content-Type": "application/x-www-form-urlencoded" }, body: "aplyDt=" + d });
  const j = await r.json();
  for (const x of [...(j.items1 || []), ...(j.items2 || [])])
    for (const s of ["1", "2"])
      if (x["hsSgn" + s]) out.push([d, x["hsSgn" + s], (x["hsSgnNm" + s] || "").trim(), x["aplyStrtDt" + s], x["btaaSelcBaseCd" + s], x["btaaSelcBaseNm" + s]]);
}
JSON.stringify(out);
```

## 7. 코드 쌍 — 품목분류 적용기준 규칙 별표와 같은 HS6 안의 세율 차이

논문 II.4·III.2. 확정 쌍 444개의 코드 254개가 패널 코드와 사건연구 대조군의 바탕이고, 변환비가 붙은 쌍(건고추·냉동고추, 건대추·냉동대추)의 단가 비율을 IV.1이 쓴다.

In [ ]:
# ===== 파일: research/scripts/16_build_code_pairs.py (그대로 옮김, 13,593바이트) =====
"""
16_build_code_pairs.py — 첫째 질문의 관측 단위인 코드 쌍(물리적으로 비슷하면서 세율이 다른 HS10 두 개)의
목록을 만든다. 연구 문서 III.3절.

두 출처:
  별표  품목분류 적용기준 규칙 별표 43개 항목의 충족 호·반대편 호를 HS10으로 편다(ITEMS에 손으로 적은 범위·키워드).
        조합이 CAP을 넘는 항목은 쌍을 만들지 않고 양쪽 코드 목록만 후보 표에 남긴다(손으로 고른다).
  규칙  같은 HS6 안에서 2025년 실행세율(mfn)이 다른 코드끼리. 품명 어휘를 상태·가공·용도·소매·기타로 표시한다.

쌍마다: 2025년 격차, 원산지별(중국·미국·EU·아세안·베트남·인도) 격차의 연도 표준편차(식별이 작동하는 원산지),
두 코드가 함께 존재한 연도 범위, 세율 미확정 표시, 2022~2025년 수입액과 중국 몫, 신설 연도·사유.
손으로 채우는 열: 확인(Y/N), 변환비, 비고.

산출: research/outputs/코드쌍_목록.csv, 코드쌍_별표_후보.csv
"""
from __future__ import annotations

import itertools
import re
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

import os
ROOT = Path(__file__).resolve().parents[1]                      # KCSTARIFF/research
REPO = ROOT.parent                                               # KCSTARIFF
KCSDB2 = Path(os.environ.get("KCSDB2_ROOT", r"C:\Work\Projects\KCSDB2"))  # 무역통계 저장소(읽기 전용 의존)
TARIFF = REPO / "data" / "processed" / "kcstariff.duckdb"
KCS = KCSDB2 / "data" / "processed" / "kcsdb.duckdb"
OUT = ROOT / "outputs"
BYEOLPYO_DIR = KCSDB2 / "data" / "external" / "HSK_별표"
BYEOLPYO = BYEOLPYO_DIR / "HSK_별표_2025.csv"
ORIG = ["cn", "us", "eu", "asean", "vn", "in"]
CAP = 40          # 별표 항목 하나의 쌍 조합 상한
CAP_ITEM = {35: 200}   # 항목별 예외
MIN_MUSD = 1.0    # 규칙 쌍: HS6 합계 수입액(2022~25, 백만 달러) 하한
MAX_CODES = 12    # 규칙 쌍: HS6 안 코드 수 상한

# 별표 항목: (번호, 품명, [A쪽 접두 목록], A 키워드(정규식, None=전부), [B쪽 접두], B 키워드)
# A = 충족 시 호, B = 반대편 호. 반대편이 없거나 「구성 성분별」인 항목은 뺐다(12·28·37~41·43).
ITEMS = [
    (1, "소뼈와 돼지뼈", ["0201", "0202", "0203"], "뼈", ["0506"], None),
    (2, "염장·염수장 쇠고기", ["0210"], "소|쇠", ["0201", "0202"], None),
    (3, "염장·염수장 수산물", ["0305", "0306", "0307"], "염장|염수|소금|젓", ["0302", "0303"], None),
    (4, "조기", ["0303"], "조기", ["0305"], "조기"),
    (5, "오징어", ["0307"], "오징어", ["1605"], "오징어"),
    (6, "밀크와 크림(농축·가당)", ["0402"], None, ["1901", "2106"], "분유|밀크|크림|유"),
    (7, "싹을 틔운 대두", ["070999"], None, ["1201"], None),
    (8, "냉동고추", ["0710807000"], None, ["090421", "090422"], None),
    (9, "염수로 일시 보존한 채소", ["0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709"], None, ["0711"], None),
    (10, "라면스프 제조용 건조채소 혼합물", ["0712"], None, ["0904"], None),
    (11, "냉동대추", ["0811"], "대추", ["0813"], "대추"),
    (13, "카사바 분말과 전분", ["110620"], None, ["110814"], None),
    (14, "고구마 분말과 전분", ["110620"], None, ["1108191000"], None),
    (15, "부순 대두와 가루상 대두", ["120190"], None, ["120810"], None),
    (16, "부순 참깨와 가루상 참깨", ["120740"], None, ["120890"], None),
    (17, "비타민E 첨가 유지", ["1517"], None, ["1507", "1508", "1509", "1510", "1511", "1512", "1513", "1514", "1515"], None),
    (18, "견과류 함유 설탕과자", ["1704"], None, ["2008"], "견과|땅콩|아몬드|호두|밤|캐슈|피스타치오|헤이즐|마카다미아"),
    (19, "감자전분 조제품", ["1901"], None, ["1108130000"], None),
    (20, "전분 조제품(감자 제외)", ["1901"], None, ["1108"], None),
    (21, "찌거나 삶은 곡물", ["1904"], "찌거나|삶", ["1006"], None),
    (22, "볶은 곡물", ["1904", "2101"], "볶", ["1006", "1003", "1005"], None),
    (23, "튀긴 쌀", ["1904"], "튀긴|튀김", ["1006"], None),
    (24, "식초·초산으로 조제한 채소", ["2001"], None, ["0711"], None),
    (25, "감자가루·플레이크 조제품", ["2005"], "감자", ["1105"], None),
    (26, "찌거나 삶아서 건조한 팥", ["2005"], "팥", ["071332"], None),
    (27, "설탕으로 보존처리한 과실", ["2006"], None, ["2008"], None),
    (29, "볶은 땅콩", ["2008"], "땅콩", ["1202"], None),
    (30, "찌거나 삶은 대두", ["2008"], "대두|콩", ["1201"], None),
    (31, "볶거나 튀긴 대두", ["2008"], "대두|콩", ["1201"], None),
    (32, "볶은 헤이즐넛·피스타치오·마카다미아", ["200819"], None, ["080221", "080222", "080251", "080252", "080261", "080262"], None),
    (33, "고추다진양념", ["2103909050", "2103909090"], None, ["090421", "090422"], None),
    (34, "고추장 제조용 고춧가루 혼합조미료", ["2103"], "혼합조미료|고추장", ["0904220000"], None),
    (35, "변성전분 조제품", ["210690"], None, ["3505103000", "3505104010", "3505105010", "3505109010"], None),
    (36, "로열젤리 첨가 천연꿀", ["2106"], "로열|꿀", ["0409"], None),
    (42, "백잠사", ["5002001"], None, ["5002009000"], None),
]

VOCAB = [
    ("상태", r"냉동|건조|신선|냉장|염장|염수|훈제|말린|얼린|생|건|살아 있는"),
    ("가공", r"조제|볶|가공|분말|가루|전분|혼합|분획|정제|삶|찌|튀긴|부순|잘게|절단|껍질|탈지|농축|가당|발효"),
    ("용도", r"용(?![가-힣])|용의|용으로|제조용|사료용|식용|공업용|의료용|등록된|한정한다|법률|법에"),
    ("소매", r"소매|포장"),
    ("기타", r"^기타$|^그 밖의"),
]


def tier(v: str) -> int:
    """손 확인 우선순위: 1 상태·가공 어휘(물리적 형태), 2 기타만, 3 용도, 4 없음."""
    if "상태" in v or "가공" in v:
        return 1
    if v == "기타":
        return 2
    if "용도" in v:
        return 3
    return 4


def vocab_tag(a: str, b: str) -> str:
    tags = []
    for name, pat in VOCAB:
        if re.search(pat, a) or re.search(pat, b):
            tags.append(name)
    return "·".join(tags) if tags else "없음"


def load() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    con = duckdb.connect()
    con.execute(f"ATTACH '{TARIFF.as_posix()}' AS tr (READ_ONLY)")
    con.execute(f"ATTACH '{KCS.as_posix()}' AS s (READ_ONLY)")
    rate = con.sql("SELECT year, hs10, mfn, rate_undetermined, " + ", ".join(f"applied_{o}" for o in ORIG) + " FROM tr.fct_applied_rate").df()
    imp = con.sql("""
        SELECT hs10, sum(imp_dlr)/1e6 musd, sum(CASE WHEN stat_cd='CN' THEN imp_dlr ELSE 0 END)/nullif(sum(imp_dlr),0) cn_share,
               sum(imp_wgt)/1e3 ton
        FROM s.fact_trade WHERE yyyymm BETWEEN 202201 AND 202512 GROUP BY 1""").df()
    rev = con.sql("SELECT hs10, rev, effective_ym, reason FROM s.dim_hsk_revision WHERE change='신설'").df()
    rev = rev.sort_values("effective_ym").drop_duplicates("hs10", keep="last")
    con.close()
    names = pd.read_csv(BYEOLPYO, dtype=str).rename(columns={"code": "hs10"})
    names["leaf"] = names.leaf.str.replace(r"\s+", " ", regex=True)
    names["path"] = names.path.fillna("").str.replace(r"\s+", " ", regex=True)
    return rate, imp, rev, names


def code_table(rate: pd.DataFrame, imp: pd.DataFrame, rev: pd.DataFrame, names: pd.DataFrame) -> pd.DataFrame:
    r25 = rate[rate.year == 2025][["hs10", "mfn", "rate_undetermined"]]
    span = rate.groupby("hs10").agg(first_year=("year", "min"), last_year=("year", "max"), undet_any=("rate_undetermined", "max")).reset_index()
    t = names.merge(r25, on="hs10", how="left").merge(span, on="hs10", how="left").merge(imp, on="hs10", how="left").merge(rev, on="hs10", how="left")
    t["musd"] = t.musd.fillna(0).round(2)
    t["ton"] = t.ton.fillna(0).round(1)
    t["cn_share"] = t.cn_share.round(3)
    return t


def gap_by_origin(rate: pd.DataFrame, h: str, l: str) -> dict:
    a = rate[rate.hs10 == h].set_index("year")
    b = rate[rate.hs10 == l].set_index("year")
    yrs = a.index.intersection(b.index)
    out = {"years_both": f"{yrs.min()}~{yrs.max()}" if len(yrs) else "", "n_years": len(yrs)}
    for o in ORIG:
        g = (a.loc[yrs, f"applied_{o}"] - b.loc[yrs, f"applied_{o}"]).dropna()
        out[f"gap_{o}_2025"] = round(float(g.get(2025, np.nan)), 1) if 2025 in g.index else np.nan
        out[f"gap_{o}_sd"] = round(float(g.std()), 1) if len(g) > 1 else np.nan
    out["origins_varying"] = "·".join(o for o in ORIG if pd.notna(out[f"gap_{o}_sd"]) and out[f"gap_{o}_sd"] > 1.0)
    return out


def make_pair(t: pd.DataFrame, rate: pd.DataFrame, h: str, l: str, source: str, item_no, item_name) -> dict:
    H = t.set_index("hs10").loc[h]
    L = t.set_index("hs10").loc[l]
    d = {"source": source, "item_no": item_no, "item": item_name,
         "hs10_high": h, "name_high": H.leaf, "path_high": H.path[:60], "mfn_high": H.mfn,
         "hs10_low": l, "name_low": L.leaf, "path_low": L.path[:60], "mfn_low": L.mfn,
         "gap_mfn_2025": round(float(H.mfn - L.mfn), 1) if pd.notna(H.mfn) and pd.notna(L.mfn) else np.nan,
         "same_hs6": h[:6] == l[:6], "vocab": vocab_tag(str(H.leaf), str(L.leaf)), "tier": tier(vocab_tag(str(H.leaf), str(L.leaf))),
         "musd_high": H.musd, "musd_low": L.musd, "cn_share_high": H.cn_share, "cn_share_low": L.cn_share,
         "undetermined": bool(H.undet_any) or bool(L.undet_any),
         "new_high": f"{H.rev}({H.reason})" if pd.notna(H.rev) else "", "new_low": f"{L.rev}({L.reason})" if pd.notna(L.rev) else ""}
    d.update(gap_by_origin(rate, h, l))
    d.update({"확인": "", "변환비": "", "비고": ""})
    return d


def side_codes(t: pd.DataFrame, prefixes: list[str], kw: str | None) -> pd.DataFrame:
    m = t[t.hs10.str.startswith(tuple(prefixes))]
    if kw:
        m = m[m.leaf.str.contains(kw, regex=True) | m.path.str.contains(kw, regex=True)]
    return m


def annex_pairs(t: pd.DataFrame, rate: pd.DataFrame) -> tuple[list[dict], list[dict]]:
    pairs, cands = [], []
    for no, name, pa, ka, pb, kb in ITEMS:
        A, B = side_codes(t, pa, ka), side_codes(t, pb, kb)
        n = len(A) * len(B)
        for side, df in (("A", A), ("B", B)):
            for _, r in df.iterrows():
                cands.append({"item_no": no, "item": name, "side": side, "hs10": r.hs10, "leaf": r.leaf, "path": r.path[:60],
                              "mfn": r.mfn, "musd": r.musd, "cn_share": r.cn_share, "n_combos": n, "paired": n <= CAP_ITEM.get(no, CAP)})
        if n == 0 or n > CAP_ITEM.get(no, CAP):
            continue
        for a, b in itertools.product(A.hs10, B.hs10):
            ma, mb = t.set_index("hs10").mfn.get(a), t.set_index("hs10").mfn.get(b)
            if pd.isna(ma) or pd.isna(mb) or ma == mb:
                continue
            h, l = (a, b) if ma > mb else (b, a)
            pairs.append(make_pair(t, rate, h, l, "별표", no, name))
    return pairs, cands


def rule_pairs(t: pd.DataFrame, rate: pd.DataFrame) -> list[dict]:
    x = t[t.mfn.notna()].assign(hs6=t.hs10.str[:6])
    g = x.groupby("hs6").agg(n=("hs10", "size"), nr=("mfn", "nunique"), musd=("musd", "sum"))
    keep = g[(g.nr > 1) & (g.n <= MAX_CODES) & (g.musd >= MIN_MUSD)].index
    out = []
    for hs6, grp in x[x.hs6.isin(keep)].groupby("hs6"):
        for a, b in itertools.combinations(grp.itertuples(index=False), 2):
            if a.mfn == b.mfn:
                continue
            h, l = (a, b) if a.mfn > b.mfn else (b, a)
            out.append(make_pair(t, rate, h.hs10, l.hs10, "규칙", "", ""))
    return out


def main() -> None:
    rate, imp, rev, names = load()
    t = code_table(rate, imp, rev, names)
    ap, cands = annex_pairs(t, rate)
    rp = rule_pairs(t, rate)
    pairs = pd.DataFrame(ap + rp)
    # 별표 쌍과 겹치는 규칙 쌍은 별표 쪽만 남긴다
    key = pairs.hs10_high + "-" + pairs.hs10_low
    dup = key.duplicated(keep="first") & (pairs.source == "규칙")
    pairs = pairs[~dup].reset_index(drop=True)
    pairs.insert(0, "pair_id", [f"P{i+1:04d}" for i in range(len(pairs))])
    OUT.mkdir(exist_ok=True)
    pairs.to_csv(OUT / "코드쌍_목록.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(cands).to_csv(OUT / "코드쌍_별표_후보.csv", index=False, encoding="utf-8-sig")
    print(f"쌍 {len(pairs)}개: 별표 {int((pairs.source=='별표').sum())}, 규칙 {int((pairs.source=='규칙').sum())} (규칙 중복 제거 {int(dup.sum())})")
    print("별표 항목별 쌍 수:", pairs[pairs.source == "별표"].groupby("item_no").size().to_dict())
    big = pd.DataFrame(cands).drop_duplicates(["item_no"]).query("n_combos > @CAP")[["item_no", "item", "n_combos"]]
    print("조합이 상한을 넘어 손으로 고를 항목:", big.to_dict("records"))
    print("규칙 쌍 우선순위(1 상태·가공/2 기타/3 용도/4 없음):", pairs[pairs.source == "규칙"].tier.value_counts().sort_index().to_dict())
    print("규칙 쌍 어휘 분포:", pairs[pairs.source == "규칙"].vocab.value_counts().head(12).to_dict())
    print("격차 ≥100%p 쌍:", int((pairs.gap_mfn_2025 >= 100).sum()), " 격차 움직이는 원산지 있음:", int((pairs.origins_varying != "").sum()))
    print("미확정 포함 쌍:", int(pairs.undetermined.sum()))


if __name__ == "__main__":
    main()

## 8. 품목 상세 화면 — 코드 쌍 후보 285개의 그 밖 FTA 세율과 사전세액심사 표시

논문 II.1·II.3. 코드×연도마다 한 번, 285코드 × 2012~2026년 = 4,275쪽.

In [ ]:
# ===== 파일: research/scripts/17_fetch_item_detail.py (그대로 옮김, 5,826바이트) =====
"""
17_fetch_item_detail.py — 관세법령정보포털 품목 상세(openULS0201007Q)에서 코드 쌍에 든 코드의
연도별 세율 전부(그 밖 FTA 포함)와 부가 정보(사전세액심사 대상 표시, 유통이력 신고대상 지정기간,
분류사례·원산지결정기준 건수)를 받는다. 상세 화면은 2012년부터만 세율을 준다.

입력: research/outputs/코드쌍_코드목록.txt (한 줄에 HS10 하나)
캐시: data/raw/clip_item_detail/<year>/<hs10>.html
산출: research/outputs/품목상세_세율_2012_2026.csv (연도×코드×구분기호, 긴 형태)
      research/outputs/품목상세_부가정보_2012_2026.csv
세율 문구에 " / "가 있으면 연중 구간이 여럿이다(구간 날짜는 화면에 없다). 첫 구간과 마지막 구간을 따로 둔다.
"""
from __future__ import annotations

import argparse
import html
import re
import time
from pathlib import Path

import pandas as pd
import requests

ROOT = Path(__file__).resolve().parents[1]                      # KCSTARIFF/research
RAW = ROOT / "data" / "raw" / "clip_item_detail"
OUT = ROOT / "outputs"
CODES = OUT / "코드쌍_코드목록.txt"
BASE = "https://unipass.customs.go.kr/clip/hsinfosrch/openULS0201007Q.do"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)", "Referer": "https://unipass.customs.go.kr/clip/index.do"}
YEARS = range(2012, 2027)
ERROR_MARK = "프로그램 오류발생"


def strip(x: str) -> str:
    return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", x))).strip()


def fetch(sess: requests.Session, code: str, year: int) -> str:
    r = sess.post(BASE, data={"searchVal": code, "aplyYy": str(year)}, timeout=60)
    r.raise_for_status()
    if ERROR_MARK in r.text:
        raise requests.RequestException("오류 안내문")
    return r.text


def parse_rate(txt: str) -> tuple[float | None, float | None]:
    """'270% 또는 6,210원' → (270.0, 6210.0); '240원' → (None, 240.0); '0%' → (0.0, None)."""
    a = re.search(r"([\d.]+)\s*%", txt)
    sp = re.search(r"([\d,]+(?:\.\d+)?)\s*원", txt)
    return (float(a.group(1)) if a else None, float(sp.group(1).replace(",", "")) if sp else None)


def parse(page: str, code: str, year: int) -> tuple[list[dict], dict]:
    rows = []
    i = page.find("세율적용 우선순위")
    j = page.find("내국세", i) if i > 0 else -1
    seg = page[i:j] if i > 0 and j > i else ""
    for tr in re.findall(r"<tr[^>]*>(.*?)</tr>", seg, re.S):
        c = [strip(x) for x in re.findall(r"<t[dh][^>]*>(.*?)</t[dh]>", tr, re.S)]
        if len(c) >= 3 and re.fullmatch(r"[A-Z][A-Z0-9]*", c[0]):
            segs = [s.strip() for s in c[1].split(" / ")]
            a1, s1 = parse_rate(segs[0])
            aN, sN = parse_rate(segs[-1])
            rows.append(dict(year=year, hs10=code, rate_cd=c[0], rate_name=c[2], rate_txt=c[1], n_seg=len(segs),
                             adval_first=a1, specific_first=s1, adval_last=aN, specific_last=sN))
    text = strip(re.sub(r"<script.*?</script>|<style.*?</style>", "", page, flags=re.S))
    m_yu = re.search(r"유통이력 신고대상물품명, 지정기간, 통보기관 (.*?) 연관정보", text)
    yu = m_yu.group(1).strip() if m_yu else ""
    if yu.startswith("There were no results"):
        yu = ""
    cnt = {k: int(m.group(1)) if (m := re.search(k + r"\s*(\d+)건", text)) else None
           for k in ["분류사례", "평가사례", "원산지결정기준", "판례·결정례"]}
    extra = dict(year=year, hs10=code, pre_assessment="사전세액대상물품" in text, deposit_price="담보기준가격" in text,
                 distribution_history=yu, n_class_case=cnt["분류사례"], n_valuation_case=cnt["평가사례"],
                 n_origin_rule=cnt["원산지결정기준"], n_ruling=cnt["판례·결정례"], n_rates=len(rows))
    return rows, extra


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--delay", type=float, default=0.5)
    ap.add_argument("--parse-only", action="store_true")
    ap.add_argument("--codes", nargs="*", help="지정하면 이 코드만")
    args = ap.parse_args()
    codes = args.codes or [c.strip() for c in CODES.read_text().splitlines() if c.strip()]
    sess = requests.Session()
    sess.headers.update(HEADERS)
    rates, extras, failed = [], [], []
    n = 0
    for code in codes:
        for year in YEARS:
            f = RAW / str(year) / f"{code}.html"
            if f.exists():
                page = f.read_text(encoding="utf-8")
            elif args.parse_only:
                continue
            else:
                try:
                    page = fetch(sess, code, year)
                except Exception as e:  # noqa: BLE001
                    failed.append((code, year, str(e)[:80]))
                    time.sleep(args.delay * 4)
                    continue
                f.parent.mkdir(parents=True, exist_ok=True)
                f.write_text(page, encoding="utf-8")
                time.sleep(args.delay)
            r, x = parse(page, code, year)
            rates.extend(r)
            extras.append(x)
        n += 1
        if n % 20 == 0:
            print(f"{n}/{len(codes)} 코드, 세율 행 {len(rates):,}, 실패 {len(failed)}", flush=True)
    pd.DataFrame(rates).to_csv(OUT / "품목상세_세율_2012_2026.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(extras).to_csv(OUT / "품목상세_부가정보_2012_2026.csv", index=False, encoding="utf-8-sig")
    print(f"완료: 코드 {len(codes)}, 세율 행 {len(rates):,}, 부가정보 {len(extras):,}, 실패 {len(failed)}")
    if failed:
        pd.DataFrame(failed, columns=["hs10", "year", "err"]).to_csv(OUT / "품목상세_실패.csv", index=False, encoding="utf-8-sig")
        print(failed[:10])


if __name__ == "__main__":
    main()

## 9. 신설 코드 처리군 — 2022·2025년 개정의 감시형·세율형과 짝

논문 II.4·III.2. 패널 코드 602개에 들어가고 사건연구 대조군에서는 뺀다.

In [ ]:
# ===== 파일: research/scripts/18_build_treatments.py (그대로 옮김, 8,257바이트) =====
"""
18_build_treatments.py — 셋째 질문의 처리군 표를 만든다(연구 문서 III.2절).

신설 코드(2022·2025 개정)를 세 유형으로 나눈다.
  감시형   개정 사유가 감시 목적(관계부처 요청, 관리·모니터링·둔갑 방지·수입량 관리)이고
           실행세율(mfn)이 분리한 코드(전신)와 같은 것.
  세율형   신설 코드의 mfn이 전신 코드의 mfn(가중평균)과 다른 것. 사유와 무관. 감시형과 겹치면 세율형.
  기계적 짝  세분이 분리한 뒤 남은 몫을 받은 기타호. 처리군이 아니라 감시형·세율형의 짝.
전신 코드: 2022년은 dim_hs10_concordance(revision 2022)의 hs_from, 2025년은 같은 8단위 안에서
2025년에 폐지된 코드, 없으면 같은 8단위(없으면 6단위)에 남은 기타 코드.
집행 지정(사전세액심사·유통이력)은 별도 파일이 이미 있어 여기서는 코드별 지정일만 붙인다.

산출: research/outputs/처리군_목록.csv
"""
from __future__ import annotations

import re
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

import os
ROOT = Path(__file__).resolve().parents[1]                      # KCSTARIFF/research
REPO = ROOT.parent                                               # KCSTARIFF
KCSDB2 = Path(os.environ.get("KCSDB2_ROOT", r"C:\Work\Projects\KCSDB2"))  # 무역통계 저장소(읽기 전용 의존)
TARIFF = REPO / "data" / "processed" / "kcstariff.duckdb"
KCS = KCSDB2 / "data" / "processed" / "kcsdb.duckdb"
OUT = ROOT / "outputs"
BYEOLPYO_DIR = KCSDB2 / "data" / "external" / "HSK_별표"
BY = BYEOLPYO_DIR
WATCH = r"관계부처|관리|모니터링|둔갑|수입량|언론|기준규칙|관세청"
NOT_WATCH = r"신성장|현행화|범위수정|업계|기타호 신설|세분류체계"


def load():
    con = duckdb.connect()
    con.execute(f"ATTACH '{KCS.as_posix()}' AS s (READ_ONLY)")
    con.execute(f"ATTACH '{TARIFF.as_posix()}' AS tr (READ_ONLY)")
    rev = con.sql("SELECT rev, effective_ym, hs10, change, reason, name_ko FROM s.dim_hsk_revision WHERE rev IN ('2022','2025')").df()
    conc = con.sql("SELECT hs_from, hs_to, weight FROM s.dim_hs10_concordance WHERE revision='2022'").df()
    rate = con.sql("SELECT year, hs10, mfn, rate_undetermined FROM tr.fct_applied_rate WHERE year BETWEEN 2021 AND 2026").df()
    imp = con.sql("""
        WITH t AS (SELECT hs10, stat_cd, sum(imp_dlr) v, min(yyyymm) first_ym FROM s.fact_trade WHERE yyyymm BETWEEN 202201 AND 202607 AND imp_dlr>0 GROUP BY 1,2),
        tot AS (SELECT hs10, sum(v) v, min(first_ym) first_ym FROM t GROUP BY 1),
        top AS (SELECT hs10, stat_cd, v, row_number() OVER (PARTITION BY hs10 ORDER BY v DESC) rn FROM t)
        SELECT tot.hs10, round(tot.v/1e6,2) musd, tot.first_ym, top.stat_cd top_origin, round(top.v/tot.v,3) top_share
        FROM tot JOIN top ON top.hs10=tot.hs10 AND rn=1""").df()
    con.close()
    names = {y: pd.read_csv(BY / f"HSK_별표_{y}.csv", dtype=str).set_index("code").leaf.to_dict() for y in (2021, 2022, 2024, 2025)}
    pre = pd.read_csv(OUT / "사전세액심사_대상_이력_2016_2026.csv", dtype=str)
    dist = pd.read_csv(OUT / "유통이력_신고물품_코드요약_2009_2026.csv", dtype=str)
    return rev, conc, rate, imp, names, pre, dist


def main() -> None:
    rev, conc, rate, imp, names, pre, dist = load()
    mfn = {y: rate[rate.year == y].set_index("hs10").mfn.to_dict() for y in (2021, 2022, 2024, 2025)}
    new22 = rev[(rev.rev == "2022") & (rev.change == "신설")]
    new25 = rev[(rev.rev == "2025") & (rev.change == "신설")]
    dead25 = set(rev[(rev.rev == "2025") & (rev.change == "폐지")].hs10)
    codes24 = set(names[2024]); codes25 = set(names[2025])
    succ = conc.groupby("hs_from").hs_to.apply(set).to_dict()
    pred = conc.groupby("hs_to").apply(lambda g: list(zip(g.hs_from, g.weight))).to_dict()
    rows = []
    for _, r in pd.concat([new22, new25]).iterrows():
        h = r.hs10
        if r.rev == "2022":
            ps = [(p, w) for p, w in pred.get(h, []) if p != h]
            y_new, y_old = 2022, 2021
            sibs = set().union(*[succ.get(p, set()) for p, _ in ps]) - {h} if ps else set()
        else:
            y_new, y_old = 2025, 2024
            ps = [(p, 1.0) for p in dead25 if p[:8] == h[:8]] or [(p, 1.0) for p in dead25 if p[:6] == h[:6]]
            if not ps:
                cont = [c for c in codes24 & codes25 if c[:8] == h[:8] and names[2024].get(c) == "기타"]
                if not cont:
                    cont = [c for c in codes24 & codes25 if c[:6] == h[:6] and names[2024].get(c) == "기타"]
                ps = [(c, 1.0) for c in cont]
            sibs = {c for c in codes25 if c[:8] == h[:8] and c != h}
        wsum = sum(w for _, w in ps)
        pm = [(mfn[y_old].get(p), w) for p, w in ps if mfn[y_old].get(p) is not None and not np.isnan(mfn[y_old].get(p))]
        mfn_pred = sum(m * w for m, w in pm) / sum(w for _, w in pm) if pm else np.nan
        mfn_new = mfn[y_new].get(h, np.nan)
        pred_rates = sorted({round(m, 2) for m, _ in pm})
        sib_rates = sorted({round(mfn[y_new].get(c), 2) for c in sibs if mfn[y_new].get(c) is not None and not np.isnan(mfn[y_new].get(c))})
        reason = str(r.reason) if pd.notna(r.reason) else ""
        watch = bool(re.search(WATCH, reason)) and not re.search(NOT_WATCH, reason)
        if pd.isna(mfn_new) or pd.isna(mfn_pred):
            typ = "전신 불명" if not ps else "세율 불명"
        elif not any(abs(mfn_new - pr) <= 0.05 for pr in pred_rates):
            typ = "세율형"          # 신설 코드의 세율이 어느 전신 코드의 세율과도 다르다
        elif len(pred_rates) > 1 and watch:
            typ = "감시형(혼합 전신)"   # 세율이 다른 전신 여럿을 합쳐 나눈 것. 세율 변화인지 손으로 확인
        elif len(pred_rates) > 1:
            typ = "혼합 전신"
        elif watch:
            typ = "감시형"
        else:
            typ = "해당 없음"
        # 기계적 짝: 같은 전신을 나눈 형제 가운데 기타, 2025년은 이어지는 기타 전신
        mate = ""
        nm = names[y_new]
        cands = [c for c in sibs if nm.get(c) == "기타" and c[:6] == h[:6]] or ([p for p, _ in ps if p in codes25 and nm.get(p) == "기타"] if r.rev == "2025" else [])
        if cands:
            mate = sorted(cands, key=lambda c: -imp.set_index("hs10").musd.get(c, 0))[0]
        rows.append(dict(hs10=h, rev=r.rev, effective_ym=r.effective_ym, name=nm.get(h, r.name_ko), reason=reason, watch_reason=watch, type=typ,
                         mfn_new=mfn_new, mfn_pred=round(mfn_pred, 2) if pd.notna(mfn_pred) else np.nan, pred_rates="·".join(map(str, pred_rates)),
                         pred_codes="·".join(p for p, _ in ps[:6]) + ("…" if len(ps) > 6 else ""), n_pred=len(ps), pred_wsum=round(wsum, 3),
                         sibling_rates="·".join(map(str, sib_rates)), n_sib=len(sibs), mate_hs10=mate, mate_mfn=mfn[y_new].get(mate, np.nan) if mate else np.nan,
                         undetermined=bool(rate[(rate.hs10 == h) & (rate.year == y_new)].rate_undetermined.any())))
    t = pd.DataFrame(rows).merge(imp, on="hs10", how="left")
    pre_d = pre.groupby("hs10").agg(pre_assess_from=("aplyStrtDt", "min"), pre_assess_last=("last_month_seen", "max")).reset_index()
    t = t.merge(pre_d, on="hs10", how="left")
    d = dist.groupby("hs10").agg(dist_from=("first_start", "min"), dist_to=("last_end", "max")).reset_index()
    t = t.merge(d, on="hs10", how="left")
    t["sub_type"] = ""   # 경계 감시 / 협정 감시 — 손으로
    t["확인"] = ""
    t = t.sort_values(["type", "rev", "hs10"])
    t.to_csv(OUT / "처리군_목록.csv", index=False, encoding="utf-8-sig")
    print("유형별:", t.groupby(["rev", "type"]).size().to_dict())
    print("감시 사유 중 유형:", t[t.watch_reason].groupby(["rev", "type"]).size().to_dict())
    print("세율형 중 수입 있음:", int((t[t.type == "세율형"].musd.fillna(0) > 0).sum()), " 감시형 중 수입 있음:", int((t[t.type == "감시형"].musd.fillna(0) > 0).sum()))
    print("짝 있음:", int((t.mate_hs10 != "").sum()))


if __name__ == "__main__":
    main()

## 10. 원산지별 실행세율 — 그 밖 FTA까지 넣은 코드 쌍의 실행세율(2012~2026)

논문 II.1. 일곱 상대의 세율은 두 출처 22,164행이 전부 일치한다.

In [ ]:
# ===== 파일: research/scripts/19_extend_applied_rate.py (그대로 옮김, 6,607바이트) =====
"""
19_extend_applied_rate.py — 코드 쌍에 든 코드에 대해 품목 상세에서 받은 그 밖 FTA 세율까지 넣어
원산지(ISO2)별 실행세율을 만든다(2012~2026). 규칙은 15와 같다: 협정세율(선택1)은 무협정 세율(mfn)보다
낮을 때만 적용하고, 한 원산지에 협정이 여럿이면(베트남: 한-베·한-아세안·RCEP) 가장 낮은 것.
선택2 이상·추천 세율은 조건부라 표시만 한다(`n_alt`). 최빈국 특혜(R)·북한산(U)은 원산지 목록 밖이라 뺀다.

입력: outputs/품목상세_세율_2012_2026.csv, kcstariff.duckdb fct_applied_rate
산출: outputs/실행세율_원산지별_쌍코드_2012_2026.csv (year, hs10, stat_cd, regime, applied_jan, applied_last, mfn, n_alt)
      outputs/실행세율_원산지별_검증.csv (일곱 상대 세율의 두 출처 대조)
"""
from __future__ import annotations

from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]                      # KCSTARIFF/research
TARIFF = ROOT.parent / "data" / "processed" / "kcstariff.duckdb"
OUT = ROOT / "outputs"
import os
KCS = Path(os.environ.get("KCSDB2_ROOT", r"C:\Work\Projects\KCSDB2")) / "data" / "processed" / "kcsdb.duckdb"   # 무역통계(읽기 전용)

EU27 = ["AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "GR", "HU", "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT", "RO", "SK", "SI", "ES", "SE"]
ASEAN = ["BN", "KH", "ID", "LA", "MY", "MM", "PH", "SG", "TH", "VN"]
# 구분기호(선택1) → 적용 원산지. 발효 전 연도에는 상세 화면에 기호가 없으므로 연도 조건은 두지 않는다.
REGIME = {
    "FCN1": ["CN"], "FRCCN1": ["CN"], "FEU1": EU27, "FUS1": ["US"], "FCA1": ["CA"], "FIN1": ["IN"], "FVN1": ["VN"],
    "FAS1": ASEAN, "FRCAS1": ASEAN, "FAU1": ["AU"], "FRCAU1": ["AU"], "FNZ1": ["NZ"], "FRCNZ1": ["NZ"], "FRCJP1": ["JP"],
    "FCL1": ["CL"], "FPE1": ["PE"], "FCO1": ["CO"], "FTR1": ["TR"], "FGB1": ["GB"], "FIL1": ["IL"], "FKH1": ["KH"],
    "FPH1": ["PH"], "FSG1": ["SG"], "FID1": ["ID"], "FAE1": ["AE"], "FEF1": ["CH", "NO", "IS", "LI"],
    "FCECR1": ["CR"], "FCEHN1": ["HN"], "FCENI1": ["NI"], "FCEPA1": ["PA"], "FCESV1": ["SV"],
    "E1": ["CN", "IN", "LK", "MN"], "E2": ["BD"], "E3": ["LA"],
}
SEVEN = {"FCN1", "FEU1", "FUS1", "FAS1", "FIN1", "FVN1", "FCA1"}


def import_share_by_regime(year: int = 2025) -> None:
    """그해 수입액에서 FTA 일곱 상대·그 밖 협정 상대(일본 따로)·무협정 원산지의 몫(%). outputs/수입몫_원산지군_<year>.csv"""
    seven = set(["CN", "US", "CA", "IN", "VN"] + EU27 + ASEAN)
    other = set(sum([v for k, v in REGIME.items() if k not in SEVEN and not k.startswith("E")], [])) - seven
    con = duckdb.connect(); con.execute(f"ATTACH '{KCS.as_posix()}' AS s (READ_ONLY)")
    imp = con.sql(f"SELECT stat_cd, sum(imp_dlr) v FROM s.fact_trade WHERE yyyymm BETWEEN {year}01 AND {year}12 GROUP BY 1").df(); con.close()
    tot = imp.v.sum()
    grp = np.where(imp.stat_cd.isin(seven), "일곱 상대", np.where(imp.stat_cd.isin(other), "그 밖 협정", "무협정"))
    out = imp.assign(grp=grp).groupby("grp").v.sum().reindex(["일곱 상대", "그 밖 협정", "무협정"]).fillna(0)
    out = pd.concat([out, pd.Series({"일본": imp[imp.stat_cd == "JP"].v.sum()})])
    t = (100 * out / tot).round(1).rename("share_pct").reset_index().rename(columns={"index": "group"}); t["year"] = year
    t.to_csv(OUT / f"수입몫_원산지군_{year}.csv", index=False, encoding="utf-8-sig")
    print(f"{year}년 수입 몫(%):", t.set_index("group").share_pct.to_dict())


def main() -> None:
    d = pd.read_csv(OUT / "품목상세_세율_2012_2026.csv", dtype={"hs10": str})
    con = duckdb.connect()
    con.execute(f"ATTACH '{TARIFF.as_posix()}' AS tr (READ_ONLY)")
    base = con.sql("SELECT year, hs10, mfn, mfn_jan, j_FCN1, j_FEU1, j_FUS1, j_FAS1, j_FIN1, j_FVN1, j_FCA1 FROM tr.fct_applied_rate WHERE year>=2012").df()
    con.close()
    codes = sorted(d.hs10.unique())
    base = base[base.hs10.isin(codes)]
    # 검증: 일곱 상대의 1월 1일 세율이 두 출처(주요세율보기 대 품목 상세)에서 같은가
    chk = d[d.rate_cd.isin(SEVEN)].merge(base, on=["year", "hs10"], how="inner")
    chk["main_src"] = [r[f"j_{c}"] for c, r in zip(chk.rate_cd, chk.to_dict("records"))]
    chk["diff"] = (chk.adval_first - chk.main_src).abs()
    bad = chk[(chk["diff"] > 0.05) | chk.main_src.isna()]
    bad[["year", "hs10", "rate_cd", "rate_txt", "adval_first", "main_src"]].to_csv(OUT / "실행세율_원산지별_검증.csv", index=False, encoding="utf-8-sig")
    print(f"일곱 상대 대조: {len(chk):,}행 중 어긋남 {len(bad):,} ({100*len(bad)/max(len(chk),1):.2f}%)")
    pd.DataFrame([("일곱 상대 대조 행", len(chk)), ("일곱 상대 어긋남", len(bad))], columns=["item", "value"]).to_csv(OUT / "실행세율_원산지별_검증_요약.csv", index=False, encoding="utf-8-sig")
    import_share_by_regime()

    # 원산지별 적용
    sel = d[d.rate_cd.isin(REGIME) & d.adval_first.notna()]
    alt = d[d.rate_cd.str.match(r"^F[A-Z]+[2-9]$|^F[A-Z]+1[0-9]$|^FEFIS$|^FCE[A-Z]{2}7$")].groupby(["year", "hs10"]).size().rename("n_alt")
    rows = []
    for (y, h), g in sel.groupby(["year", "hs10"]):
        b = base[(base.year == y) & (base.hs10 == h)]
        if b.empty or pd.isna(b.mfn.iloc[0]):
            continue
        mfn, mfn_jan = float(b.mfn.iloc[0]), float(b.mfn_jan.iloc[0])
        per = {}
        for r in g.itertuples(index=False):
            for o in REGIME[r.rate_cd]:
                per.setdefault(o, []).append((r.rate_cd, r.adval_first, r.adval_last))
        for o, lst in per.items():
            jan = min([mfn_jan] + [a for _, a, _ in lst if pd.notna(a)])
            last = min([mfn] + [a for _, _, a in lst if pd.notna(a)])
            used = min(lst, key=lambda x: x[1] if pd.notna(x[1]) else np.inf)
            rows.append(dict(year=y, hs10=h, stat_cd=o, regime=used[0] if used[1] < mfn_jan else "MFN", applied_jan=jan, applied_last=last, mfn=mfn))
    t = pd.DataFrame(rows).merge(alt, on=["year", "hs10"], how="left")
    t["n_alt"] = t.n_alt.fillna(0).astype(int)
    t.to_csv(OUT / "실행세율_원산지별_쌍코드_2012_2026.csv", index=False, encoding="utf-8-sig")
    print(f"원산지별 실행세율: {len(t):,}행, 코드 {t.hs10.nunique()}, 원산지 {t.stat_cd.nunique()}, 협정 적용 몫 {100*(t.regime!='MFN').mean():.1f}%")
    print("규정 분포:", t.regime.value_counts().head(12).to_dict())


if __name__ == "__main__":
    main()

## 11. 통관 패널 — 원산지×월×코드, 쌍×원산지×월

논문 II.4. `panel_codes.parquet`(하한 검사)와 `panel_pairs.parquet`(변환비 쌍의 단가 비율)를 만든다. `--link`는 논문 B의 강건성용이라 기본값 `raw`만 쓴다.

In [ ]:
# ===== 파일: research/scripts/20_build_panel.py (그대로 옮김, 16,476바이트) =====
"""
20_build_panel.py — 3단계 통관 패널(연구 문서 VI 3단계).

대상 코드 = 확정 코드 쌍(확인 Y 또는 별표)의 코드 + 처리군(감시형·세율형·혼합 전신·짝)과 그 전신·후신.
1) 원산지×월×코드 패널: fact_trade 수입액·중량, 연도별 원산지별 실행세율(쌍 코드는 19의 원산지별 표, 그 밖은
   fct_applied_rate의 일곱 상대 세율), 개정을 건너 잇는 hs2022(dim_hs10_to_2022 최빈 승계, chain만).
2) 쌍×원산지×월 표: 높은 쪽·낮은 쪽의 금액·중량·단가·세율·격차.
3) 점검: 세분 전후 합의 연속성(전신 12개월 대 신설+형제 12개월), 잔여 통관 창(폐지 코드가 개정 뒤에도 잡히는 달 수),
   2022년 개정으로 코드를 옮긴 수입액, 표 4의 2022년 수입액 몫 하락이 어느 HS6에서 왔는가.

산출: outputs/panel/panel_codes.parquet, panel_pairs.parquet, 패널_연속성_그룹.csv, 패널_표4_HS6.csv

--link {raw,mode,apportion}(2026-09-13, 논문 B 강건성): 쌍×원산지×월 표에서 2022년 전 계열을 잇는 방법.
  raw(기본) = 2022년 판 코드 번호 그대로(같은 번호의 옛 계열을 그대로 쓰고 신설 코드는 2022년 전 행이 없다. 기존 산출과 같다).
  mode = dim_hs10_to_2022(chain)의 최빈 승계 — 전신 코드의 금액·중량을 가중치가 가장 큰 후신 하나에 전부 배정.
  apportion = 안분 — 전신 코드의 금액·중량을 후신들에 weight 비례로 나눈다.
  raw가 아니면 산출은 outputs/panel_<link>/ 에 쓰고 기존 outputs/panel/ 은 건드리지 않는다.
"""
from __future__ import annotations

import argparse
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

import os
ROOT = Path(__file__).resolve().parents[1]                      # KCSTARIFF/research
REPO = ROOT.parent                                               # KCSTARIFF
KCSDB2 = Path(os.environ.get("KCSDB2_ROOT", r"C:\Work\Projects\KCSDB2"))  # 무역통계 저장소(읽기 전용 의존)
TARIFF = REPO / "data" / "processed" / "kcstariff.duckdb"
KCS = KCSDB2 / "data" / "processed" / "kcsdb.duckdb"
OUT = ROOT / "outputs"
BYEOLPYO_DIR = KCSDB2 / "data" / "external" / "HSK_별표"
PAN = OUT / "panel"
EU27 = set("AT BE BG HR CY CZ DK EE FI FR DE GR HU IE IT LV LT LU MT NL PL PT RO SK SI ES SE".split())
ASEAN = set("BN KH ID LA MY MM PH SG TH VN".split())


def regime_col(o: str) -> str:
    if o == "CN": return "applied_cn"
    if o == "US": return "applied_us"
    if o in EU27: return "applied_eu"
    if o == "VN": return "applied_vn"
    if o in ASEAN: return "applied_asean"
    if o == "IN": return "applied_in"
    if o == "CA": return "applied_ca"
    return "mfn"


def main(link: str = "raw") -> None:
    out_dir = OUT if link == "raw" else OUT / f"panel_{link}"
    pan = PAN if link == "raw" else out_dir
    pan.mkdir(parents=True, exist_ok=True)
    pairs = pd.read_csv(OUT / "코드쌍_목록.csv", dtype={"hs10_high": str, "hs10_low": str})
    pairs = pairs[(pairs.확인.fillna("") == "Y") | pairs.source.isin(["별표", "별표(선택)"])].copy()
    treat = pd.read_csv(OUT / "처리군_목록.csv", dtype={"hs10": str, "mate_hs10": str, "pred_codes": str})
    tr = treat[treat.type.isin(["감시형", "감시형(혼합 전신)", "세율형"])].copy()
    pred_codes = {c for s in tr.pred_codes.fillna("") for c in s.replace("…", "").split("·") if c}
    codes = set(pairs.hs10_high) | set(pairs.hs10_low) | set(tr.hs10) | set(tr.mate_hs10.dropna()) | pred_codes
    codes = sorted(c for c in codes if isinstance(c, str) and len(c) == 10)
    print(f"대상 코드 {len(codes)} (쌍 {pairs.hs10_high.nunique() + pairs.hs10_low.nunique()}, 처리군 {len(tr)}, 전신 {len(pred_codes)})")

    con = duckdb.connect()
    con.execute(f"ATTACH '{KCS.as_posix()}' AS s (READ_ONLY)")
    con.execute(f"ATTACH '{TARIFF.as_posix()}' AS tr (READ_ONLY)")
    con.register("codes", pd.DataFrame({"hs10": codes}))
    # 1) 원산지×월×코드
    f = con.sql("""
        SELECT f.yyyymm, f.stat_cd, f.hs10, f.imp_dlr, f.imp_wgt
        FROM s.fact_trade f JOIN codes USING (hs10) WHERE f.imp_dlr > 0 OR f.imp_wgt > 0""").df()
    f["year"] = f.yyyymm // 100
    # 최빈 승계 hs2022
    m = con.sql("SELECT hs_past, past_version, hs2022, weight FROM s.dim_hs10_to_2022 WHERE method='chain'").df()
    m = m.sort_values("weight", ascending=False).drop_duplicates(["hs_past", "past_version"])
    ver = np.where(f.year <= 2011, "2007", np.where(f.year <= 2016, "2012", np.where(f.year <= 2021, "2017", "2022")))
    f["past_version"] = ver
    f = f.merge(m[["hs_past", "past_version", "hs2022"]], left_on=["hs10", "past_version"], right_on=["hs_past", "past_version"], how="left").drop(columns=["hs_past"])
    f["hs2022"] = np.where(f.past_version == "2022", f.hs10, f.hs2022)
    # 세율: 쌍 코드는 원산지별 표, 나머지는 일곱 상대
    ext = pd.read_csv(OUT / "실행세율_원산지별_쌍코드_2012_2026.csv", dtype={"hs10": str})
    ext = ext.rename(columns={"applied_last": "tau_ext", "mfn": "mfn_ext"})[["year", "hs10", "stat_cd", "tau_ext", "regime"]]
    base = con.sql("SELECT year, hs10, mfn, applied_cn, applied_us, applied_eu, applied_asean, applied_vn, applied_in, applied_ca, floor_won_kg, rate_undetermined FROM tr.fct_applied_rate").df()
    base = base[base.hs10.isin(codes)]
    f = f.merge(base, on=["year", "hs10"], how="left").merge(ext, on=["year", "hs10", "stat_cd"], how="left")
    cols = f[["applied_cn", "applied_us", "applied_eu", "applied_asean", "applied_vn", "applied_in", "applied_ca", "mfn"]].to_numpy()
    idx = {"applied_cn": 0, "applied_us": 1, "applied_eu": 2, "applied_asean": 3, "applied_vn": 4, "applied_in": 5, "applied_ca": 6, "mfn": 7}
    f["tau_seven"] = [cols[i, idx[regime_col(o)]] for i, o in enumerate(f.stat_cd)]
    f["tau"] = f.tau_ext.where(f.tau_ext.notna(), f.tau_seven)
    f["uv"] = f.imp_dlr / f.imp_wgt.replace(0, np.nan)
    f = f.drop(columns=["applied_cn", "applied_us", "applied_eu", "applied_asean", "applied_vn", "applied_in", "applied_ca", "tau_seven"])
    f.to_parquet(pan / "panel_codes.parquet", index=False)
    print(f"panel_codes {len(f):,}행, 코드 {f.hs10.nunique()}, 원산지 {f.stat_cd.nunique()}, 세율 결측 {f.tau.isna().mean():.1%}, hs2022 결측(2022 전) {f[f.past_version!='2022'].hs2022.isna().mean():.1%}")

    # 2) 쌍×원산지×월
    key = ["yyyymm", "stat_cd", "hs10"]
    g = f[key + ["imp_dlr", "imp_wgt", "uv", "floor_won_kg"]]
    if link != "raw":
        # 2022년 전 계열을 연계표로 잇는다: 쌍 코드(2022년 판)의 전신 코드 전부를 가져와 mode(최빈, 가중치 1)·apportion(weight 비례)으로 배정
        pc_ = sorted(set(pairs.hs10_high) | set(pairs.hs10_low))
        lk = con.sql("SELECT hs_past, past_version, hs2022, weight FROM s.dim_hs10_to_2022 WHERE method='chain'").df()
        lk = lk[lk.hs2022.isin(pc_)].copy()
        if link == "mode":
            best = con.sql("SELECT hs_past, past_version, hs2022 FROM s.dim_hs10_to_2022 WHERE method='chain'").df().merge(m[["hs_past", "past_version", "hs2022"]], on=["hs_past", "past_version", "hs2022"])
            lk = lk.merge(best, on=["hs_past", "past_version", "hs2022"]); lk["weight"] = 1.0
        con.register("pastc", pd.DataFrame({"hs10": sorted(lk.hs_past.unique())}))
        old = con.sql("SELECT f.yyyymm, f.stat_cd, f.hs10, f.imp_dlr, f.imp_wgt FROM s.fact_trade f JOIN pastc USING (hs10) WHERE f.yyyymm < 202201 AND (f.imp_dlr > 0 OR f.imp_wgt > 0)").df()
        old["year"] = old.yyyymm // 100
        old["past_version"] = np.where(old.year <= 2011, "2007", np.where(old.year <= 2016, "2012", "2017"))
        old = old.merge(lk, left_on=["hs10", "past_version"], right_on=["hs_past", "past_version"], how="inner")
        old["imp_dlr"] = old.imp_dlr * old.weight; old["imp_wgt"] = old.imp_wgt * old.weight
        old = old.groupby(["yyyymm", "stat_cd", "hs2022"]).agg(imp_dlr=("imp_dlr", "sum"), imp_wgt=("imp_wgt", "sum")).reset_index().rename(columns={"hs2022": "hs10"})
        old["uv"] = old.imp_dlr / old.imp_wgt.replace(0, np.nan)
        fl_ = f[["year", "hs10", "floor_won_kg"]].drop_duplicates(["year", "hs10"]); old["year"] = old.yyyymm // 100
        old = old.merge(fl_, on=["year", "hs10"], how="left").drop(columns="year")
        g = pd.concat([g[(g.yyyymm >= 202201) | ~g.hs10.isin(pc_)], old[key + ["imp_dlr", "imp_wgt", "uv", "floor_won_kg"]]], ignore_index=True)
        print(f"연계({link}): 전신 코드 {lk.hs_past.nunique()}개 → 쌍 코드 {lk.hs2022.nunique()}개, 2022년 전 행 {len(old):,}")
    origins = sorted(f.stat_cd.unique())
    rl = base[["year", "hs10", "mfn", "applied_cn", "applied_us", "applied_eu", "applied_asean", "applied_vn", "applied_in", "applied_ca"]].merge(pd.DataFrame({"stat_cd": origins}), how="cross")
    rc = np.array([regime_col(o) for o in rl.stat_cd])
    arr = rl[["applied_cn", "applied_us", "applied_eu", "applied_asean", "applied_vn", "applied_in", "applied_ca", "mfn"]].to_numpy()
    rl["tau"] = arr[np.arange(len(rl)), [idx[c] for c in rc]]
    rl = rl[["year", "hs10", "stat_cd", "tau"]].merge(ext[["year", "hs10", "stat_cd", "tau_ext"]], on=["year", "hs10", "stat_cd"], how="left")
    rl["tau"] = rl.tau_ext.where(rl.tau_ext.notna(), rl.tau)
    rl = rl.drop(columns="tau_ext")
    rows = []
    for r in pairs.itertuples(index=False):
        H = g[g.hs10 == r.hs10_high].drop(columns="hs10").rename(columns=lambda c: c if c in ("yyyymm", "stat_cd") else c + "_H")
        L = g[g.hs10 == r.hs10_low].drop(columns="hs10").rename(columns=lambda c: c if c in ("yyyymm", "stat_cd") else c + "_L")
        x = H.merge(L, on=["yyyymm", "stat_cd"], how="outer")
        x.insert(0, "pair_id", r.pair_id); x.insert(1, "hs10_high", r.hs10_high); x.insert(2, "hs10_low", r.hs10_low)
        rows.append(x)
    pp = pd.concat(rows, ignore_index=True)
    pp["year"] = pp.yyyymm // 100
    pp = pp.merge(rl.rename(columns={"hs10": "hs10_high", "tau": "tau_H"}), on=["year", "hs10_high", "stat_cd"], how="left")
    pp = pp.merge(rl.rename(columns={"hs10": "hs10_low", "tau": "tau_L"}), on=["year", "hs10_low", "stat_cd"], how="left")
    pp["gap"] = pp.tau_H - pp.tau_L
    pp["share_L_v"] = pp.imp_dlr_L.fillna(0) / (pp.imp_dlr_L.fillna(0) + pp.imp_dlr_H.fillna(0)).replace(0, np.nan)
    pp.to_parquet(pan / "panel_pairs.parquet", index=False)
    print(f"panel_pairs {len(pp):,}행, 쌍 {pp.pair_id.nunique()}, 격차 결측 {pp.gap.isna().mean():.1%}")

    # 3) 점검 ① 세분 전후 합의 연속성 — 전신·후신 그래프의 연결 성분(닫힌 집단) 단위로 2021년 합과 2022년 합을 비교한다.
    #    2022년에도 남은 전신 코드는 후신에도 넣는다(연계표의 moved 관계에는 일부만 옮긴 코드가 있다).
    conc22 = con.sql("SELECT hs_from, hs_to FROM s.dim_hs10_concordance WHERE revision='2022' AND hs_from<>hs_to").df()
    dead = set(con.sql("SELECT hs10 FROM s.dim_hsk_revision WHERE change='폐지' AND rev IN ('2022','2025')").df().hs10)
    alive22 = set(pd.read_csv(BYEOLPYO_DIR / "HSK_별표_2022.csv", dtype=str).code)
    par = {}
    def find(x):
        while par.setdefault(x, x) != x:
            par[x] = par[par[x]]; x = par[x]
        return x
    for a_, b_ in zip(conc22.hs_from, conc22.hs_to):
        par[find("F" + a_)] = find("T" + b_)
    tr22 = tr[tr.rev == 2022]
    targets = {find("T" + h) for h in tr22.hs10 if "T" + h in par}
    groups = {}
    for n in list(par):
        r = find(n)
        if r in targets:
            groups.setdefault(r, {"F": set(), "T": set()})[n[0]].add(n[1:])
    for g in groups.values():
        g["T"] |= {h for h in g["F"] if h in alive22}
    # 2025년은 전신(폐지 또는 이어지는 기타)과 신설+짝
    for r in tr[tr.rev == 2025].itertuples(index=False):
        ps = {c for c in str(r.pred_codes or "").replace("…", "").split("·") if c}
        if ps:
            groups[("2025", r.hs10)] = {"F": ps, "T": {r.hs10} | ({r.mate_hs10} if isinstance(r.mate_hs10, str) else set()) | {c for c in ps if c not in dead}}
    need = sorted(set().union(*[g["F"] | g["T"] for g in groups.values()]))
    con.register("need", pd.DataFrame({"hs10": need}))
    mon = con.sql("SELECT yyyymm, hs10, sum(imp_dlr) v FROM s.fact_trade JOIN need USING (hs10) GROUP BY 1,2").df()
    rows = []
    for cid, g in groups.items():
        rev = 2025 if isinstance(cid, tuple) else 2022
        y0 = rev
        bsum = mon[(mon.yyyymm // 100 == y0 - 1)].groupby("hs10").v.sum(); asum = mon[(mon.yyyymm // 100 == y0)].groupby("hs10").v.sum()
        bf = sum(bsum.get(h, 0) for h in g["F"]); af = sum(asum.get(h, 0) for h in g["T"])
        deadF = g["F"] & dead
        resid = mon[mon.hs10.isin(deadF) & (mon.yyyymm >= y0 * 100 + 1)]
        rows.append(dict(rev=rev, n_from=len(g["F"]), n_to=len(g["T"]), treat="·".join(sorted(set(tr.hs10) & g["T"]))[:60],
                         before_musd=round(bf / 1e6, 2), after_musd=round(af / 1e6, 2), ratio=round(af / bf, 2) if bf > 0 else np.nan,
                         residual_months=int(resid.yyyymm.nunique()), residual_musd=round(resid.v.sum() / 1e6, 3)))
    cont = pd.DataFrame(rows).sort_values(["rev", "before_musd"], ascending=[True, False])
    cont.to_csv(out_dir / "패널_연속성_그룹.csv", index=False, encoding="utf-8-sig")
    big = cont[cont.before_musd >= 1]
    tot = con.sql("SELECT sum(CASE WHEN yyyymm>=202201 THEN imp_dlr END)/sum(CASE WHEN yyyymm<=202112 THEN imp_dlr END) FROM s.fact_trade WHERE yyyymm BETWEEN 202101 AND 202212").fetchone()[0]
    print(f"연속성(집단 {len(cont)}개, 전신 수입 1백만 달러 이상 {len(big)}개): 비율 분위 {big.ratio.describe(percentiles=[.1,.25,.5,.75,.9]).round(2)[['10%','25%','50%','75%','90%']].to_dict()}, 0.5~2.0 안 {int(big.ratio.between(0.5,2.0).sum())}, 총액 비율 {big.after_musd.sum()/big.before_musd.sum():.2f}(전체 수입 2022/2021 {tot:.2f})")
    print(f"잔여 통관: 폐지 전신이 있는 집단 {int((cont.residual_months>0).sum())}개, 최대 {int(cont.residual_months.max())}개월, 잔여 수입액 합 {cont.residual_musd.sum():.1f}백만 달러")

    # 점검 ② 2022년 개정으로 코드를 옮긴 수입액 (전체 자료)
    mv = con.sql("""
        WITH d AS (SELECT hs10 FROM s.dim_hsk_revision WHERE rev='2022' AND change='폐지'),
        t AS (SELECT sum(imp_dlr) v FROM s.fact_trade WHERE yyyymm BETWEEN 201701 AND 202112),
        x AS (SELECT sum(imp_dlr) v FROM s.fact_trade f JOIN d USING (hs10) WHERE yyyymm BETWEEN 201701 AND 202112)
        SELECT x.v/t.v pct FROM x, t""").fetchone()[0]
    print(f"2022년 개정으로 코드를 옮긴 수입액: 2017~2021년 총수입의 {100*mv:.2f}%")

    # 점검 ③ 표 4 HS6 몫: 기본세율이 다른 HS6의 수입액 몫, 2021 대 2022 (어느 HS6가 빠졌나)
    h6 = con.sql("""
        WITH r AS (SELECT year, hs10, substr(hs10,1,6) hs6, adval FROM tr.tariff_rate WHERE rate_cd='A' AND adval IS NOT NULL AND year IN (2021,2022)),
        g AS (SELECT year, hs6 FROM r GROUP BY 1,2 HAVING count(DISTINCT adval)>1),
        imp AS (SELECT yyyymm//100 AS year, substr(hs10,1,6) hs6, sum(imp_dlr) v FROM s.fact_trade WHERE yyyymm BETWEEN 202101 AND 202212 GROUP BY 1,2),
        tot AS (SELECT year, sum(v) tv FROM imp GROUP BY 1)
        SELECT imp.hs6, imp.year, imp.v/tot.tv*100 pct, CASE WHEN g.hs6 IS NULL THEN 0 ELSE 1 END diff
        FROM imp JOIN tot USING (year) LEFT JOIN g ON g.year=imp.year AND g.hs6=imp.hs6""").df()
    w = h6.pivot_table(index="hs6", columns="year", values=["pct", "diff"]).fillna(0)
    w.columns = [f"{a}_{b}" for a, b in w.columns]
    lost = w[(w.diff_2021 == 1) & (w.diff_2022 == 0)].sort_values("pct_2021", ascending=False)
    lost.to_csv(out_dir / "패널_표4_HS6.csv", encoding="utf-8-sig")
    print(f"표 4: 기본세율 다른 HS6 몫 2021 {w[w.diff_2021==1].pct_2021.sum():.1f}% → 2022 {w[w.diff_2022==1].pct_2022.sum():.1f}%; 2022에 빠진 HS6 {len(lost)}개, 그 2021 몫 합 {lost.pct_2021.sum():.1f}%p")
    print(lost.head(8)[["pct_2021", "pct_2022"]].round(2).to_string())
    con.close()


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--link", choices=["raw", "mode", "apportion"], default="raw", help="2022년 전 계열의 연계 방법(raw=기존, mode=최빈 승계, apportion=안분)")
    a = ap.parse_args()
    main(a.link)

## 12. 미러 통계 — UN Comtrade 공개 미리보기 API와 갭 회귀

논문 II.5·III.4·IV.2·IV.5. 21이 보고국 30개의 대한국 HS6 수출을 받고(응답 없는 두 나라를 빼면 28개국 91,318행), 22가 HS6 집단×보고국×연도 패널 21,060개에서 금액·물량·단가 갭을 실행세율 수준과 HS6 안 최대 격차에 회귀하고 생강의 미러 단가(표 3)를 남긴다.

In [ ]:
# ===== 파일: research/scripts/21_fetch_comtrade_mirror.py (그대로 옮김, 6,037바이트) =====
"""
21_fetch_comtrade_mirror.py — UN Comtrade 공개 미리보기 API(키 없음, 호출당 500행)로 주요 원산지가 보고한
대한국 HS6 수출(미러 통계)을 받는다. 대상 HS6는 확정 코드 쌍과 처리군 코드의 앞 6자리이며, 그해 HS 판본의
옛 코드(dim_hs6_concordance)도 함께 요청한다.

캐시: data/raw/comtrade/<reporter>_<year>.json
산출: research/outputs/comtrade_mirror_hs6_2012_2024.csv
      (reporter, reporterISO, year, cmdCode, classification, value_usd, net_kg, qty, qty_unit)
"""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import duckdb
import pandas as pd
import requests

import os
ROOT = Path(__file__).resolve().parents[1]                      # KCSTARIFF/research
KCSDB2 = Path(os.environ.get("KCSDB2_ROOT", r"C:\Work\Projects\KCSDB2"))
KCS = KCSDB2 / "data" / "processed" / "kcsdb.duckdb"
OUT = ROOT / "outputs"
RAW = ROOT / "data" / "raw" / "comtrade"
URL = "https://comtradeapi.un.org/public/v1/preview/C/A/HS"
# 관세청 stat_cd → Comtrade reporterCode (UN M49). 대만은 490(Other Asia, nes).
REPORTERS = {"CN": 156, "US": 842, "JP": 392, "VN": 704, "AU": 36, "MY": 458, "DE": 276, "SG": 702, "BR": 76, "ID": 360,
             "AR": 32, "TH": 764, "NL": 528, "RU": 643, "TW": 490, "AE": 784, "FR": 250, "IT": 380, "NZ": 554, "IN": 699,
             "CA": 124, "GB": 826, "CH": 756, "ES": 724, "PE": 604, "MX": 484, "PH": 608, "TR": 792, "CL": 152, "EG": 818}
YEARS = range(2012, 2025)


def scope() -> tuple[list[str], pd.DataFrame]:
    pairs = pd.read_csv(OUT / "코드쌍_목록.csv", dtype={"hs10_high": str, "hs10_low": str})
    pairs = pairs[(pairs.확인.fillna("") == "Y") | pairs.source.isin(["별표", "별표(선택)"])]
    treat = pd.read_csv(OUT / "처리군_목록.csv", dtype={"hs10": str, "mate_hs10": str})
    tr = treat[treat.type.isin(["감시형", "감시형(혼합 전신)", "세율형"])]
    codes = set(pairs.hs10_high) | set(pairs.hs10_low) | set(tr.hs10) | set(tr.mate_hs10.dropna())
    hs6 = sorted({c[:6] for c in codes})
    con = duckdb.connect(); con.execute(f"ATTACH '{KCS.as_posix()}' AS s (READ_ONLY)")
    con.register("h6", pd.DataFrame({"hs6": hs6}))
    past = con.sql("SELECT DISTINCT hs2022, hs_past, past_version FROM s.dim_hs6_concordance JOIN h6 ON hs2022=hs6").df()
    con.close()
    return hs6, past


def codes_for_year(hs6: list[str], past: pd.DataFrame, year: int) -> list[str]:
    ver = "2012" if year <= 2016 else "2017" if year <= 2021 else None
    extra = set(past[past.past_version == ver].hs_past) if ver else set()
    return sorted(set(hs6) | extra)


def fetch(sess: requests.Session, reporter: int, year: int, codes: list[str], delay: float = 1.5) -> dict:
    """500행 상한에 걸리면 코드를 반으로 나눠 다시 받아 합친다. 429는 30초 쉬고 두 번 더 시도한다."""
    for attempt in range(3):
        r = sess.get(URL, params=dict(reporterCode=reporter, partnerCode=410, period=str(year), cmdCode=",".join(codes), flowCode="X"), timeout=120)
        if r.status_code == 429:
            time.sleep(30 * (attempt + 1)); continue
        r.raise_for_status(); j = r.json(); break
    else:
        r.raise_for_status()
    if j.get("count", 0) >= 500 and len(codes) > 1:
        h = len(codes) // 2; time.sleep(delay)
        a = fetch(sess, reporter, year, codes[:h], delay); time.sleep(delay); b = fetch(sess, reporter, year, codes[h:], delay)
        return {"count": a.get("count", 0) + b.get("count", 0), "data": a.get("data", []) + b.get("data", []), "split": True}
    return j


def main() -> None:
    ap = argparse.ArgumentParser(); ap.add_argument("--delay", type=float, default=1.5); ap.add_argument("--parse-only", action="store_true"); ap.add_argument("--refetch-capped", action="store_true", help="500행에 걸린 캐시를 나눠 다시 받는다")
    args = ap.parse_args()
    hs6, past = scope()
    RAW.mkdir(parents=True, exist_ok=True)
    sess = requests.Session(); sess.headers.update({"User-Agent": "Mozilla/5.0"})
    rows, failed = [], []
    for iso, rep in REPORTERS.items():
        for y in YEARS:
            f = RAW / f"{iso}_{y}.json"
            if f.exists() and not args.refetch_capped:
                j = json.loads(f.read_text(encoding="utf-8"))
            elif f.exists() and json.loads(f.read_text(encoding="utf-8")).get("count", 0) < 500:
                j = json.loads(f.read_text(encoding="utf-8"))
            elif args.parse_only:
                continue
            else:
                try:
                    j = fetch(sess, rep, y, codes_for_year(hs6, past, y), args.delay)
                except Exception as e:  # noqa: BLE001
                    failed.append((iso, y, str(e)[:80])); time.sleep(args.delay * 4); continue
                f.write_text(json.dumps(j, ensure_ascii=False), encoding="utf-8"); time.sleep(args.delay)
            for d in j.get("data", []):
                rows.append(dict(reporter=iso, reporterISO=d.get("reporterISO"), year=y, cmdCode=d["cmdCode"], classification=d.get("classificationCode"),
                                 value_usd=d.get("primaryValue"), net_kg=d.get("netWgt"), qty=d.get("qty"), qty_unit=d.get("qtyUnitAbbr"), count=j.get("count")))
        print(f"{iso}: 행 {sum(1 for r in rows if r['reporter']==iso):,}, 실패 {len(failed)}", flush=True)
    df = pd.DataFrame(rows)
    df.to_csv(OUT / "comtrade_mirror_hs6_2012_2024.csv", index=False, encoding="utf-8-sig")
    capped = int(sum(1 for iso in REPORTERS for y in YEARS if (RAW / f"{iso}_{y}.json").exists() and json.loads((RAW / f"{iso}_{y}.json").read_text(encoding="utf-8")).get("count", 0) >= 500 and not json.loads((RAW / f"{iso}_{y}.json").read_text(encoding="utf-8")).get("split")))
    print(f"완료: {len(df):,}행, 보고국 {df.reporter.nunique()}, HS6 {df.cmdCode.nunique()}, 500행 상한에 걸린 채 남은 호출 {capped}, 실패 {len(failed)}")
    if failed:
        print(failed[:10])


if __name__ == "__main__":
    main()

In [ ]:
# ===== 파일: research/scripts/22_mirror_gap.py (그대로 옮김, 9,150바이트) =====
"""
22_mirror_gap.py — 미러 갭 회귀(연구 문서 IV.3절). 상대국이 보고한 대한국 수출(21의 Comtrade)과 한국 수입(fact_trade)을
HS6 집단×원산지×연도로 맞대 갭 = ln X − ln M 을 금액·물량·단가로 나누고, 그 집단의 실행세율 수준과
집단 안 최대 격차에 회귀한다(집단 고정효과, 원산지×연도 고정효과, 집단 군집).

HS6 집단: 2012·2017·2022 개정으로 갈라지거나 합쳐진 HS6를 dim_hs6_concordance의 연결 성분으로 묶는다.
산출: outputs/미러_갭_패널.csv, outputs/미러_갭_회귀.csv, outputs/미러_건조생강.csv
"""
from __future__ import annotations

from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

import os
ROOT = Path(__file__).resolve().parents[1]                      # KCSTARIFF/research
REPO = ROOT.parent                                               # KCSTARIFF
KCSDB2 = Path(os.environ.get("KCSDB2_ROOT", r"C:\Work\Projects\KCSDB2"))  # 무역통계 저장소(읽기 전용 의존)
TAR = REPO / "data" / "processed" / "kcstariff.duckdb"
KCS = KCSDB2 / "data" / "processed" / "kcsdb.duckdb"
OUT = ROOT / "outputs"
BYEOLPYO_DIR = KCSDB2 / "data" / "external" / "HSK_별표"


def fe_ols(df, y, xcols, fe, cluster, tol=1e-9, maxit=200):
    d = df[list(dict.fromkeys([y] + xcols + fe + [cluster]))].dropna().copy()
    Z = d[[y] + xcols].to_numpy(dtype=float)
    groups = [d[f].astype("category").cat.codes.to_numpy() for f in fe]
    for _ in range(maxit):
        Z0 = Z.copy()
        for g in groups:
            m = np.zeros((g.max() + 1, Z.shape[1])); n = np.bincount(g)
            np.add.at(m, g, Z); Z = Z - m[g] / n[g][:, None]
        if np.abs(Z - Z0).max() < tol:
            break
    yv, X = Z[:, 0], Z[:, 1:]
    XtX_inv = np.linalg.pinv(X.T @ X); b = XtX_inv @ X.T @ yv; e = yv - X @ b
    cl = d[cluster].astype("category").cat.codes.to_numpy(); G = cl.max() + 1
    S = np.zeros((G, X.shape[1])); np.add.at(S, cl, X * e[:, None]); meat = S.T @ S
    n, k = X.shape; V = XtX_inv @ meat @ XtX_inv * (G / (G - 1)) * ((n - 1) / (n - k))
    se = np.sqrt(np.diag(V))
    return pd.DataFrame({"coef": b, "se": se, "t": b / se}, index=xcols).assign(n=n, clusters=G)


def main() -> None:
    cm = pd.read_csv(OUT / "comtrade_mirror_hs6_2012_2024.csv", dtype={"cmdCode": str})
    con = duckdb.connect(); con.execute(f"ATTACH '{KCS.as_posix()}' AS s (READ_ONLY)"); con.execute(f"ATTACH '{TAR.as_posix()}' AS tr (READ_ONLY)")
    conc = con.sql("SELECT hs2022, hs_past, past_version FROM s.dim_hs6_concordance").df()
    # 집단: 범위 안 hs2022와 그 옛 코드의 연결 성분
    scope = set(cm.cmdCode)
    sub = conc[conc.hs2022.isin(scope) | conc.hs_past.isin(scope)]
    par = {}
    def find(x):
        while par.setdefault(x, x) != x:
            par[x] = par[par[x]]; x = par[x]
        return x
    for a, b in zip(sub.hs2022, sub.hs_past):
        par[find("N" + a)] = find("P" + b)
    def gid(code, version):
        key = ("N" if version == "2022" else "P") + code
        return find(key) if key in par else "N" + code
    grp_name = {}
    for n in list(par):
        r = find(n); grp_name.setdefault(r, set()).add(n[1:])
    label = {r: min(v) for r, v in grp_name.items()}
    def ver_of(year): return "2012" if year <= 2016 else "2017" if year <= 2021 else "2022"
    cm["grp"] = [label.get(gid(c, ver_of(y)), c) for c, y in zip(cm.cmdCode, cm.year)]
    X = cm.groupby(["reporter", "year", "grp"]).agg(x_usd=("value_usd", "sum"), x_kg=("net_kg", "sum")).reset_index()
    # 한국 수입
    con.register("rep", pd.DataFrame({"stat_cd": sorted(cm.reporter.unique())}))
    M = con.sql("""SELECT stat_cd AS reporter, yyyymm//100 AS year, substr(hs10,1,6) hs6, sum(imp_dlr) m_usd, sum(imp_wgt) m_kg
                   FROM s.fact_trade JOIN rep USING (stat_cd) WHERE yyyymm BETWEEN 201201 AND 202412 GROUP BY 1,2,3""").df()
    M["grp"] = [label.get(gid(c, ver_of(y)), c) for c, y in zip(M.hs6, M.year)]
    M = M[M.grp.isin(set(X.grp))].groupby(["reporter", "year", "grp"]).agg(m_usd=("m_usd", "sum"), m_kg=("m_kg", "sum")).reset_index()
    P = X.merge(M, on=["reporter", "year", "grp"], how="outer")
    P = P[(P.x_usd > 0) & (P.m_usd > 0)].copy()
    P["gap_v"] = np.log(P.x_usd / P.m_usd); P["gap_q"] = np.where((P.x_kg > 0) & (P.m_kg > 0), np.log(P.x_kg / P.m_kg), np.nan); P["gap_uv"] = P.gap_v - P.gap_q
    # 세율: 집단 안 HS10의 mfn(수입 가중 평균)과 최대 격차
    rt = con.sql("SELECT year, hs10, substr(hs10,1,6) hs6, mfn FROM tr.fct_applied_rate WHERE year BETWEEN 2012 AND 2024 AND mfn IS NOT NULL").df()
    w = con.sql("SELECT yyyymm//100 AS year, hs10, sum(imp_dlr) v FROM s.fact_trade WHERE yyyymm BETWEEN 201201 AND 202412 GROUP BY 1,2").df()
    rt = rt.merge(w, on=["year", "hs10"], how="left"); rt["v"] = rt.v.fillna(0) + 1.0
    rt["grp"] = [label.get(gid(c, ver_of(y)), c) for c, y in zip(rt.hs6, rt.year)]
    T = rt[rt.grp.isin(set(P.grp))].groupby(["year", "grp"]).apply(lambda g: pd.Series(dict(mfn_mean=np.average(g.mfn, weights=g.v), mfn_max=g.mfn.max(), mfn_min=g.mfn.min()))).reset_index()
    T["mfn_gap"] = T.mfn_max - T.mfn_min
    P = P.merge(T, on=["year", "grp"], how="left"); P["ry"] = P.reporter + "|" + P.year.astype(str)
    P.to_csv(OUT / "미러_갭_패널.csv", index=False, encoding="utf-8-sig")
    print(f"패널 {len(P):,} (집단 {P.grp.nunique()}, 보고국 {P.reporter.nunique()}, 연도 {P.year.min()}~{P.year.max()}); 갭 중위 금액 {P.gap_v.median():.2f} 물량 {P.gap_q.median():.2f} 단가 {P.gap_uv.median():.2f}")
    res = []
    for y in ["gap_v", "gap_q", "gap_uv"]:
        r = fe_ols(P, y, ["mfn_mean", "mfn_gap"], ["grp", "ry"], "grp")
        for x in ["mfn_mean", "mfn_gap"]:
            res.append(dict(dep=y, var=x, coef=r.loc[x, "coef"], se=r.loc[x, "se"], t=r.loc[x, "t"], n=int(r.n.iloc[0]), clusters=int(r.clusters.iloc[0])))
        # 중국만
        rc = fe_ols(P[P.reporter == "CN"].assign(ry=lambda d: d.year.astype(str)), y, ["mfn_mean", "mfn_gap"], ["grp", "ry"], "grp")
        for x in ["mfn_mean", "mfn_gap"]:
            res.append(dict(dep=y + "(중국)", var=x, coef=rc.loc[x, "coef"], se=rc.loc[x, "se"], t=rc.loc[x, "t"], n=int(rc.n.iloc[0]), clusters=int(rc.clusters.iloc[0])))
    R = pd.DataFrame(res).round(4); R.to_csv(OUT / "미러_갭_회귀.csv", index=False, encoding="utf-8-sig"); print(R.to_string(index=False))
    # 집단화 없이 HS6 그대로(2017~2024, HS2017·HS2022에서 코드가 같은 것만): 건조생강 0910.12 등
    F = cm[cm.year >= 2017].groupby(["reporter", "year", "cmdCode"]).agg(x_usd=("value_usd", "sum"), x_kg=("net_kg", "sum")).reset_index().rename(columns={"cmdCode": "hs6"})
    M6 = con.sql("""SELECT stat_cd AS reporter, yyyymm//100 AS year, substr(hs10,1,6) hs6, sum(imp_dlr) m_usd, sum(imp_wgt) m_kg
                    FROM s.fact_trade JOIN rep USING (stat_cd) WHERE yyyymm BETWEEN 201701 AND 202412 GROUP BY 1,2,3""").df()
    F = F.merge(M6, on=["reporter", "year", "hs6"], how="inner"); F = F[(F.x_usd > 0) & (F.m_usd > 0) & (F.x_kg > 0) & (F.m_kg > 0)].copy()
    F["gap_v"] = np.log(F.x_usd / F.m_usd); F["gap_q"] = np.log(F.x_kg / F.m_kg); F["gap_uv"] = F.gap_v - F.gap_q
    F["uv_x"] = F.x_usd / F.x_kg; F["uv_m"] = F.m_usd / F.m_kg
    F.to_csv(OUT / "미러_갭_HS6_2017_2024.csv", index=False, encoding="utf-8-sig")
    g = F[(F.reporter == "CN") & (F.hs6.isin({"091012", "091011"}))].sort_values(["hs6", "year"])
    g.to_csv(OUT / "미러_건조생강.csv", index=False, encoding="utf-8-sig")
    print("건조생강·신선생강 중국(HS6 그대로, 2017~):")
    print(g[["hs6", "year", "x_usd", "x_kg", "m_usd", "m_kg", "uv_x", "uv_m", "gap_uv"]].round(2).to_string(index=False))
    # 단가 갭이 가장 큰 HS6×원산지(2017~2024 평균, 수입 100만 달러 이상)
    top = F[F.m_usd >= 1e6].groupby(["reporter", "hs6"]).agg(gap_uv=("gap_uv", "mean"), n=("year", "size"), m_usd=("m_usd", "sum")).reset_index()
    top = top[top.n >= 4].sort_values("gap_uv", ascending=False)
    top["agri"] = top.hs6.str[:2].astype(int) <= 24
    print("단가 갭 상위, 농식품(1~24류) HS6×원산지(2017~2024 평균, 4년 이상, 수입 100만 달러 이상):")
    print(top[top.agri].head(15).round(2).to_string(index=False))
    print("농식품 단가 갭 분위:", top[top.agri].gap_uv.describe(percentiles=[.1, .5, .9]).round(2).to_dict())
    top.to_csv(OUT / "미러_단가갭_HS6원산지.csv", index=False, encoding="utf-8-sig")
    # 농식품만 갭 회귀
    Pa = P[P.grp.str[:2].astype(int) <= 24]
    ra = []
    for y in ["gap_v", "gap_q", "gap_uv"]:
        r = fe_ols(Pa, y, ["mfn_mean", "mfn_gap"], ["grp", "ry"], "grp")
        for x in ["mfn_mean", "mfn_gap"]:
            ra.append(dict(dep=y + "(농식품)", var=x, coef=r.loc[x, "coef"], se=r.loc[x, "se"], t=r.loc[x, "t"], n=int(r.n.iloc[0]), clusters=int(r.clusters.iloc[0])))
    Ra = pd.DataFrame(ra).round(4); print(Ra.to_string(index=False))
    pd.concat([R, Ra]).to_csv(OUT / "미러_갭_회귀.csv", index=False, encoding="utf-8-sig")
    con.close()


if __name__ == "__main__":
    main()

## 13. 파일럿 계열 — 건조생강·건고추의 원산지×월 수입

논문 IV.1·IV.2. 중국산 건조생강 단가(2012년 1.94달러 → 2020~2021년 0.46달러, 신선×10 대비 3~4%), 2021년 11월 0톤, 페루·베트남산의 대체, 건고추 하한/단가 49~60%와 생강 6~9%는 이 계열에서 14의 §3d 셀이 계산한다.

In [ ]:
# 파일럿 계열 outputs/pilot_series.csv 의 재생성(2026-09-13). 원래는 2026-09-12에 대화형으로 뽑았다.
# 고추·생강·다데기 계열의 코드 집단(grp)을 무역통계 fact_trade에서 원산지×월×코드로 받는다(수입액이나 중량이 0보다 큰 행).
# 기존 파일이 있으면 덮어쓰지 않고 같은지 확인한다(2026-09-13 확인: 20,933행 전부 같음).
import os, duckdb, pandas as pd
KCS = os.path.join(os.environ.get("KCSDB2_ROOT", r"C:\Work\Projects\KCSDB2"), "data", "processed", "kcsdb.duckdb")
OUT = os.path.join("outputs", "pilot_series.csv")
GRP = {"건고추": ["0904201000", "0904210000"], "고춧가루": ["0904202000", "0904220000"], "냉동고추": ["0710807000"], "냉동대파": ["0710809010"],
       "냉동생강": ["0910113000", "0910123000"], "냉동채소기타": ["0710809000", "0710809090"], "다데기": ["2103909050"],
       "생강건조": ["0910112000", "0910122000", "0910129000"], "생강신선": ["0910100000", "0910101000", "0910111000"], "소스기타": ["2103909090"], "혼합조미료": ["2103909030"]}
codes = [c for v in GRP.values() for c in v]
con = duckdb.connect(); con.execute(f"ATTACH '{KCS.replace(os.sep, '/')}' AS s (READ_ONLY)")
ps_new = con.sql(f\"\"\"SELECT yyyymm, stat_cd, hs10, imp_dlr, imp_wgt FROM s.fact_trade
    WHERE hs10 IN ({",".join(repr(c) for c in codes)}) AND yyyymm BETWEEN 200701 AND 202607 AND (imp_dlr > 0 OR imp_wgt > 0)\"\"\").df()
ps_new["grp"] = ps_new.hs10.map({c: g for g, v in GRP.items() for c in v}); ps_new["yr"] = ps_new.yyyymm // 100
ps_new = ps_new.sort_values(["grp", "hs10", "stat_cd", "yyyymm"]).reset_index(drop=True)
if os.path.exists(OUT):
    old = pd.read_csv(OUT, dtype={"hs10": str}, keep_default_na=False)
    m = old.merge(ps_new, on=["yyyymm", "stat_cd", "hs10"], how="outer", suffixes=("_o", "_n"), indicator=True)
    same = (m._merge == "both").all() and (m.imp_dlr_o == m.imp_dlr_n).all() and (m.imp_wgt_o == m.imp_wgt_n).all() and (m.grp_o == m.grp_n).all()
    print(f"기존 {len(old):,}행 / 재생성 {len(ps_new):,}행 / 같음: {bool(same)}")
else:
    ps_new.to_csv(OUT, index=False, encoding="utf-8-sig"); print(f"{OUT}: {len(ps_new):,}행")

## 14. 분석 — 논문 A의 수치를 만드는 셀

아래 셀은 세 논문 공용 셀 창고(`cache/cells.py`)에서 논문 A의 본문·표 수치를 만드는 것만 순서대로 옮긴 것이다. 첫째 질문의 격차 회귀(§2), 혼합 역산·감시형 세분·적응 속도(§3 일부·§4a·4d·4f·4h·4i)는 논문 B의 것이라 뺐고, 여러 논문이 함께 쓰는 셀(§1b 자료 수치, §3d 파일럿, §4e 이질성, §5 강건성)은 논문 A가 인용하는 줄만 남겼다. 마지막 셀 §6은 논문 A가 인용한 수치 389건을 `EXPECT_A`(라벨 → (인용값, 허용 오차))로 대조해 하나라도 어긋나면 멈추고, 등록값을 `outputs/논문A_수치.json`, 결과를 `outputs/논문A_검증_결과.json`에 남긴다. 입력은 앞의 스크립트가 만든 `outputs/`의 표와 무역통계 DB이며, 이 절만 담은 `논문A_분석.ipynb`를 `research/`에서 실행하면 약 5분 걸린다.

In [ ]:
import os, re, json, warnings, glob
import numpy as np, pandas as pd
import duckdb
import statsmodels.api as sm
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 120)
# 이 노트북은 KCSTARIFF/research/ 에서 연다. 무역통계(KCSDB2)는 환경변수 KCSDB2_ROOT 또는 기본 경로에서 읽는다.
A = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(A, "outputs", "panel")):
    up = os.path.dirname(A)
    if up == A: raise FileNotFoundError("research/outputs/panel 을 찾지 못했습니다")
    A = up
O = os.path.join(A, "outputs"); P = os.path.join(O, "panel")
KCSDB2 = os.environ.get("KCSDB2_ROOT", r"C:\Work\Projects\KCSDB2")
KCS = os.path.join(KCSDB2, "data", "processed", "kcsdb.duckdb"); TAR = os.path.join(os.path.dirname(A), "data", "processed", "kcstariff.duckdb")
CHK = {}
def chk(label, value, digits=2):
    """본문 인용 수치를 등록한다. §6에서 문서와 대조."""
    CHK[label] = round(float(value), digits) if isinstance(value, (int, float, np.floating, np.integer)) else value
    return CHK[label]

def fe_ols(df, y, xcols, fe, cluster, tol=1e-9, maxit=200):
    """다중 고정효과 OLS: 교대 사영으로 중심화한 뒤 OLS, 군집 표준오차(cluster)."""
    d = df[[y] + xcols + fe + [cluster]].dropna().copy()
    Z = d[[y] + xcols].to_numpy(dtype=float)
    groups = [d[f].astype("category").cat.codes.to_numpy() for f in fe]
    for _ in range(maxit):
        Z0 = Z.copy()
        for g in groups:
            m = np.zeros((g.max() + 1, Z.shape[1])); n = np.bincount(g)
            np.add.at(m, g, Z); Z = Z - m[g] / n[g][:, None]
        if np.abs(Z - Z0).max() < tol: break
    yv, X = Z[:, 0], Z[:, 1:]
    XtX_inv = np.linalg.pinv(X.T @ X); b = XtX_inv @ X.T @ yv; e = yv - X @ b
    cl = d[cluster].astype("category").cat.codes.to_numpy(); G = cl.max() + 1
    S = np.zeros((G, X.shape[1])); np.add.at(S, cl, X * e[:, None]); meat = S.T @ S
    n, k = X.shape; V = XtX_inv @ meat @ XtX_inv * (G / (G - 1)) * ((n - 1) / (n - k))
    se = np.sqrt(np.diag(V)); t = b / se
    return pd.DataFrame({"coef": b, "se": se, "t": t, "within_sd": X.std(axis=0)}, index=xcols).assign(n=n, clusters=G)
print("research:", A); print("KCSDB2:", KCSDB2)

### §1 자료 — 패널·코드 쌍·처리군·집행 지정 목록·환율

In [ ]:
pc = pd.read_parquet(os.path.join(P, "panel_codes.parquet"))
pp = pd.read_parquet(os.path.join(P, "panel_pairs.parquet"))
pairs = pd.read_csv(os.path.join(O, "코드쌍_목록.csv"), dtype={"hs10_high": str, "hs10_low": str})
pairs = pairs[(pairs.확인.fillna("") == "Y") | pairs.source.isin(["별표", "별표(선택)"])].copy()
pairs["src"] = np.where(pairs.source.isin(["별표", "별표(선택)"]), "별표", "규칙")
pairs["agri"] = pairs.hs10_high.str[:2].astype(int) <= 24
treat = pd.read_csv(os.path.join(O, "처리군_목록.csv"), dtype={"hs10": str, "mate_hs10": str, "pred_codes": str})
pre = pd.read_csv(os.path.join(O, "사전세액심사_대상_이력_2016_2026.csv"), dtype=str)
dist = pd.read_csv(os.path.join(O, "유통이력_신고물품_통합_구간_2009_2026.csv"), dtype=str)
fx = pd.read_csv(os.path.join(O, "환율_월별_USDKRW.csv"))
fxy = fx.groupby("year").krw_per_usd.mean()
print(f"panel_codes {len(pc):,} / panel_pairs {len(pp):,} / 쌍 {len(pairs)} (별표 {int((pairs.src=='별표').sum())}, 규칙 {int((pairs.src=='규칙').sum())}) / 처리군 {len(treat)}")
chk("쌍 수", len(pairs), 0); chk("쌍 코드 수", len(set(pairs.hs10_high) | set(pairs.hs10_low)), 0)
chk("쌍 별표", int((pairs.src == "별표").sum()), 0); chk("쌍 규칙", int((pairs.src == "규칙").sum()), 0)
chk("패널 코드 수", pc.hs10.nunique(), 0); chk("패널 원산지 수", pc.stat_cd.nunique(), 0)
for (rv, ty), n in treat.groupby(["rev", "type"]).size().items(): chk(f"처리군 {rv} {ty}", n, 0)
for st, n in treat[treat.type.str.startswith("감시형")].groupby("sub_type").size().items(): chk(f"감시형 sub_type {st}", n, 0)
chk("사전세액심사 코드 수", pre.hs10.nunique(), 0); chk("유통이력 코드 수", dist.hs10.nunique(), 0); chk("유통이력 구간 수", len(dist), 0)
cg = pd.read_csv(os.path.join(O, "패널_연속성_그룹.csv"))
cg1 = cg[cg.before_musd >= 1]
chk("연속성 집단 수", len(cg), 0); chk("연속성 집단(1백만 달러 이상)", len(cg1), 0); chk("연속성 비율 중위", cg1.ratio.median(), 2); chk("연속성 비율 10분위", cg1.ratio.quantile(.1), 2); chk("연속성 비율 90분위", cg1.ratio.quantile(.9), 2)
chk("연속성 총액 비율", cg1.after_musd.sum() / cg1.before_musd.sum(), 2); chk("잔여 통관 최대 개월", cg.residual_months.max(), 0)

In [ ]:
# 1b 자료 수치(셀 창고 §1b 가운데 논문 A가 인용하는 줄만): 세율 DB·양허 별표·품목 상세·미러·수집 검증 CSV
con1 = duckdb.connect(); con1.execute(f"ATTACH '{TAR.replace(os.sep, '/')}' AS tr (READ_ONLY)"); con1.execute(f"ATTACH '{KCS.replace(os.sep, '/')}' AS s (READ_ONLY)")
ar = con1.sql("SELECT * FROM tr.fct_applied_rate").df()
chk("자료 하한 있는 행", int(ar.floor_won_kg.notna().sum()), 0)
by = pd.read_csv(os.path.join(O, "양허관세_별표1_2025.csv"), dtype={"hs10": str})
chk("자료 별표 1의 가 행", int((by.byeolpyo == "1의 가").sum()), 0); chk("자료 별표 1의 나 행", int((by.byeolpyo == "1의 나").sum()), 0)
ho = by[by.higher_of.astype(str).str.lower() == "true"]
chk("자료 종량 대안 코드", ho.hs10.nunique(), 0); chk("자료 종량 대안 코드 가", ho[ho.byeolpyo == "1의 가"].hs10.nunique(), 0); chk("자료 종량 대안 코드 나", ho[ho.byeolpyo == "1의 나"].hs10.nunique(), 0)
fl_ = ho.specific_won_kg / (ho.adval / 100); chk("자료 하한 최소 원", fl_.min(), 0); chk("자료 하한 최대 원", fl_.max(), 0)
dd = pd.read_csv(os.path.join(O, "품목상세_세율_2012_2026.csv"), dtype={"hs10": str, "year": int})
seven = ["FCN1", "FEU1", "FUS1", "FAS1", "FIN1", "FVN1", "FCA1"]
mr = con1.sql("SELECT year, hs10, rate_cd, adval FROM tr.tariff_rate WHERE source='main' AND rate_cd IN ('FCN1','FEU1','FUS1','FAS1','FIN1','FVN1','FCA1') AND month(valid_from)=1 AND day(valid_from)=1").df()
cmp_ = dd[dd.rate_cd.isin(seven)].merge(mr, on=["year", "hs10", "rate_cd"])
chk("자료 일곱 상대 대조 행", len(cmp_), 0); chk("자료 일곱 상대 대조 어긋남", int((cmp_.adval_first.astype(float).round(2) != cmp_.adval.round(2)).sum()), 0)
chk("자료 품목 상세 코드", dd.hs10.nunique(), 0); chk("자료 품목 상세 구분기호", dd.rate_cd.nunique(), 0); chk("자료 선택1 협정", dd[dd.rate_cd.str.match(r"^F[A-Z]+1$")].rate_cd.nunique(), 0)
cm_ = pd.read_csv(os.path.join(O, "comtrade_mirror_hs6_2012_2024.csv"), dtype={"cmdCode": str}); chk("자료 미러 행", len(cm_), 0); chk("자료 미러 보고국", cm_.reporter.nunique(), 0); chk("자료 미러 HS6", cm_.cmdCode.nunique(), 0)
REPO = os.path.dirname(A)
vc = pd.read_csv(os.path.join(REPO, "outputs", "수집_검증.csv"), dtype={"key": str})
for cd in ["A", "C"]:
    chk(f"자료 두 화면 일치율 {cd}", float(vc[vc.item == f"두 화면 일치율 {cd}"].value.iloc[0]), 2); chk(f"자료 두 화면 대조 쌍 {cd}", float(vc[vc.item == f"두 화면 대조 쌍 {cd}"].value.iloc[0]), 0)
vs = pd.read_csv(os.path.join(O, "실행세율_원산지별_검증_요약.csv")).set_index("item").value; chk("자료 19 일곱 상대 대조 행", float(vs["일곱 상대 대조 행"]), 0); chk("자료 19 일곱 상대 어긋남", float(vs["일곱 상대 어긋남"]), 0)

### §3 종량 하한 — 하한 아래 몫(표 2)과 건고추 저가 신고 사건, 변환비 쌍의 단가 비율, 미러(표 3·표 7과 후보 대조), 파일럿 계열(건조생강·건고추)

In [ ]:
fl = pc[pc.floor_won_kg.notna() & (pc.imp_wgt > 0)].merge(fx[["yyyymm", "krw_per_usd"]], on="yyyymm", how="left")
fl["uv_won"] = fl.uv * fl.krw_per_usd; fl["below"] = fl.uv_won < fl.floor_won_kg
fl["ratio"] = fl.uv_won / fl.floor_won_kg
def wshare(g): return np.average(g.below, weights=g.imp_dlr)
t8 = fl[fl.year >= 2012].groupby(["hs10"]).apply(lambda g: pd.Series(dict(코드수=1, 수입_백만달러=g.imp_dlr.sum()/1e6, 하한_원kg=g.floor_won_kg.iloc[-1], 하한아래_금액몫=wshare(g), 중국_하한아래=wshare(g[g.stat_cd=="CN"]) if (g.stat_cd=="CN").any() else np.nan, 비율중위=g.ratio.median()))).reset_index()
names = pd.read_csv(os.path.join(KCSDB2, "data", "external", "HSK_별표", "HSK_별표_2025.csv"), dtype=str).set_index("code").leaf
t8["품명"] = t8.hs10.map(names)
t8 = t8.sort_values("수입_백만달러", ascending=False)
display(t8.head(25).round(3))
print("하한 코드 수", len(t8), " 금액 가중 하한 아래 몫:", round(np.average(t8.하한아래_금액몫, weights=t8.수입_백만달러), 3))
chk("하한 코드 수", len(t8), 0); chk("하한 아래 몫 전체", np.average(t8.하한아래_금액몫, weights=t8.수입_백만달러), 3)
for h in ["0904210000", "0910112000", "1207400000", "1201909000"]:
    if h in set(t8.hs10): chk(f"하한 아래 몫 {h}", float(t8.set_index("hs10").loc[h, "하한아래_금액몫"]), 3)
t8i = t8.set_index("hs10")
for h in ["1201901000", "1201909000", "1207400000", "1207991000", "0904210000", "0409000000", "0703209000"]:
    if h in t8i.index:
        chk(f"하한 표 {h} 수입", t8i.loc[h, "수입_백만달러"], 0); chk(f"하한 표 {h} 하한", t8i.loc[h, "하한_원kg"], 0); chk(f"하한 표 {h} 몫", t8i.loc[h, "하한아래_금액몫"], 3); chk(f"하한 표 {h} 비율중위", t8i.loc[h, "비율중위"], 2)
# 건고추 저가 신고 사건(조세심판원 2024-05-21 결정, 조심 2023관0105; 논문 A IV.1): 결정문의 신고 단가 1,600달러/톤을 2022-11 월평균 환율로 원/kg으로 바꿔 종량 하한·종량세와 비교한다
fx2211 = float(fx[fx.yyyymm == 202211].krw_per_usd.iloc[0]); chk("환율 2022.11", fx2211, 1)
uv_case = 1600 / 1000 * fx2211; chk("사건 건고추 신고 단가 원/kg", uv_case, 0)
ar22 = duckdb.connect().execute(f"ATTACH '{TAR.replace(os.sep, '/')}' AS tr (READ_ONLY)").sql("SELECT floor_won_kg, specific_won_kg FROM tr.fct_applied_rate WHERE year=2022 AND hs10='0904210000'").df().iloc[0]
chk("사건 건고추 하한 2022", ar22.floor_won_kg, 0); chk("사건 건고추 종량세 2022", ar22.specific_won_kg, 0); chk("사건 건고추 하한 아래", int(uv_case < ar22.floor_won_kg), 0)
# I장 예: 건고추(270%)와 냉동고추(27%)의 세율 차이 — 세율 DB의 2022년 무협정 실행세율
mf22 = duckdb.connect().execute(f"ATTACH '{TAR.replace(os.sep, '/')}' AS tr (READ_ONLY)").sql("SELECT hs10, mfn FROM tr.fct_applied_rate WHERE year=2022 AND hs10 IN ('0904210000','0710807000')").df().set_index("hs10").mfn
chk("건고추 mfn 2022", mf22["0904210000"], 1); chk("냉동고추 mfn 2022", mf22["0710807000"], 1)

In [ ]:
# 변환비 쌍의 단가 비율 r = uv_H / (uv_L × k), 원산지×연도
CONV = {"0710807000": ("0904210000", 5.0), "0811902000": ("0813402000", 3.0), "0714104000": ("0714101000", 1.0), "0714104000b": ("0714103000", 1.0), "0714401000": ("0714409000", 1.0)}
conv_pairs = pairs[pairs.apply(lambda r: (r.hs10_low, r.hs10_high) in {("0710807000", "0904210000"), ("0811902000", "0813402000"), ("0714104000", "0714101000"), ("0714104000", "0714103000"), ("0714401000", "0714409000"), ("0710807000", "0904220000")}, axis=1)].copy()
kmap = {"0904210000": 5.0, "0904220000": 5.0, "0813402000": 3.0, "0714101000": 1.0, "0714103000": 1.0, "0714409000": 1.0}
rows = []
for r in conv_pairs.itertuples(index=False):
    x = pp[pp.pair_id == r.pair_id].groupby(["stat_cd", "year"]).agg(VH=("imp_dlr_H", "sum"), QH=("imp_wgt_H", "sum"), VL=("imp_dlr_L", "sum"), QL=("imp_wgt_L", "sum")).reset_index()
    x = x[(x.QH > 0) & (x.QL > 0)]; x["r"] = (x.VH / x.QH) / ((x.VL / x.QL) * kmap[r.hs10_high]); x["pair_id"] = r.pair_id; x["high"] = r.hs10_high; x["low"] = r.hs10_low
    rows.append(x)
rr = pd.concat(rows)
t9 = rr[rr.stat_cd == "CN"].pivot_table(index="year", columns="pair_id", values="r").round(2)
display(t9)
c = rr[(rr.stat_cd == "CN") & (rr.high == "0904210000") & (rr.low == "0710807000")].set_index("year").r
chk("r 건고추/냉동고추 중국 2012", c.get(2012, np.nan), 2); chk("r 건고추/냉동고추 중국 2024", c.get(2024, np.nan), 2)
c12 = c[c.index >= 2012]; chk("r 건고추/냉동고추 중국 최소", c12.min(), 2); chk("r 건고추/냉동고추 중국 최대", c12.max(), 2)
cj = rr[(rr.stat_cd == "CN") & (rr.high == "0813402000") & (rr.low == "0811902000") & (rr.year >= 2017)].r
chk("r 건대추/냉동대추 중국 최소", cj.min(), 2); chk("r 건대추/냉동대추 중국 최대", cj.max(), 2)
# 건대추 쌍(2026-09-13): 비율 0.02~0.06은 냉동대추 물량이 0톤인 해의 표본 단가에서 나온다. 두 코드 모두 5톤 이상인 해만 따로 등록하고 중국산 냉동대추 톤·단가를 남긴다
cjt = rr[(rr.stat_cd == "CN") & (rr.high == "0813402000") & (rr.low == "0811902000") & (rr.year >= 2017) & (rr.year <= 2025)].set_index("year")
cj5 = cjt[(cjt.QH >= 5000) & (cjt.QL >= 5000)]
chk("r 건대추/냉동대추 중국 5톤 이상 최소", cj5.r.min(), 2); chk("r 건대추/냉동대추 중국 5톤 이상 최대", cj5.r.max(), 2); chk("r 건대추/냉동대추 5톤 이상 해 수", len(cj5), 0)
chk("r 건대추/냉동대추 5톤 이상 첫해", int(cj5.index.min()), 0); chk("r 건대추/냉동대추 5톤 이상 끝해", int(cj5.index.max()), 0)
cj0 = cjt[cjt.QL < 5000]; chk("냉동대추 중국 5톤 미만 해 수", len(cj0), 0); chk("냉동대추 중국 5톤 미만 해 단가 최소", (cj0.VL / cj0.QL).min(), 0); chk("냉동대추 중국 5톤 미만 해 단가 최대", (cj0.VL / cj0.QL).max(), 0); chk("r 건대추/냉동대추 5톤 미만 최대", cj0.r.max(), 2)
for y in [2018, 2019, 2020, 2021, 2022, 2024]: chk(f"냉동대추 중국 톤 {y}", cjt.QL[y] / 1000, 1); chk(f"냉동대추 중국 단가 {y}", cjt.VL[y] / cjt.QL[y], 2)
chk("건대추 중국 톤 2017~2025 최소", cjt.QH.min() / 1000, 0); chk("건대추 중국 톤 2017~2025 최대", cjt.QH.max() / 1000, 0); chk("건대추 중국 단가 2017~2025 최소", (cjt.VH / cjt.QH).min(), 2); chk("건대추 중국 단가 2017~2025 최대", (cjt.VH / cjt.QH).max(), 2)
chk("건대추 하한 원", float(pc[pc.hs10 == "0813402000"].floor_won_kg.dropna().iloc[-1]), 0)
jj_ = pp[(pp.pair_id == cjt.pair_id.iloc[0]) & (pp.year >= 2017) & (pp.year <= 2025)]; chk("건대추 중국 몫 2017~2025", jj_[jj_.stat_cd == "CN"].imp_dlr_H.sum() / jj_.imp_dlr_H.sum(), 2)

In [ ]:
# 3c 미러 통계(scripts/21·22의 산출을 읽어 본문 수치를 등록한다. 갭 회귀 자체는 22가 정본)
mg = pd.read_csv(os.path.join(O, "미러_갭_회귀.csv")); mp = pd.read_csv(os.path.join(O, "미러_갭_패널.csv"), dtype={"grp": str})
gg = pd.read_csv(os.path.join(O, "미러_건조생강.csv"), dtype={"hs6": str}); tg = pd.read_csv(os.path.join(O, "미러_단가갭_HS6원산지.csv"), dtype={"hs6": str})
display(mg); display(gg[["hs6", "year", "uv_x", "uv_m", "gap_uv"]].round(2))
chk("미러 패널 관측", len(mp), 0); chk("미러 집단 수", mp.grp.nunique(), 0); chk("미러 보고국 수", mp.reporter.nunique(), 0)
for row in mg.itertuples(index=False): chk(f"미러 {row.dep} {row.var}", row.coef, 4); chk(f"미러 {row.dep} {row.var} t", row.t, 2)
g12 = gg[gg.hs6 == "091012"].set_index("year")
for y in [2017, 2020, 2021, 2022, 2023, 2024]: chk(f"건조생강 미러 단가 갭 {y}", float(g12.gap_uv.get(y, np.nan)), 2)
chk("건조생강 중국 보고 단가 2020", float(g12.uv_x.get(2020, np.nan)), 2); chk("건조생강 한국 단가 2020", float(g12.uv_m.get(2020, np.nan)), 2)
ta = tg[tg.agri]; chk("농식품 단가 갭 중위", ta.gap_uv.median(), 2); chk("농식품 단가 갭 90분위", ta.gap_uv.quantile(.9), 2)
chk("농식품 HS6×보고국 수", len(ta), 0)
for i, v in enumerate(ta.sort_values("gap_uv", ascending=False).gap_uv.head(5).tolist(), 1): chk(f"농식품 단가 갭 상위 {i}", v, 2)
for row in mg.itertuples(index=False): chk(f"미러 {row.dep} {row.var} 관측", row.n, 0); chk(f"미러 {row.dep} {row.var} 군집", row.clusters, 0)
chk("미러 갭 중위 금액", mp.gap_v.median(), 2); chk("미러 갭 중위 물량", mp.gap_q.median(), 2); chk("미러 갭 중위 단가", mp.gap_uv.median(), 2)
for y in range(2017, 2025):
    if y in g12.index: chk(f"건조생강 미러 단가 갭 {y}", float(g12.gap_uv[y]), 2); chk(f"건조생강 중국 보고 단가 {y}", float(g12.uv_x[y]), 2); chk(f"건조생강 한국 단가 {y}", float(g12.uv_m[y]), 2); chk(f"건조생강 중국 보고 톤 {y}", float(g12.x_kg[y]) / 1000, 0); chk(f"건조생강 한국 톤 {y}", float(g12.m_kg[y]) / 1000, 0)
g11 = gg[gg.hs6 == "091011"]
chk("신선생강 미러 단가 갭 최소", g11.gap_uv.min(), 2); chk("신선생강 미러 단가 갭 최대", g11.gap_uv.max(), 2); chk("신선생강 중국 보고 단가 최소", g11.uv_x.min(), 2); chk("신선생강 중국 보고 단가 최대", g11.uv_x.max(), 2); chk("신선생강 한국 단가 최소", g11.uv_m.min(), 2); chk("신선생강 한국 단가 최대", g11.uv_m.max(), 2)

In [ ]:
# 3e 미러 단가 갭 후보의 통관 대조(2026-09-13): 농식품 단가 갭 상위 5(HS6×보고국)의 10단위 코드별 연 단가(그 원산지)·다른 원산지 단가·미러의 금액·물량 갭·적용 세율·하한·집행 지정
mh = pd.read_csv(os.path.join(O, "미러_갭_HS6_2017_2024.csv"), dtype={"hs6": str})
top5 = tg[tg.agri].sort_values("gap_uv", ascending=False).head(5)
con5 = duckdb.connect(); con5.execute(f"ATTACH '{KCS.replace(os.sep, '/')}' AS s (READ_ONLY)"); con5.execute(f"ATTACH '{TAR.replace(os.sep, '/')}' AS tr (READ_ONLY)")
ar24 = con5.sql("SELECT hs10, mfn, floor_won_kg, applied_vn, applied_cn, applied_asean FROM tr.fct_applied_rate WHERE year=2024").df().set_index("hs10")
dsg = dist.groupby("hs10").agg(start=("start", "min"), end=("end", "max"))
rows, summ = [], []
for c in top5.itertuples(index=False):
    q = con5.sql(f"SELECT hs10, stat_cd, yyyymm//100 AS yr, sum(imp_dlr) v, sum(imp_wgt) w FROM s.fact_trade WHERE substr(hs10,1,6)='{c.hs6}' AND yyyymm BETWEEN 201701 AND 202512 AND imp_wgt>0 AND imp_dlr>0 GROUP BY 1,2,3").df()
    g = mh[(mh.reporter == c.reporter) & (mh.hs6 == c.hs6)].set_index("year"); g5 = g[g.index <= 2021]
    for h, qq in q.groupby("hs10"):
        tot = qq.groupby("yr").agg(v=("v", "sum"), w=("w", "sum")); og = qq[qq.stat_cd == c.reporter].groupby("yr").agg(v=("v", "sum"), w=("w", "sum")); oth = qq[qq.stat_cd != c.reporter].groupby("yr").agg(v=("v", "sum"), w=("w", "sum"))
        for y in og.index:
            rows.append(dict(reporter=c.reporter, hs6=c.hs6, hs10=h, name=names.get(h, ""), year=int(y), ton=og.w[y] / 1000, musd=og.v[y] / 1e6, uv=og.v[y] / og.w[y], uv_others=(oth.v[y] / oth.w[y]) if y in oth.index and oth.w[y] > 0 else np.nan,
                             share_origin=og.v[y] / tot.v[y], gap_v=g.gap_v.get(y, np.nan), gap_q=g.gap_q.get(y, np.nan), gap_uv=g.gap_uv.get(y, np.nan), x_ton=g.x_kg.get(y, np.nan) / 1000, m_ton=g.m_kg.get(y, np.nan) / 1000))
    d = pd.DataFrame([r for r in rows if r["reporter"] == c.reporter and r["hs6"] == c.hs6])
    h0 = d.groupby("hs10").musd.sum().idxmax(); x = d[d.hs10 == h0]; xb = x[x.musd > 0.5]
    a = ar24.loc[h0] if h0 in ar24.index else None
    summ.append(dict(reporter=c.reporter, hs6=c.hs6, gap_uv_2017_2024=c.gap_uv, gap_v_to2021=g5.gap_v.mean(), gap_q_to2021=g5.gap_q.mean(), gap_uv_to2021=g5.gap_uv.mean(), x_over_m_ton=g5.x_kg.sum() / g5.m_kg.sum(),
                     hs10=h0, name=names.get(h0, ""), share_of_hs6=x.musd.sum() / d.musd.sum(), uv_min=x.uv.min(), uv_max=x.uv.max(), uv_slope=np.polyfit(xb.year, np.log(xb.uv), 1)[0] if len(xb) >= 4 else np.nan, ratio_others_median=x.ratio_to_others.median() if "ratio_to_others" in x else (x.uv / x.uv_others).median(),
                     mfn=a.mfn if a is not None else np.nan, applied_origin=(a.applied_vn if c.reporter == "VN" else a.applied_cn) if a is not None else np.nan, floor_won_kg=a.floor_won_kg if a is not None else np.nan,
                     pre_review=h0 in set(pre.hs10), dist_start=dsg.start.get(h0, ""), dist_end=dsg.end.get(h0, "")))
cand = pd.DataFrame(rows); cand["ratio_to_others"] = cand.uv / cand.uv_others
cand.to_csv(os.path.join(O, "미러_후보_통관대조.csv"), index=False, encoding="utf-8-sig")
t7b = pd.DataFrame(summ); t7b.to_csv(os.path.join(O, "미러_후보_통관대조_요약.csv"), index=False, encoding="utf-8-sig"); display(t7b.round(2))
for r_ in t7b.itertuples(index=False):
    k = f"{r_.reporter} {r_.hs6}"
    chk(f"후보 {k} 금액 갭", r_.gap_v_to2021, 2); chk(f"후보 {k} 물량 갭", r_.gap_q_to2021, 2); chk(f"후보 {k} 단가 갭", r_.gap_uv_to2021, 2); chk(f"후보 {k} 상대국 톤/한국 톤", r_.x_over_m_ton, 2)
    chk(f"후보 {k} 대표 코드 몫", r_.share_of_hs6, 2); chk(f"후보 {k} 단가 최소", r_.uv_min, 2); chk(f"후보 {k} 단가 최대", r_.uv_max, 2); chk(f"후보 {k} 단가 기울기", r_.uv_slope, 3); chk(f"후보 {k} 다른 원산지 대비 중위", r_.ratio_others_median, 2)
    chk(f"후보 {k} mfn", r_.mfn, 1); chk(f"후보 {k} 적용 세율", r_.applied_origin, 1); chk(f"후보 {k} 하한 있음", int(pd.notna(r_.floor_won_kg)), 0); chk(f"후보 {k} 유통이력", int(bool(r_.dist_start)), 0); chk(f"후보 {k} 사전세액심사", int(r_.pre_review), 0)
chk("후보 금액 갭 절대치 최대(베트남)", t7b[t7b.reporter == "VN"].gap_v_to2021.abs().max(), 2); chk("후보 물량 갭 최소", t7b.gap_q_to2021.min(), 2); chk("후보 물량 갭 최대", t7b.gap_q_to2021.max(), 2)
# 2022년 이후 베트남 보고 금액이 한국 수입액을 넘어선다(경유·재수출)
v22 = mh[(mh.reporter == "VN") & mh.hs6.isin(top5[top5.reporter == "VN"].hs6) & (mh.year >= 2022)]; chk("후보 베트남 2022~ 금액 갭 최소", v22.gap_v.min(), 2); chk("후보 베트남 2022~ 금액 갭 최대", v22.gap_v.max(), 2)
# 견과 조제품: 2022년 볶은 참깨 분리 뒤 기타 코드의 베트남 단가
s19 = cand[(cand.reporter == "VN") & (cand.hs10 == "2008199000")].set_index("year"); chk("후보 견과 기타 베트남 단가 2021", s19.uv[2021], 2); chk("후보 견과 기타 베트남 단가 2025", s19.uv[2025], 2)
s193 = cand[(cand.reporter == "VN") & (cand.hs10 == "2008193000")]; chk("후보 볶은 참깨 베트남 천 톤 최소", s193.ton.min() / 1000, 0); chk("후보 볶은 참깨 베트남 천 톤 최대", s193.ton.max() / 1000, 0); chk("후보 볶은 참깨 베트남 단가 최소", s193.uv.min(), 2); chk("후보 볶은 참깨 베트남 단가 최대", s193.uv.max(), 2)

In [ ]:
# 3d 파일럿(고추·생강)의 인용 수치: outputs/pilot_series.csv에서 계산해 등록한다. 계산 근거는 파일럿_고추_생강.md
ps = pd.read_csv(os.path.join(O, "pilot_series.csv"), dtype={"hs10": str})
def ann(g, cc="CN"):
    x = ps[(ps.grp == g) & (ps.stat_cd == cc)].groupby("yr").agg(v=("imp_dlr", "sum"), w=("imp_wgt", "sum")); x = x[x.w > 0]; x["uv"] = x.v / x.w; x["ton"] = x.w / 1000; return x
def mon(g, cc="CN"):
    x = ps[(ps.grp == g) & (ps.stat_cd == cc)].groupby("yyyymm").agg(v=("imp_dlr", "sum"), w=("imp_wgt", "sum")); x["uv"] = np.where(x.w > 0, x.v / x.w.replace(0, np.nan), np.nan); x["ton"] = x.w / 1000; return x
gd, gf = ann("생강건조"), ann("생강신선")
for y in [2012, 2020, 2021]: chk(f"파일럿 건조생강 중국 단가 {y}", gd.uv[y], 2); chk(f"파일럿 건조/신선×10 {y}", gd.uv[y] / (gf.uv[y] * 10), 2)
gm = mon("생강건조"); chk("파일럿 건조생강 중국 2021.11 톤", float(gm.ton.get(202111, 0)), 0)
g22 = gm[(gm.index >= 202201)]; chk("파일럿 건조생강 중국 2022~ 월 톤 최대", g22.ton.max(), 0)
gda = gd.loc[2022:2025]; chk("파일럿 건조생강 중국 연 단가 2022~2025 최소", gda.uv.min(), 2); chk("파일럿 건조생강 중국 연 단가 2022~2025 최대", gda.uv.max(), 2)
pe = mon("생강건조", "PE"); pe = pe[(pe.index >= 202111) & (pe.index <= 202209) & (pe.w > 0)]
chk("파일럿 건조생강 페루 월 톤 최소", pe.ton.min(), 0); chk("파일럿 건조생강 페루 월 톤 최대", pe.ton.max(), 0); chk("파일럿 건조생강 페루 단가 최소", pe.uv.min(), 2); chk("파일럿 건조생강 페루 단가 최대", pe.uv.max(), 2)
vn = ann("생강건조", "VN"); chk("파일럿 건조생강 베트남 단가 2022", float(vn.uv.get(2022, np.nan)), 2)
fr, dr = ann("냉동고추"), ann("건고추"); yy = [y for y in range(2007, 2026) if y in fr.index and y in dr.index]
chk("파일럿 냉동고추 중국 연 톤 최소", fr.ton.loc[yy].min(), 0); chk("파일럿 냉동고추 중국 연 톤 최대", fr.ton.loc[yy].max(), 0); chk("파일럿 건고추 중국 연 톤 최소", dr.ton.loc[yy].min(), 0); chk("파일럿 건고추 중국 연 톤 최대", dr.ton.loc[yy].max(), 0)
chk("파일럿 냉동/건조 톤 배수 최소", (fr.ton.loc[yy] / dr.ton.loc[yy]).min(), 0); chk("파일럿 냉동/건조 톤 배수 최대", (fr.ton.loc[yy] / dr.ton.loc[yy]).max(), 0)
chk("파일럿 냉동×5/건조 최소", (5 * fr.uv.loc[yy] / dr.uv.loc[yy]).min(), 2); chk("파일럿 냉동×5/건조 최대", (5 * fr.uv.loc[yy] / dr.uv.loc[yy]).max(), 2)
d35 = dr.loc[2023:2025]; chk("파일럿 건고추 중국 단가 2023~2025 최소", d35.uv.min(), 2); chk("파일럿 건고추 중국 단가 2023~2025 최대", d35.uv.max(), 2)
fr_ = pd.Series({y: 2300 / (d35.uv[y] * fxy[y]) for y in d35.index}); chk("파일럿 건고추 하한/단가 최소", fr_.min(), 2); chk("파일럿 건고추 하한/단가 최대", fr_.max(), 2)
gr_ = pd.Series({y: 247 / (gd.uv[y] * fxy[y]) for y in range(2022, 2026) if y in gd.index}); chk("파일럿 생강 하한/건조 단가 최소", gr_.min(), 2); chk("파일럿 생강 하한/건조 단가 최대", gr_.max(), 2)
gr2 = pd.Series({y: 247 / (gf.uv[y] * 10 * fxy[y]) for y in range(2012, 2022) if y in gf.index}); chk("파일럿 생강 하한/신선×10 최소", gr2.min(), 2); chk("파일럿 생강 하한/신선×10 최대", gr2.max(), 2)

### §4 집행 지정의 사건연구 — 표 4(사전세액심사·유통이력·건조생강 두 사건일), 표 5(코드별 단절), 표 6(지정 전 과소신고 유무 분할)

In [ ]:
def event_study(series, treated, events, controls, y="lnq", win=24, min_post=6):
    """series: (hs10, stat_cd, yyyymm, y). treated: dict hs10->event yyyymm. controls: hs10 목록(사건 없음).
    처리 코드마다 자기 사건월을, 대조 코드는 같은 창을 처리 코호트별로 쌓는다(stacked)."""
    def mdiff(a, b): return (a // 100 - b // 100) * 12 + (a % 100 - b % 100)
    stacks = []
    for h, m in treated.items():
        ctrl = controls[h] if isinstance(controls, dict) else controls
        s = series[series.hs10.isin([h] + list(ctrl))].copy()
        s["j"] = s.yyyymm.map(lambda x: mdiff(x, m)); s = s[(s.j >= -win) & (s.j <= win)]
        s["tr"] = (s.hs10 == h).astype(int); s["stk"] = h; stacks.append(s)
    d = pd.concat(stacks); d = d[d.groupby(["stk", "hs10", "stat_cd"]).j.transform("size") >= 24]
    # 대조 코드는 처리 코드가 있는 원산지에서만, 그리고 창 안 수입액 상위 100개까지만 쓴다
    tro = set(d[d.tr == 1].stat_cd); d = d[d.stat_cd.isin(tro)]
    top = d[d.tr == 0].groupby("hs10").v.sum().sort_values(ascending=False).head(100).index
    d = d[(d.tr == 1) | d.hs10.isin(top)]
    d["ic"] = d.stk + "|" + d.hs10 + "|" + d.stat_cd; d["ct"] = d.stk + "|" + d.stat_cd + "|" + d.yyyymm.astype(str)
    js = [j for j in range(-win, win + 1) if j != -1]
    for j in js: d[f"D{j}"] = ((d.j == j) & (d.tr == 1)).astype(float)
    xs = [f"D{j}" for j in js if d[f"D{j}"].sum() > 0]
    r = fe_ols(d, y, xs, ["ic", "ct"], "hs10"); r["j"] = [int(c[1:]) for c in r.index]
    return r.reset_index(drop=True), d
def monthly(codes):
    con2 = duckdb.connect(); con2.execute(f"ATTACH '{KCS.replace(os.sep, '/')}' AS s (READ_ONLY)")
    q = con2.sql(f"""SELECT hs10, stat_cd, yyyymm, sum(imp_dlr) v, sum(imp_wgt) w FROM s.fact_trade WHERE hs10 IN ({",".join("'"+c+"'" for c in codes)}) AND imp_wgt>0 AND imp_dlr>0 GROUP BY 1,2,3""").df()
    q["lnq"] = np.log(q.w); q["lnuv"] = np.log(q.v / q.w); q["lnv"] = np.log(q.v); return q
def summarize(r, label):
    pre = r[(r.j < -1) & (r.j >= -12)]; post = r[(r.j >= 3) & (r.j <= 12)]
    out = dict(label=label, pre_mean=pre.coef.mean(), pre_absmax_t=pre.t.abs().max(), post_mean=post.coef.mean(), post_mean_t=post.coef.mean() / np.sqrt((post.se ** 2).mean() / len(post)) if len(post) else np.nan, n=int(r.n.iloc[0]), clusters=int(r.clusters.iloc[0]))
    return out
pair_codes = set(pairs.hs10_high) | set(pairs.hs10_low)
treated_all = set(treat[treat.type.isin(["감시형", "감시형(혼합 전신)", "세율형"])].hs10) | set(treat.mate_hs10.dropna())
designated = set(pre.hs10) | set(dist.hs10)
controls = sorted(pair_codes - treated_all - designated)
print("대조 코드", len(controls))

In [ ]:
# 4b 집행 지정: 사전세액심사(원산지 중국 단가·물량), 유통이력(지정 코드 물량). 파동별 지정일, 24개월 창
pre_ev = pre.assign(m=pre.aplyStrtDt.str[:7].str.replace("-", "").astype(int)).groupby("hs10").m.min().to_dict()
pre_ev = {h: m for h, m in pre_ev.items() if m >= 201401}   # 2016년 이전 표시분(2007 파동)은 창 밖
dist_ev = dist.assign(m=dist.start.str[:7].str.replace("-", "").astype(int)).groupby("hs10").m.min().to_dict()
dist_ev = {h: m for h, m in dist_ev.items() if m >= 200901}
ser2 = monthly(sorted(set(pre_ev) | set(dist_ev) | set(controls)))
out = []
for name, evd, yv, orig in [("사전세액심사 단가(중국)", pre_ev, "lnuv", "CN"), ("사전세액심사 물량(중국)", pre_ev, "lnq", "CN"), ("사전세액심사 단가(전체)", pre_ev, "lnuv", None), ("유통이력 물량(전체)", dist_ev, "lnq", None), ("유통이력 물량(중국)", dist_ev, "lnq", "CN")]:
    s = ser2 if orig is None else ser2[ser2.stat_cd == orig]
    r, d = event_study(s, evd, None, [c for c in controls if c not in evd], y=yv)
    x = summarize(r, name); x["r"] = r; out.append(x)
t11 = pd.DataFrame([{k: v for k, v in x.items() if k != "r"} for x in out]).round(3); display(t11)
for x in out: chk(x["label"] + " post", x["post_mean"], 3); chk(x["label"] + " pre|t|max", x["pre_absmax_t"], 2); chk(x["label"] + " t", x["post_mean_t"], 2); chk(x["label"] + " 관측", x["n"], 0); chk(x["label"] + " 군집", x["clusters"], 0)
ES_PRE = {x["label"]: x["r"] for x in out}

In [ ]:
# 4c 건조생강: 공식 지정일(2022-03) 대 선행 사건일(2021-11)
gin = {"0910112000": 202203, "0910122000": 202203}
res_g = {}
for lab, m in [("공식 지정 2022-03", 202203), ("선행 사건 2021-11", 202111)]:
    r, d = event_study(ser2[ser2.stat_cd == "CN"], {h: m for h in gin}, None, [c for c in controls], y="lnuv")
    res_g[lab] = summarize(r, "건조생강 단가 " + lab)
t12 = pd.DataFrame(res_g).T.round(3); display(t12)
for lab, x in res_g.items(): chk(x["label"] + " post", x["post_mean"], 3); chk(x["label"] + " pre|t|max", x["pre_absmax_t"], 2); chk(x["label"] + " t", x["post_mean_t"], 2); chk(x["label"] + " 관측", x["n"], 0); chk(x["label"] + " 군집", x["clusters"], 0)

In [ ]:
# 4e 이질성: 사건연구 평균 뒤의 코드별 단절. 처리 코드마다 (사후 3~12개월 평균 − 사전 12개월 평균)에서 대조 코드 평균의 같은 차이를 뺀 값
def code_breaks(series_all, evd, y, controls_):
    def md(a, b): return (a // 100 - b // 100) * 12 + (a % 100 - b % 100)
    out = []
    for h, m in evd.items():
        t = series_all[series_all.hs10 == h].copy(); t["j"] = t.yyyymm.map(lambda x: md(x, m))
        pre_ = t[(t.j >= -12) & (t.j <= -1)]; post = t[(t.j >= 3) & (t.j <= 12)]
        if len(pre_) < 6 or len(post) < 6: continue
        c = series_all[series_all.hs10.isin(controls_)].copy(); c["j"] = c.yyyymm.map(lambda x: md(x, m))
        cc = c[(c.j >= -12) & (c.j <= -1)].groupby("hs10")[y].mean().to_frame("a").join(c[(c.j >= 3) & (c.j <= 12)].groupby("hs10")[y].mean().to_frame("b")).dropna()
        out.append(dict(hs10=h, event=m, d=(post[y].mean() - pre_[y].mean()) - (cc.b - cc.a).mean(), pre_v_musd=pre_.v.sum() / 1e6))
    return pd.DataFrame(out)
# 사전세액심사: 중국산 단가
cn = ser2[ser2.stat_cd == "CN"]
hb = code_breaks(cn, pre_ev, "lnuv", [c for c in controls if c not in pre_ev])
rt21 = duckdb.connect().execute(f"ATTACH '{TAR.replace(os.sep, '/')}' AS tr (READ_ONLY)").sql("SELECT hs10, mfn, floor_won_kg FROM tr.fct_applied_rate WHERE year=2021").df()
hb = hb.merge(rt21, on="hs10", how="left"); hb["name"] = hb.hs10.map(pre.drop_duplicates("hs10").set_index("hs10").name_ko)
hb.to_csv(os.path.join(O, "사건연구_이질성_사전세액심사.csv"), index=False, encoding="utf-8-sig")
display(hb.sort_values("d", ascending=False).round(2).head(12))
print("중국 단가 단절 분위:", hb.d.describe(percentiles=[.1, .25, .5, .75, .9]).round(2).to_dict())
chk("사전세액심사 단가 단절 중위", hb.d.median(), 2); chk("사전세액심사 단가 단절 90분위", hb.d.quantile(.9), 2); chk("사전세액심사 단가 단절 코드 수", len(hb), 0)
chk("단가 단절 건조생강 부순 것", float(hb.set_index("hs10").d.get("0910122000", np.nan)), 2); chk("단가 단절 건조생강", float(hb.set_index("hs10").d.get("0910112000", np.nan)), 2)
for row in hb.itertuples(index=False):
    chk(f"단절 {row.hs10} d", row.d, 2); chk(f"단절 {row.hs10} mfn", row.mfn, 1); chk(f"단절 {row.hs10} 지정 전 수입", row.pre_v_musd, 1)
    if pd.notna(row.floor_won_kg): chk(f"단절 {row.hs10} 하한", row.floor_won_kg, 0)
chk("단가 단절 mfn 상관", hb[["d", "mfn"]].corr().iloc[0, 1], 2); chk("단가 단절 하한 있음 중위", hb[hb.floor_won_kg.notna()].d.median(), 2); chk("단가 단절 하한 없음 중위", hb[hb.floor_won_kg.isna()].d.median(), 2)

In [ ]:
def mdiff(a, b): return (a // 100 - b // 100) * 12 + (a % 100 - b % 100)
# 4g 사전세액심사 처리군을 지정 전 과소신고 크기로 분할(2026-09-13). 원가 대리: 변환비 코드(건조생강 = 신선×10, 건고추·고춧가루 = 냉동×5)는 지정 전 12개월 중국산 통관 단가,
# 그 밖은 미러 통계의 중국 보고 단가(집단×연도 단가 갭 gap_uv, 지정 전 3년 평균; 집단 라벨은 scripts/22와 같은 연계표 연결 성분). 갭 ≥ 0.5(로그)이면 "과소신고 있음".
cm_ = pd.read_csv(os.path.join(O, "comtrade_mirror_hs6_2012_2024.csv"), dtype={"cmdCode": str})
conc = duckdb.connect().execute(f"ATTACH '{KCS.replace(os.sep, '/')}' AS s (READ_ONLY)").sql("SELECT hs2022, hs_past FROM s.dim_hs6_concordance").df()
scope = set(cm_.cmdCode); sub = conc[conc.hs2022.isin(scope) | conc.hs_past.isin(scope)]
par = {}
def find(x):
    while par.setdefault(x, x) != x:
        par[x] = par[par[x]]; x = par[x]
    return x
for a_, b_ in zip(sub.hs2022, sub.hs_past): par[find("N" + a_)] = find("P" + b_)
grp_name = {}
for n_ in list(par): grp_name.setdefault(find(n_), set()).add(n_[1:])
label = {r_: min(v_) for r_, v_ in grp_name.items()}
def grp_of(hs6): return label.get(find("N" + hs6), hs6) if ("N" + hs6) in par else hs6
mcn = mp[mp.reporter == "CN"]
CONV_PRE = {"0910112000": ("0910111000", 10.0), "0910122000": ("0910111000", 10.0), "0904210000": ("0710807000", 5.0), "0904220000": ("0710807000", 5.0)}
rows = []
for row in hb.itertuples(index=False):
    m = int(row.event); ey = m // 100; g = grp_of(row.hs10[:6])
    if row.hs10 in CONV_PRE:
        base, k = CONV_PRE[row.hs10]
        t_ = cn[cn.hs10 == row.hs10].copy(); t_["j"] = t_.yyyymm.map(lambda x: mdiff(x, m)); t_ = t_[(t_.j >= -12) & (t_.j <= -1)]
        b_ = cn[cn.hs10 == base].copy(); b_["j"] = b_.yyyymm.map(lambda x: mdiff(x, m)); b_ = b_[(b_.j >= -12) & (b_.j <= -1)]
        gap = np.log(k * (b_.v.sum() / b_.w.sum()) / (t_.v.sum() / t_.w.sum())) if len(t_) and len(b_) and t_.w.sum() > 0 else np.nan
        src = f"변환비 {base}x{k:g}"; ny = len(t_)
    else:
        x_ = mcn[(mcn.grp == g) & mcn.year.between(ey - 3, ey - 1)]
        gap = x_.gap_uv.mean() if len(x_) else np.nan; src = f"미러 {g} {ey-3}~{ey-1}"; ny = len(x_)
    rows.append(dict(hs10=row.hs10, name=row.name, event=m, d=row.d, pre_gap=gap, gap_src=src, gap_n=ny, mfn=row.mfn, floor_won_kg=row.floor_won_kg, pre_v_musd=row.pre_v_musd))
sp = pd.DataFrame(rows)
sp["group"] = np.where(sp.pre_gap.isna(), "갭 없음", np.where(sp.pre_gap >= 0.5, "과소신고 있음", "과소신고 없음"))
sp.to_csv(os.path.join(O, "사건연구_사전세액심사_분할_코드.csv"), index=False, encoding="utf-8-sig")
display(sp.sort_values("pre_gap", ascending=False).round(2))
for gname, n_ in sp.group.value_counts().items(): chk(f"분할 {gname} 코드 수", n_, 0)
for row in sp[sp.pre_gap.notna()].itertuples(index=False): chk(f"분할 갭 {row.hs10}", row.pre_gap, 2)
chk("분할 갭 있음 중위 단절", sp[sp.group == "과소신고 있음"].d.median(), 2); chk("분할 갭 없음 중위 단절", sp[sp.group == "과소신고 없음"].d.median(), 2)
yes = sp[sp.group == "과소신고 있음"].hs10.tolist(); no = sp[sp.group == "과소신고 없음"].hs10.tolist(); gn = sp[sp.group == "갭 없음"].hs10.tolist()
yes3 = sp[sp.pre_gap >= 0.3].hs10.tolist(); no3 = sp[sp.pre_gap.notna() & (sp.pre_gap < 0.3)].hs10.tolist()
lead = lambda hs: {h: (202111 if h in ("0910112000", "0910122000") else pre_ev[h]) for h in hs}
runs = [("있음 공식 지정일", {h: pre_ev[h] for h in yes}, "lnuv"), ("있음 선행 사건일", lead(yes), "lnuv"), ("없음", {h: pre_ev[h] for h in no}, "lnuv"), ("갭 없음", {h: pre_ev[h] for h in gn}, "lnuv"),
        ("있음(0.3) 공식 지정일", {h: pre_ev[h] for h in yes3}, "lnuv"), ("있음(0.3) 선행 사건일", lead(yes3), "lnuv"), ("없음(0.3)", {h: pre_ev[h] for h in no3}, "lnuv"),
        ("있음 물량 선행 사건일", lead(yes), "lnq"), ("없음 물량", {h: pre_ev[h] for h in no}, "lnq")]
out = []
for lab, evd, yv in runs:
    if not evd: continue
    r_, _ = event_study(cn, evd, None, [c for c in controls if c not in evd], y=yv); x_ = summarize(r_, lab); x_["n_codes"] = len(evd); out.append(x_)
t16 = pd.DataFrame(out).round(3); display(t16); t16.to_csv(os.path.join(O, "사건연구_사전세액심사_분할.csv"), index=False, encoding="utf-8-sig")
for x_ in out: chk("분할 " + x_["label"] + " post", x_["post_mean"], 3); chk("분할 " + x_["label"] + " t", x_["post_mean_t"], 2); chk("분할 " + x_["label"] + " pre 평균", x_["pre_mean"], 2); chk("분할 " + x_["label"] + " pre|t|max", x_["pre_absmax_t"], 2); chk("분할 " + x_["label"] + " 관측", x_["n"], 0); chk("분할 " + x_["label"] + " 군집", x_["clusters"], 0); chk("분할 " + x_["label"] + " 코드", x_["n_codes"], 0)

### §5 강건성 — 잔여 통관 창(사후 3·6개월 제외), 연평균 환율의 하한 아래 몫

In [ ]:
rob = {}
# 둘째 잔여 통관 창: 사전세액심사 단가(중국)에서 사후 0~2, 0~5개월을 뺀 평균
r0 = ES_PRE["사전세액심사 단가(중국)"]
rob["사전세액심사 단가 post 3~12"] = pd.Series(dict(coef=r0[(r0.j >= 3) & (r0.j <= 12)].coef.mean()))
rob["사전세액심사 단가 post 6~12"] = pd.Series(dict(coef=r0[(r0.j >= 6) & (r0.j <= 12)].coef.mean()))
# 일곱째 환율: 연평균 환율로 하한 아래 몫
fl2 = fl.copy(); fl2["uv_won_y"] = fl2.uv * fl2.year.map(fxy); fl2["below_y"] = fl2.uv_won_y < fl2.floor_won_kg
rob["하한 아래 몫 월환율"] = pd.Series(dict(coef=np.average(fl[fl.year >= 2012].below, weights=fl[fl.year >= 2012].imp_dlr)))
rob["하한 아래 몫 연환율"] = pd.Series(dict(coef=np.average(fl2[fl2.year >= 2012].below_y, weights=fl2[fl2.year >= 2012].imp_dlr)))
t14 = pd.DataFrame(rob).T; display(t14.round(4))
chk("잔여 창 사전세액심사 단가 3~12", rob["사전세액심사 단가 post 3~12"]["coef"], 3); chk("잔여 창 사전세액심사 단가 6~12", rob["사전세액심사 단가 post 6~12"]["coef"], 3)
chk("하한 아래 몫 월환율", rob["하한 아래 몫 월환율"]["coef"], 4); chk("하한 아래 몫 연환율", rob["하한 아래 몫 연환율"]["coef"], 4)

### §6 검증 — 논문 A의 인용 수치 대조

In [ ]:
EXPECT_A = {
 # II장 자료
 "자료 일곱 상대 대조 행": (22164, 0), "자료 일곱 상대 대조 어긋남": (0, 0), "자료 하한 있는 행": (1092, 0), "자료 종량 대안 코드": (92, 0),
 "자료 하한 최소 원": (88, 0.5), "자료 하한 최대 원": (34000, 500), "자료 별표 1의 가 행": (9944, 0), "자료 별표 1의 나 행": (278, 0),
 "사전세액심사 코드 수": (48, 0), "유통이력 코드 수": (155, 0), "유통이력 구간 수": (237, 0), "자료 품목 상세 코드": (285, 0),
 "자료 미러 행": (91318, 0), "자료 미러 보고국": (28, 0), "쌍 수": (444, 0), "패널 코드 수": (602, 0), "패널 원산지 수": (233, 0),
 "자료 두 화면 일치율 A": (99.80, 0.005), "자료 두 화면 일치율 C": (99.97, 0.005), "자료 19 일곱 상대 대조 행": (22164, 0), "자료 19 일곱 상대 어긋남": (0, 0),
 # IV.1 표 2 하한
 "하한 코드 수": (31, 0), "하한 아래 몫 전체": (0.001, 0.0005), "하한 아래 몫 연환율": (0.0008, 0.00005),
 "하한 표 1201901000 수입": (6981, 0.5), "하한 표 1201901000 하한": (196, 0.5), "하한 표 1201901000 몫": (0.000, 0.0005), "하한 표 1201901000 비율중위": (3.81, 0.005),
 "하한 표 1201909000 수입": (2523, 0.5), "하한 표 1201909000 하한": (196, 0.5), "하한 표 1201909000 몫": (0.000, 0.0005), "하한 표 1201909000 비율중위": (6.27, 0.005),
 "하한 표 1207400000 수입": (2152, 0.5), "하한 표 1207400000 하한": (1057, 0.5), "하한 표 1207400000 몫": (0.000, 0.0005), "하한 표 1207400000 비율중위": (2.99, 0.005),
 "하한 표 1207991000 수입": (640, 0.5), "하한 표 1207991000 하한": (1025, 0.5), "하한 표 1207991000 몫": (0.000, 0.0005), "하한 표 1207991000 비율중위": (3.04, 0.005),
 "하한 표 0904210000 수입": (187, 0.5), "하한 표 0904210000 하한": (2300, 0.5), "하한 표 0904210000 몫": (0.000, 0.0005), "하한 표 0904210000 비율중위": (5.19, 0.005),
 "하한 표 0409000000 수입": (184, 0.5), "하한 표 0409000000 하한": (767, 0.5), "하한 표 0409000000 몫": (0.000, 0.0005), "하한 표 0409000000 비율중위": (26.48, 0.005),
 "하한 표 0703209000 수입": (167, 0.5), "하한 표 0703209000 하한": (500, 0.5), "하한 표 0703209000 몫": (0.013, 0.0005), "하한 표 0703209000 비율중위": (2.42, 0.005),
 "r 건고추/냉동고추 중국 2012": (1.10, 0.005), "r 건고추/냉동고추 중국 2024": (0.97, 0.005), "r 건고추/냉동고추 중국 최소": (0.83, 0.005), "r 건고추/냉동고추 중국 최대": (1.10, 0.005),
 "r 건대추/냉동대추 중국 최소": (0.02, 0.005), "r 건대추/냉동대추 중국 최대": (0.88, 0.005),
 "파일럿 건고추 중국 단가 2023~2025 최소": (2.7, 0.05), "파일럿 건고추 중국 단가 2023~2025 최대": (3.6, 0.05), "파일럿 건고추 하한/단가 최소": (0.49, 0.005), "파일럿 건고추 하한/단가 최대": (0.60, 0.005),  # 논문 A: 50~65% → 49~60% 수정 대기
 # IV.1 건고추 저가 신고 사건(조심 2023관0105) — 신고 단가 1,600달러/톤을 원/kg으로 바꾼 값과 하한·종량세
 "건고추 mfn 2022": (270, 0.05), "냉동고추 mfn 2022": (27, 0.05),
 "환율 2022.11": (1364.1, 0.05), "사건 건고추 신고 단가 원/kg": (2180, 5), "사건 건고추 하한 2022": (2300, 0.5), "사건 건고추 종량세 2022": (6210, 0.5), "사건 건고추 하한 아래": (1, 0),
 # IV.2 건조생강
 "파일럿 건조생강 중국 단가 2012": (1.94, 0.005), "파일럿 건조생강 중국 단가 2020": (0.46, 0.005), "파일럿 건조/신선×10 2020": (0.04, 0.005), "파일럿 건조/신선×10 2021": (0.03, 0.005),
 "파일럿 건조생강 중국 2021.11 톤": (0, 0.5), "파일럿 건조생강 중국 2022~ 월 톤 최대": (26, 0.5), "파일럿 건조생강 중국 연 단가 2022~2025 최소": (2.16, 0.005), "파일럿 건조생강 중국 연 단가 2022~2025 최대": (2.98, 0.005),  # 논문 A: 월 2.1~2.7달러 → 연평균 2.2~3.0달러 수정 대기
 "파일럿 건조생강 페루 월 톤 최소": (51, 0.5), "파일럿 건조생강 페루 월 톤 최대": (179, 0.5), "파일럿 건조생강 페루 단가 최소": (3.05, 0.005), "파일럿 건조생강 페루 단가 최대": (4.26, 0.005), "파일럿 건조생강 베트남 단가 2022": (1.53, 0.005),  # 논문 A: 2021.11~2022.09 창, 3.1~4.3달러·베트남 1.5달러 수정 대기
 "파일럿 생강 하한/건조 단가 최소": (0.06, 0.005), "파일럿 생강 하한/건조 단가 최대": (0.09, 0.005),  # 논문 A: 7~10% → 6~9% 수정 대기
 # IV.2 표 3 미러 생강
 **{f"건조생강 미러 단가 갭 {y}": (v, 0.005) for y, v in zip(range(2017, 2025), [1.44, 1.59, 1.76, 2.40, 2.33, 0.29, -0.02, 0.15])},
 **{f"건조생강 중국 보고 단가 {y}": (v, 0.005) for y, v in zip(range(2017, 2025), [2.72, 3.48, 3.70, 4.98, 4.97, 2.24, 2.20, 2.80])},
 **{f"건조생강 한국 단가 {y}": (v, 0.005) for y, v in zip(range(2017, 2025), [0.64, 0.71, 0.64, 0.45, 0.48, 1.68, 2.24, 2.41])},
 **{f"건조생강 중국 보고 톤 {y}": (v, 0.5) for y, v in zip(range(2017, 2025), [448, 368, 294, 205, 183, 39, 46, 46])},
 **{f"건조생강 한국 톤 {y}": (v, 0.5) for y, v in zip(range(2017, 2025), [394, 392, 507, 953, 532, 40, 10, 9])},
 "신선생강 중국 보고 단가 최소": (0.47, 0.005), "신선생강 중국 보고 단가 최대": (1.73, 0.005), "신선생강 한국 단가 최소": (0.58, 0.005), "신선생강 한국 단가 최대": (1.71, 0.005), "신선생강 미러 단가 갭 최소": (-0.21, 0.005), "신선생강 미러 단가 갭 최대": (0.04, 0.005),
 # IV.3 표 4 사건연구
 "사전세액심사 단가(중국) post": (-0.039, 0.0005), "사전세액심사 단가(중국) t": (-0.93, 0.005), "사전세액심사 단가(중국) pre|t|max": (2.11, 0.005), "사전세액심사 단가(중국) 관측": (136561, 0), "사전세액심사 단가(중국) 군집": (132, 0),
 "사전세액심사 물량(중국) post": (0.089, 0.0005), "사전세액심사 물량(중국) t": (0.53, 0.005), "사전세액심사 물량(중국) pre|t|max": (1.56, 0.005), "사전세액심사 물량(중국) 관측": (136561, 0), "사전세액심사 물량(중국) 군집": (132, 0),
 "사전세액심사 단가(전체) post": (0.050, 0.0005), "사전세액심사 단가(전체) t": (1.56, 0.005), "사전세액심사 단가(전체) pre|t|max": (2.11, 0.005), "사전세액심사 단가(전체) 관측": (953813, 0), "사전세액심사 단가(전체) 군집": (134, 0),
 "유통이력 물량(전체) post": (0.031, 0.0005), "유통이력 물량(전체) t": (0.77, 0.005),  # 논문 A 표 4: 0.78 → 0.77 수정 대기 "유통이력 물량(전체) pre|t|max": (1.88, 0.005), "유통이력 물량(전체) 관측": (5229992, 0), "유통이력 물량(전체) 군집": (216, 0),
 "유통이력 물량(중국) post": (-0.102, 0.0005), "유통이력 물량(중국) t": (-1.58, 0.005), "유통이력 물량(중국) pre|t|max": (1.12, 0.005), "유통이력 물량(중국) 관측": (519087, 0), "유통이력 물량(중국) 군집": (188, 0),
 "건조생강 단가 공식 지정 2022-03 post": (0.039, 0.0005), "건조생강 단가 공식 지정 2022-03 t": (0.40, 0.005), "건조생강 단가 공식 지정 2022-03 pre|t|max": (28.73, 0.005), "건조생강 단가 공식 지정 2022-03 관측": (9015, 0), "건조생강 단가 공식 지정 2022-03 군집": (102, 0),
 "건조생강 단가 선행 사건 2021-11 post": (2.239, 0.0005), "건조생강 단가 선행 사건 2021-11 t": (27.26, 0.005), "건조생강 단가 선행 사건 2021-11 pre|t|max": (3.98, 0.005), "건조생강 단가 선행 사건 2021-11 관측": (9021, 0), "건조생강 단가 선행 사건 2021-11 군집": (102, 0),
 "잔여 창 사전세액심사 단가 3~12": (-0.039, 0.0005), "잔여 창 사전세액심사 단가 6~12": (-0.045, 0.0005),
 # IV.4 표 5 이질성 — 논문 A 표 5: 참깨 단절 −0.25→−0.26·지정 전 수입 39.6→39.5, 메밀 −0.78→−0.79 수정 대기
 "사전세액심사 단가 단절 중위": (-0.01, 0.005), "사전세액심사 단가 단절 90분위": (0.93, 0.005), "사전세액심사 단가 단절 코드 수": (31, 0), "단가 단절 mfn 상관": (0.12, 0.005), "단가 단절 하한 있음 중위": (0.06, 0.005), "단가 단절 하한 없음 중위": (-0.02, 0.005),
 **{f"단절 {h} d": (d, 0.005) for h, d in [("0910122000", 2.28), ("0910112000", 1.33), ("1006301000", 0.97), ("0712200000", 0.93), ("0704902000", 0.36), ("0703201000", 0.32), ("1207400000", -0.26), ("0713329000", -0.35), ("1202420000", -0.45), ("1605542091", -0.50), ("1008100000", -0.79), ("0703209000", -0.83)]},
 **{f"단절 {h} mfn": (m, 0.05) for h, m in [("0910122000", 377.3), ("0910112000", 377.3), ("1006301000", 513.0), ("0712200000", 135.0), ("0704902000", 27.0), ("0703201000", 360.0), ("1207400000", 630.0), ("0713329000", 420.8), ("1202420000", 230.5), ("1605542091", 20.0), ("1008100000", 256.1), ("0703209000", 360.0)]},
 **{f"단절 {h} 하한": (f, 0.5) for h, f in [("0910122000", 247), ("0910112000", 247), ("0712200000", 133), ("0703201000", 500), ("1207400000", 1057), ("0703209000", 500)]},
 **{f"단절 {h} 지정 전 수입": (v, 0.05) for h, v in [("0910122000", 0.1), ("0910112000", 0.0), ("1006301000", 0.0), ("0712200000", 1.8), ("0704902000", 6.8), ("0703201000", 1.8), ("1207400000", 39.5), ("0713329000", 29.2), ("1202420000", 0.8), ("1605542091", 15.6), ("1008100000", 0.9), ("0703209000", 74.1)]},
 # IV.5 표 6 미러 회귀
 "미러 패널 관측": (21060, 0), "미러 집단 수": (154, 0), "미러 보고국 수": (28, 0), "미러 갭 중위 금액": (0.23, 0.005), "미러 갭 중위 물량": (0.44, 0.005), "미러 갭 중위 단가": (-0.12, 0.005),
 "미러 gap_v mfn_mean": (-0.0007, 0.00005), "미러 gap_v mfn_mean t": (-1.40, 0.005), "미러 gap_v mfn_gap": (-0.0002, 0.00005), "미러 gap_v mfn_gap t": (-0.43, 0.005), "미러 gap_v mfn_mean 관측": (21060, 0), "미러 gap_v mfn_mean 군집": (154, 0),
 "미러 gap_q mfn_mean": (-0.0004, 0.00005), "미러 gap_q mfn_mean t": (-0.58, 0.005), "미러 gap_q mfn_gap": (-0.0009, 0.00005), "미러 gap_q mfn_gap t": (-2.05, 0.005), "미러 gap_q mfn_mean 관측": (20359, 0), "미러 gap_q mfn_mean 군집": (154, 0),
 "미러 gap_uv mfn_mean": (-0.0001, 0.00005), "미러 gap_uv mfn_mean t": (-0.54, 0.005), "미러 gap_uv mfn_gap": (0.0005, 0.00005), "미러 gap_uv mfn_gap t": (1.65, 0.005), "미러 gap_uv mfn_mean 관측": (20359, 0), "미러 gap_uv mfn_mean 군집": (154, 0),
 "미러 gap_uv(중국) mfn_mean": (-0.0004, 0.00005), "미러 gap_uv(중국) mfn_mean t": (-1.51, 0.005), "미러 gap_uv(중국) mfn_gap": (0.0010, 0.00005), "미러 gap_uv(중국) mfn_gap t": (2.02, 0.005), "미러 gap_uv(중국) mfn_mean 관측": (1518, 0), "미러 gap_uv(중국) mfn_mean 군집": (137, 0),
 "미러 gap_q(농식품) mfn_mean": (-0.0001, 0.00005), "미러 gap_q(농식품) mfn_mean t": (-0.11, 0.005), "미러 gap_q(농식품) mfn_gap": (-0.0010, 0.00005), "미러 gap_q(농식품) mfn_gap t": (-2.01, 0.005), "미러 gap_q(농식품) mfn_mean 관측": (11457, 0), "미러 gap_q(농식품) mfn_mean 군집": (102, 0),
 "미러 gap_uv(농식품) mfn_mean": (-0.0001, 0.00005), "미러 gap_uv(농식품) mfn_mean t": (-0.26, 0.005), "미러 gap_uv(농식품) mfn_gap": (0.0002, 0.00005), "미러 gap_uv(농식품) mfn_gap t": (1.52, 0.005), "미러 gap_uv(농식품) mfn_mean 관측": (11457, 0), "미러 gap_uv(농식품) mfn_mean 군집": (102, 0),
 "농식품 HS6×보고국 수": (399, 0), "농식품 단가 갭 중위": (-0.07, 0.005), "농식품 단가 갭 90분위": (0.30, 0.005),
 **{f"농식품 단가 갭 상위 {i}": (v, 0.005) for i, v in enumerate([2.52, 2.37, 1.49, 1.41, 1.37], 1)},
 # IV.4 분할(2026-09-13, §4g) — 논문 A IV.4 표 6 반영
 "분할 과소신고 있음 코드 수": (3, 0), "분할 과소신고 없음 코드 수": (22, 0), "분할 갭 없음 코드 수": (6, 0),
 "분할 갭 0910122000": (3.56, 0.005), "분할 갭 0910112000": (3.51, 0.005), "분할 갭 2001909060": (0.54, 0.005), "분할 갭 0910111000": (0.40, 0.005), "분할 갭 1605542091": (0.36, 0.005), "분할 갭 0904210000": (-0.02, 0.005),
 "분할 있음 공식 지정일 post": (0.104, 0.0005), "분할 있음 공식 지정일 t": (1.39, 0.005), "분할 있음 공식 지정일 pre|t|max": (2.48, 0.005), "분할 있음 공식 지정일 관측": (13164, 0), "분할 있음 공식 지정일 군집": (103, 0),
 "분할 있음 선행 사건일 post": (1.205, 0.0005), "분할 있음 선행 사건일 t": (5.07, 0.005), "분할 있음 선행 사건일 pre|t|max": (1.33, 0.005), "분할 있음 선행 사건일 관측": (12954, 0), "분할 있음 선행 사건일 군집": (103, 0),
 "분할 없음 post": (-0.107, 0.0005), "분할 없음 t": (-2.10, 0.005), "분할 없음 pre|t|max": (1.52, 0.005), "분할 없음 관측": (74932, 0), "분할 없음 군집": (122, 0),
 "분할 갭 없음 post": (0.129, 0.0005), "분할 갭 없음 t": (2.09, 0.005), "분할 갭 없음 pre|t|max": (1.43, 0.005), "분할 갭 없음 관측": (19789, 0), "분할 갭 없음 군집": (106, 0),
 "분할 있음(0.3) 공식 지정일 post": (-0.034, 0.0005), "분할 있음(0.3) 선행 사건일 post": (0.509, 0.0005), "분할 있음(0.3) 선행 사건일 t": (3.09, 0.005), "분할 없음(0.3) post": (-0.093, 0.0005), "분할 없음(0.3) t": (-1.68, 0.005), "분할 있음(0.3) 공식 지정일 코드": (5, 0),
 # IV.5 미러 후보 통관 대조(2026-09-13, §3e) — 논문 A IV.5 끝 문단 반영
 **{f"후보 {k} 금액 갭": (v, 0.005) for k, v in [("VN 030695", -0.06), ("VN 230990", -0.01), ("VN 210690", -0.22), ("VN 200819", -0.03), ("CN 051199", -2.02)]},
 **{f"후보 {k} 물량 갭": (v, 0.005) for k, v in [("VN 030695", -2.60), ("VN 230990", -2.43), ("VN 210690", -1.76), ("VN 200819", -1.57), ("CN 051199", -3.41)]},
 **{f"후보 {k} 단가 갭": (v, 0.005) for k, v in [("VN 030695", 2.54), ("VN 230990", 2.41), ("VN 210690", 1.54), ("VN 200819", 1.54), ("CN 051199", 1.39)]},
 **{f"후보 {k} 상대국 톤/한국 톤": (v, 0.005) for k, v in [("VN 030695", 0.07), ("VN 230990", 0.09), ("VN 210690", 0.18), ("VN 200819", 0.22), ("CN 051199", 0.03)]},
 **{f"후보 {k} 단가 최소": (v, 0.005) for k, v in [("VN 030695", 0.70), ("VN 230990", 0.12), ("VN 210690", 1.07), ("VN 200819", 1.31), ("CN 051199", 0.34)]},
 **{f"후보 {k} 단가 최대": (v, 0.005) for k, v in [("VN 030695", 0.91), ("VN 230990", 0.15), ("VN 210690", 2.33), ("VN 200819", 8.23), ("CN 051199", 0.56)]},
 **{f"후보 {k} 다른 원산지 대비 중위": (v, 0.005) for k, v in [("VN 030695", 0.67), ("VN 230990", 0.42), ("VN 210690", 0.07), ("VN 200819", 0.91), ("CN 051199", 0.89)]},
 **{f"후보 {k} mfn": (v, 0.05) for k, v in [("VN 030695", 32.0), ("VN 230990", 4.2), ("VN 210690", 8.0), ("VN 200819", 45.0), ("CN 051199", 8.0)]},
 **{f"후보 {k} 적용 세율": (0.0, 0.05) for k in ["VN 030695", "VN 230990", "VN 210690", "VN 200819", "CN 051199"]},
 **{f"후보 {k} 하한 있음": (0, 0) for k in ["VN 030695", "VN 230990", "VN 210690", "VN 200819", "CN 051199"]},
 **{f"후보 {k} 사전세액심사": (0, 0) for k in ["VN 030695", "VN 230990", "VN 210690", "VN 200819", "CN 051199"]},
 **{f"후보 {k} 유통이력": (v, 0) for k, v in [("VN 030695", 1), ("VN 230990", 0), ("VN 210690", 0), ("VN 200819", 1), ("CN 051199", 0)]},
 "후보 금액 갭 절대치 최대(베트남)": (0.22, 0.005), "후보 물량 갭 최소": (-3.41, 0.005), "후보 물량 갭 최대": (-1.57, 0.005), "후보 베트남 2022~ 금액 갭 최소": (0.33, 0.005), "후보 베트남 2022~ 금액 갭 최대": (0.69, 0.005),
 "후보 견과 기타 베트남 단가 2021": (1.91, 0.005), "후보 견과 기타 베트남 단가 2025": (8.23, 0.005), "후보 볶은 참깨 베트남 천 톤 최소": (27, 0.5), "후보 볶은 참깨 베트남 천 톤 최대": (35, 0.5), "후보 볶은 참깨 베트남 단가 최소": (1.81, 0.005), "후보 볶은 참깨 베트남 단가 최대": (2.18, 0.005),
 # IV.1 건대추(2026-09-13) — 논문 A IV.1·논문 B V.2 문장 반영
 "r 건대추/냉동대추 중국 5톤 이상 최소": (0.43, 0.005), "r 건대추/냉동대추 중국 5톤 이상 최대": (0.88, 0.005), "r 건대추/냉동대추 5톤 이상 해 수": (6, 0), "r 건대추/냉동대추 5톤 이상 첫해": (2018, 0), "r 건대추/냉동대추 5톤 이상 끝해": (2024, 0),
 "냉동대추 중국 5톤 미만 해 수": (3, 0), "냉동대추 중국 5톤 미만 해 단가 최소": (9, 0.5), "냉동대추 중국 5톤 미만 해 단가 최대": (13, 0.5), "r 건대추/냉동대추 5톤 미만 최대": (0.06, 0.005), "냉동대추 중국 톤 2018": (20.4, 0.05), "냉동대추 중국 톤 2020": (653.1, 0.05), "냉동대추 중국 톤 2021": (9.2, 0.05), "냉동대추 중국 톤 2022": (55.0, 0.05), "냉동대추 중국 톤 2024": (7.4, 0.05),
 "냉동대추 중국 단가 2020": (0.72, 0.005), "건대추 중국 톤 2017~2025 최소": (222, 0.5), "건대추 중국 톤 2017~2025 최대": (640, 0.5), "건대추 중국 단가 2017~2025 최소": (0.80, 0.005), "건대추 중국 단가 2017~2025 최대": (1.68, 0.005), "건대추 하한 원": (948, 0.5), "건대추 중국 몫 2017~2025": (0.96, 0.005),
}
def check(name, exp):
    bad, miss, ok = [], [], 0
    for k, (v, tol) in exp.items():
        if k not in CHK: miss.append(k); continue
        got = CHK[k]
        try: g = float(got)
        except (TypeError, ValueError): g = np.nan
        if not np.isfinite(g) or abs(g - v) > tol + 1e-9: bad.append((k, v, got))
        else: ok += 1
    print(f"[{name}] 대조 {len(exp)}건: 통과 {ok}, 불일치 {len(bad)}, 미등록 {len(miss)}")
    for b in bad: print("   불일치", b)
    for m in miss: print("   미등록", m)
    return bad, miss
RES = {"논문 A": check("논문 A", EXPECT_A)}
cited = set(EXPECT_A)
print("등록만 되고 인용되지 않은 라벨:", len([k for k in CHK if k not in cited]))
json.dump(CHK, open(os.path.join(O, "논문A_수치.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=float)
json.dump({n: dict(bad=[list(map(str, b)) for b in r[0]], miss=r[1]) for n, r in RES.items()}, open(os.path.join(O, "논문A_검증_결과.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
n_bad = sum(len(r[0]) + len(r[1]) for r in RES.values())
assert n_bad == 0, f"논문 대조 실패 {n_bad}건 — outputs/논문A_검증_결과.json"
print("논문 A 대조 통과:", len(EXPECT_A), "건")